In [ ]:
#| default_exp magic/strategy

In [ ]:
#| export
from __future__ import annotations
from datetime import datetime
import numpy as np
from dataclasses import dataclass, field
import io, base64, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from typing import NamedTuple
from enum import Enum
from collections import defaultdict
import pandas as pd
import random
import math
import uuid
import httpx
import heapq

import networkx as nx

from monsterui.all import *
from fastcore.basics import patch

In [ ]:
#| export
from fasthtml.common import *
from fasthtml.jupyter import *
from fasthtml.common import *

In [ ]:
#| export
from HexMagic.primitives import HexTouchMap, HexDragMap, HexTouchMap, MapCord , PrimitiveDemo, Hex, HexWrapper
from HexMagic.primitives import MapRect, MapPath,MapCord,  MapSize, HexRegion
from HexMagic.styles import StyleCSS, SVGBuilder,SVGLayer, SVGPatternLoader, preview, app, rt, StyleDemo
from HexMagic.overlay import  TerrainDisplay, TerrainOverlay, ClimateOverlay, TerraDemo, DrainageBasins, OverlaySpec
from HexMagic.overlay import Terrain, HexGrid, HexPosition
from HexMagic.overlay import CreamOverlay, RiverOverlay,OceanWaveOverlay


In [ ]:
#| export
from HexMagic.magic.pieces import GameParts, Piece, Squad, PieceType, TargetType, SquadTarget, PlacementStrategy, PlanContext
from HexMagic.magic.food import FOOD_TABLE, FoodStats
 # Build main map
from HexMagic.magic.pieces import PieceOverlay, LanternOverlay, SquadNamesOverlay
from HexMagic.magic.moves import MoveError, Move, MoveLog, PurchasePreview, Placement, Purchase, Rotation, PURCHASE_COST
from HexMagic.magic.network import SquadFlow, Squad
from HexMagic.magic.turns import GameRecord, TurnProcessor, TurnCell

## Placement

In [ ]:
#| export
def _score_network(squad: Squad, parts: GameParts, flow: SquadFlow | None = None) -> float:
    """Score a squad's current facing configuration, weighted by active target."""
    if flow is None:
        flow = SquadFlow.from_squad(squad)
    G = flow.G
    if len(G) < 2: return 0.0
    
    king_comp = flow.king_component
    if not king_comp: return 0.0
    
    # --- Base: connectivity + structure ---
    connectivity = (len(king_comp) / len(G)) * 100
    bottleneck_penalty = len(flow.bottlenecks) * 5
    U = G.to_undirected(as_view=True)
    edge_bonus = U.number_of_edges() * 0.5
    base = connectivity - bottleneck_penalty + edge_bonus
    
    target = getattr(squad, '_target', None)
    
    # --- CONSOLIDATE or no target ---
    if target is None or target.type == TargetType.CONSOLIDATE:
        food_reach = sum(G.nodes[n]['harvest'] for n in king_comp)
        return base + food_reach
    
    target_hp = parts.i2hp(target.hex_idx)
    projection = flow.sight_projection(parts)
    
    if target.type == TargetType.KING:
        # Maximize drain on enemies near target king
        drain_score = 0.0
        for n in G.nodes:
            p = G.nodes[n]['piece']
            if p.food.drain <= 0: continue
            cone = p.sight().hexes
            for enemy_sq in parts.squads:
                if enemy_sq is squad: continue
                for ep in enemy_sq.pieces:
                    if ep.place is None: continue
                    if parts.hp2i(ep.place.location) in cone:
                        dist_to_target = max(1, ep.place.location.distance(target_hp))
                        drain_score += p.food.drain * (3.0 / dist_to_target)
        
        return base + drain_score * 3.0 + projection * 2.0
    
    else:  # OASIS
        land_set = {i for i in range(len(parts.terr.elevations))
                    if parts.terr.elevations[i] > 0}
        
        all_visible = set()
        for n in G.nodes:
            all_visible |= G.nodes[n]['piece'].sight().hexes
        new_land = len(all_visible & land_set)
        
        oasis_bonus = 0.0
        for n in G.nodes:
            cone = G.nodes[n]['piece'].sight().hexes
            for h in cone:
                if h in land_set:
                    dist = parts.i2hp(h).distance(target_hp)
                    if dist <= 3:
                        oasis_bonus += parts.fy.tiers[h] * 0.5
        
        food_reach = sum(G.nodes[n]['harvest'] for n in king_comp)
        return base + new_land * 1.5 + (projection + oasis_bonus) * 2.0 + food_reach


In [ ]:
#| export
@patch
def optimize_facings(self: Squad, parts: GameParts, max_iters: int = 3) -> int:
    """Coordinate descent over facings using target-aware objective."""
    dirs = HexPosition.directions()
    placed = [p for p in self.pieces if p.place is not None]
    if len(placed) < 2:
        return 0
    
    # Seed with cheap heuristic
    self.orient_facings(parts)
    
    # Priority ordering (computed once)
    flow = SquadFlow.topology_only(self)
    U = flow.G.to_undirected(as_view=True)
    king_id = next((n for n, d in flow.G.nodes(data=True) if d['is_king']), None)
    bottlenecks = set(nx.articulation_points(U)) if len(flow.G) > 1 else set()
    king_neighbors = set(U.neighbors(king_id)) if king_id else set()
    
    target = getattr(self, '_target', None)
    target_hp = parts.i2hp(target.hex_idx) if target and target.hex_idx >= 0 else None
    
    def priority(p):
        nid = p.id
        is_bn = nid in bottlenecks
        is_kn = nid in king_neighbors
        dist = p.place.location.distance(target_hp) if target_hp else 999
        return (not is_bn, not is_kn, dist)
    
    total_changes = 0
    
    for _ in range(max_iters):
        changed = False
        ordered = sorted(placed, key=priority)
        
        for piece in ordered:
            if piece.place is None: continue
            original = piece.place.facing
            best_facing = original
            
            # Score current state
            flow = SquadFlow.topology_only(self)
            best_score = _score_network(self, parts, flow=flow)
            
            for d in dirs:
                if d == original: continue
                piece.place.facing = d
                flow = SquadFlow.topology_only(self)
                s = _score_network(self, parts, flow=flow)
                if s > best_score:
                    best_score = s
                    best_facing = d
            
            piece.place.facing = best_facing
            if best_facing != original:
                changed = True
                total_changes += 1
        
        if not changed: break
    
    return total_changes


In [ ]:
#| export

@patch
def can_afford(self:PlacementStrategy, squad: Squad, parts: GameParts, cost: int) -> bool:
    king = next((p for p in squad.pieces 
                if p.type == PieceType.KING and p.place is not None), None)
    if not king: return False
    
    placed = [p for p in squad.pieces if p.place is not None]
    pantry = king.place.pantry
    
    # Dynamic reserve: scales with army size
    reserve = len(placed) * self.min_reserve_per_piece
    if pantry - cost < reserve:
        return False
    
    # Budget ratio check — but skip during bootstrap (only king placed)
    income = sum(int(p.food.harvest * p.place.harvest) for p in placed)
    upkeep = sum(p.food.diet for p in placed)
    if len(placed) > 1 and upkeep > 0 and income / upkeep < self.min_budget_ratio:
        return False
    
    # Net food check: don't buy while bleeding (but allow if plenty of reserves)
    net = income - upkeep
    if net < 0 and pantry < reserve * 2 and len(placed) > afforable_min:
        return False
    
    return True

@patch
def assess(self:PlacementStrategy, squad: Squad, parts: GameParts) -> str:
    king = next((p for p in squad.pieces 
                if p.type == PieceType.KING and p.place is not None), None)
    if not king: return "defend"
    
    pantry = king.place.pantry
    placed = [p for p in squad.pieces if p.place is not None]
    G = squad.to_nx()
    U = G.to_undirected(as_view=True) if len(G) > 1 else nx.Graph()
    bottlenecks = len(list(nx.articulation_points(U))) if len(G) > 1 else 0
    
    king_hp = king.place.location
    threat_count = sum(
        1 for sq in parts.squads if sq is not squad
        for p in sq.pieces if p.place is not None
        and parts.i2hp(parts.hp2i(p.place.location)).distance(king_hp) <= 4
    )
    
    if threat_count >= self.defend_threat or bottlenecks >= 2:
        return "defend"
    elif pantry > self.attack_pantry and len(placed) >= self.attack_placed:
        return "attack"
    else:
        return "expand"


In [ ]:
PlacementStrategy.EXPANSIVE = PlacementStrategy(
    "expansive", defend_threat=3, attack_pantry=80, attack_placed=6,
    min_reserve_per_piece=4, min_budget_ratio=0.8, buy_cooldown=0,
    target_composition={PieceType.BISHOP: 4, PieceType.PAWN: 0})

PlacementStrategy.AGGRESSIVE = PlacementStrategy(
    "aggressive", defend_threat=1, attack_pantry=40, attack_placed=3,
    knight_roi_mult=0.7, min_reserve_per_piece=3, min_budget_ratio=0.6, buy_cooldown=0,
    target_composition={PieceType.KNIGHT: 2, PieceType.BISHOP: 1})

PlacementStrategy.BALANCED = PlacementStrategy(
    "balanced", defend_threat=2, attack_pantry=60, attack_placed=4,
    min_reserve_per_piece=5, min_budget_ratio=1.0, buy_cooldown=1,
    target_composition={PieceType.BISHOP: 2, PieceType.KNIGHT: 1, PieceType.QUEEN: 1})

PlacementStrategy.TURTLE = PlacementStrategy(
    "turtle", defend_threat=1, attack_pantry=999, attack_placed=99,
    min_reserve_per_piece=8, min_budget_ratio=1.2, buy_cooldown=2,
    target_composition={PieceType.BISHOP: 3, PieceType.PAWN: 0})


In [ ]:
#| export
TARGET_MIX = {
    TargetType.KING: {          # hunting an enemy king
        PieceType.KNIGHT: 0.35,  # main attackers
        PieceType.BISHOP: 0.30,  # connectivity + sight
        PieceType.QUEEN:  0.15,  # drain amplifier
        PieceType.PAWN:   0.20,  # minimum food engine
    },
    TargetType.OASIS: {         # expanding toward resources
        PieceType.BISHOP: 0.50,  # sight + harvest
        PieceType.PAWN:   0.35,  # food engine
        PieceType.KNIGHT: 0.05,  # scout/defense only
        PieceType.QUEEN:  0.10,
    },
    TargetType.CONSOLIDATE: {   # turtling up
        PieceType.PAWN:   0.40,  # max harvest
        PieceType.BISHOP: 0.40,  # connectivity
        PieceType.QUEEN:  0.15,  # defense
        PieceType.KNIGHT: 0.05,
    },
}


In [ ]:
#| export
def _army_complete(squad: Squad, strategy: PlacementStrategy) -> bool:
    """Check if target_composition is satisfied by currently placed pieces."""
    if not strategy.target_composition:
        return False
    counts = defaultdict(int)
    for p in squad.pieces:
        if p.place is not None:
            counts[p.type] += 1
    return all(counts[pt] >= n for pt, n in strategy.target_composition.items())

def _graduate(strategy: PlacementStrategy, 
              needs: BoardNeeds | None = None) -> PlacementStrategy:
    if needs is None or not strategy.graduation:
        # Fall back to fixed graduation
        if strategy.graduation == "random":
            return random.choice(_STRATEGY_ROTATION)
        return next((s for s in _STRATEGY_ROTATION
                     if s.name == strategy.graduation), strategy)
    
    # Context-aware graduation
    if needs.king_threat >= 2.0:
        return PlacementStrategy.AGGRESSIVE      # under pressure → fight
    elif needs.food_gap > 3:
        return PlacementStrategy.TURTLE          # starving → consolidate
    elif needs.frontier_need > 0.6:
        return PlacementStrategy.EXPANSIVE       # lots unseen → explore
    else:
        return PlacementStrategy.BALANCED        # stable → balanced push




@patch
def _graduate(strategy: PlacementStrategy, 
              needs: BoardNeeds | None = None) -> PlacementStrategy:
    if needs is None or not strategy.graduation:
        # Fall back to fixed graduation
        if strategy.graduation == "random":
            return random.choice(_STRATEGY_ROTATION)
        return next((s for s in _STRATEGY_ROTATION
                     if s.name == strategy.graduation), strategy)
    
    # Context-aware graduation
    if needs.king_threat >= 2.0:
        return PlacementStrategy.AGGRESSIVE      # under pressure → fight
    elif needs.food_gap > 3:
        return PlacementStrategy.TURTLE          # starving → consolidate
    elif needs.frontier_need > 0.6:
        return PlacementStrategy.EXPANSIVE       # lots unseen → explore
    else:
        return PlacementStrategy.BALANCED        # stable → balanced push


In [ ]:
#tp.turnLogDisplay( tp.summaries_by_turn)

In [ ]:
#| export
# What the board currently needs (computed from state)
@dataclass
class BoardNeeds:
    food_gap: float      # how much more harvest/turn we need
    drain_need: float    # enemy value in sight range
    connect_gap: float   # bottlenecks or isolated pieces
    frontier_need: float # fraction of nearby land unseen
    king_threat: float   # enemy drain pressure on our king

    @classmethod
    def assess(cls, squad: Squad, parts: GameParts, flow: SquadFlow) -> 'BoardNeeds':
        king = next((p for p in squad.pieces 
                     if p.type == PieceType.KING and p.place is not None), None)
        if not king: return cls(0,0,0,0,0)
        
        placed = [p for p in squad.pieces if p.place is not None]
        
        # Food gap: are we losing food each turn?
        income = sum(int(p.food.harvest * p.place.harvest) for p in placed)
        upkeep = sum(p.food.diet for p in placed)
        food_gap = max(0, upkeep - income + 2)  # want +2 surplus minimum
        
        # Drain need: how many enemy pieces are near us?
        current_visible = set()
        for p in placed: current_visible |= p.sight().hexes
        enemy_in_sight = 0
        for sq in parts.squads:
            if sq is squad: continue
            for p in sq.pieces:
                if p.place and parts.hp2i(p.place.location) in current_visible:
                    enemy_in_sight += 1
        
        # Connectivity gap
        connect_gap = len(flow.bottlenecks) + len(squad.isolated_pieces())
        
        # Frontier: how much unseen land near king?
        king_ring = set(parts.grid.indices_in_range(
            parts.hp2i(king.place.location), 6))
        land_ring = {i for i in king_ring if parts.terr.elevations[i] > 0}
        seen = land_ring & current_visible
        frontier_need = 1.0 - (len(seen) / max(1, len(land_ring)))
        
        # King threat: enemy drain on our king's hex
        king_idx = parts.hp2i(king.place.location)
        king_threat = 0.0
        for sq in parts.squads:
            if sq is squad: continue
            for p in sq.pieces:
                if p.place and king_idx in p.sight().hexes:
                    king_threat += p.food.drain
        
        return cls(food_gap, enemy_in_sight, connect_gap, frontier_need, king_threat)


In [ ]:
# strategy_name -> stance -> weight vector matching BoardNeeds fields
PURCHASE_OBJ: dict[str, dict[str, tuple[float,...]]] = {
    #                        food  drain  connect  frontier  threat
    "aggressive": {
        "expand":           (0.5,  3.0,   1.0,     1.0,      0.5),
        "defend":           (1.0,  2.0,   3.0,     0.5,      3.0),
        "attack":           (0.3,  5.0,   1.0,     0.5,      0.5),
    },
    "expansive": {
        "expand":           (1.5,  0.5,   2.0,     4.0,      0.5),
        "defend":           (2.0,  0.5,   3.0,     1.0,      2.0),
        "attack":           (1.0,  1.5,   1.5,     3.0,      0.5),
    },
    "balanced": {
        "expand":           (2.0,  1.0,   2.0,     2.0,      1.0),
        "defend":           (2.0,  1.0,   3.0,     1.0,      3.0),
        "attack":           (1.0,  3.0,   1.5,     1.0,      1.0),
    },
    "turtle": {
        "expand":           (3.0,  0.3,   4.0,     1.0,      1.0),
        "defend":           (3.0,  0.5,   4.0,     0.5,      4.0),
        "attack":           (2.0,  1.0,   3.0,     0.5,      2.0),
    },
}


In [ ]:
#| export
# ── Locked compositions from Optuna studies ──────────────────────────────

PlacementStrategy.AGGRESSIVE = PlacementStrategy(
    "aggressive",
    defend_threat=1, attack_pantry=40, attack_placed=3,
    knight_roi_mult=0.7, min_reserve_per_piece=3,
    min_budget_ratio=0.6, buy_cooldown=0,
    target_composition={
        PieceType.PAWN:   5,
        PieceType.KNIGHT: 3,
        PieceType.BISHOP: 3,
    },
    graduation="random")

PlacementStrategy.TURTLE = PlacementStrategy(
    "turtle",
    defend_threat=1, attack_pantry=999, attack_placed=99,
    min_reserve_per_piece=4, min_budget_ratio=1.0, buy_cooldown=0,
    target_composition={
        PieceType.PAWN:   6,
        PieceType.BISHOP: 5,
        PieceType.QUEEN:  2,
    },
    graduation="balanced")

PlacementStrategy.EXPANSIVE = PlacementStrategy(
    "expansive",
    defend_threat=3, attack_pantry=80, attack_placed=6,
    min_reserve_per_piece=4, min_budget_ratio=0.8, buy_cooldown=0,
    target_composition={
        PieceType.PAWN:   4,
        PieceType.BISHOP: 5,  # bishops = sight range = territory
    },
    graduation="turtle")   # once you reach oasis, sit on it

PlacementStrategy.BALANCED = PlacementStrategy(
    "balanced",
    defend_threat=2, attack_pantry=60, attack_placed=4,
    min_reserve_per_piece=5, min_budget_ratio=1.0, buy_cooldown=0,
    target_composition={
        PieceType.PAWN:   4,
        PieceType.BISHOP: 2,
        PieceType.KNIGHT: 1,
        PieceType.QUEEN:  1,
    },
    graduation="aggressive")

# ── Locked PURCHASE_OBJ weights ───────────────────────────────────────────
#                        food  drain  connect  frontier  threat
PURCHASE_OBJ = {
    "aggressive": {
        "expand":        (0.7,  2.7,   3.7,     4.9,     4.6),
        "defend":        (0.7,  2.7,   3.7,     4.9,     4.6),
        "attack":        (0.7,  2.7,   3.7,     4.9,     4.6),
    },
    "turtle": {
        "expand":        (1.6,  4.3,   2.6,     2.4,     3.9),
        "defend":        (1.6,  4.3,   2.6,     2.4,     3.9),
        "attack":        (1.6,  4.3,   2.6,     2.4,     3.9),
    },
    "expansive": {
        "expand":        (1.1,  0.7,   1.5,     4.4,     1.0),
        "defend":        (1.1,  0.7,   1.5,     4.4,     1.0),
        "attack":        (1.1,  0.7,   1.5,     4.4,     1.0),
    },
    "balanced": {
        "expand":        (2.0,  1.0,   2.0,     2.0,     1.0),
        "defend":        (2.0,  1.0,   3.0,     1.0,     3.0),
        "attack":        (1.0,  3.0,   1.5,     1.0,     1.0),
    },
}

_STRATEGY_ROTATION = [
    PlacementStrategy.EXPANSIVE,
    PlacementStrategy.AGGRESSIVE,
    PlacementStrategy.BALANCED,
    PlacementStrategy.TURTLE,
]


In [ ]:
#| export
_STRATEGY_ROTATION = [
    PlacementStrategy.EXPANSIVE,
    PlacementStrategy.AGGRESSIVE, 
    PlacementStrategy.BALANCED,
    PlacementStrategy.TURTLE,
]

@patch
def assign_strategy(self: Squad, idx: int, randomize: bool = False) -> None:
    """Assign a starting strategy by rotation index or randomly."""
    if randomize:
        self.strategy = random.choice(_STRATEGY_ROTATION)
    else:
        self.strategy = _STRATEGY_ROTATION[idx % len(_STRATEGY_ROTATION)]


In [ ]:
#| export
class QueueEntry(NamedTuple):
    priority: float
    seq: int
    piece_type: PieceType


In [ ]:
#| export

@patch
def execute_queue(self: Squad, parts: GameParts) -> Move | None:
    """Pop next queue entry and resolve placement at execution time."""
    while self.queue:
        entry = self.queue.pop(0)
        # Delegate to piece-specific placement logic
        move = {
            PieceType.PAWN:   self.best_pawn_move,
            PieceType.BISHOP: self.best_bishop_move,
            PieceType.KNIGHT: self.best_knight_move,
            PieceType.QUEEN:  self.best_queen_move,
        }.get(entry.piece_type, lambda p: None)(parts)
        
        if move is not None:
            return move
        # If placement failed, skip and try next in queue
    return None

_queue_seq = 0

def _next_seq() -> int:
    global _queue_seq
    _queue_seq += 1
    return _queue_seq


#| export
def OptimalPlacementOverlay(top_n: int = 3, **kw) -> OverlaySpec:
    """Show top-N candidate hexes per piece type using flag-colored chess pieces."""
    
    def render(ctx) -> str:
        grid = ctx.grid
        N = len(grid.hexes)
        parts_list = []
        
        for squad in ctx.squads:
            placed = [p for p in squad.pieces if p.place is not None]
            if not placed: continue
            parts_obj = placed[0].place.parent
            terr = parts_obj.terr
            flag = squad.flag
            
            current_visible = set()
            for ally in placed:
                current_visible |= ally.sight().hexes
            land_set = {i for i in range(len(terr.elevations)) if terr.elevations[i] > 0}
            occupied = {parts_obj.hp2i(p.place.location)
                        for sq in parts_obj.squads for p in sq.pieces if p.place is not None}
            
            for pt, scorer in PLACEMENT_SCORERS.items():
                unplaced = [p for p in squad.pieces if p.type == pt and p.place is None]
                if not unplaced: continue
                
                scored = []
                for idx in current_visible:
                    s = scorer(idx, parts_obj, squad, current_visible, occupied, land_set)
                    if s > -999:
                        scored.append((idx, s))
                if not scored: continue
                
                scored.sort(key=lambda x: -x[1])
                top = scored[:top_n]
                
                scale = grid.radius * 0.7 / 22.5
                
                for rank, (idx, score) in enumerate(top):
                    opacity = 0.8 - rank * 0.2
                    
                    for fi in ctx.fine_indices(idx):
                        if 0 <= fi < N:
                            c = grid.hexes[fi].center
                            
                            # Ghost ring background
                            r = grid.radius * 0.6
                            parts_list.append(
                                f'<circle cx="{c.x:.1f}" cy="{c.y:.1f}" r="{r:.1f}" '
                                f'fill="{flag.lightPrimary}" fill-opacity="{opacity * 0.5:.2f}" '
                                f'stroke="{flag.darkPrimary}" stroke-opacity="{opacity:.2f}" '
                                f'stroke-width="1.5" stroke-dasharray="3,2"/>')
                            
                            # Chess piece silhouette
                            svg_str, pat_def = flag.piece_svg(
                                pt, MapCord(c.x, c.y),
                                scale=scale, size='board',
                                piece_id=f"opt_{squad.name}_{pt.value}_{rank}_{fi}")
                            if pat_def is not None:
                                ctx.builder.add_definition(pat_def)
                            parts_list.append(
                                f'<g opacity="{opacity:.2f}">{svg_str}</g>')
                            
                            # Rank number (top-right)
                            parts_list.append(
                                f'<text x="{c.x + r * 0.7:.1f}" y="{c.y - r * 0.7:.1f}" '
                                f'text-anchor="middle" dominant-baseline="central" '
                                f'font-size="{max(9, grid.radius * 0.35):.0f}" '
                                f'fill="{flag.darkPrimary}" font-weight="bold" '
                                f'opacity="{opacity:.2f}">{rank + 1}</text>')
                            
                            # Hex index (bottom)
                            parts_list.append(
                                f'<text x="{c.x:.1f}" y="{c.y + r * 1.1:.1f}" '
                                f'text-anchor="middle" dominant-baseline="central" '
                                f'font-size="{max(7, grid.radius * 0.28):.0f}" '
                                f'fill="{flag.darkPrimary}" '
                                f'opacity="{opacity * 0.7:.2f}">#{idx}</text>')
        
        return '\n'.join(parts_list)
    
    return OverlaySpec("optimal_placement", render, requires={'squads'}, priority=75)


and reconsider?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

```python
@patch
def reconsider(self: Squad, parts: GameParts) -> bool:
    """Re-evaluate strategy and target. Replans queue if anything changed.
    Returns True if a replan was triggered."""
    
    ctx = self.current_ctx(parts)
    
    # --- Check if we even need to reconsider ---
    if self.plan_ctx is not None and not self.plan_ctx.has_drifted(ctx) and self.queue:
        return False

    # --- Level 1: Should we change PlacementStrategy? ---
    new_strategy = self.strategy
    
    if ctx.threat >= 3.0 and self.strategy.name not in ("aggressive", "balanced"):
        new_strategy = PlacementStrategy.AGGRESSIVE  # under heavy fire
    elif ctx.threat == 0.0 and ctx.stance == "expand" and self.strategy.name == "aggressive":
        new_strategy = PlacementStrategy.EXPANSIVE   # no enemies, explore
    elif ctx.pantry > self.strategy.attack_pantry and ctx.placed >= self.strategy.attack_placed:
        new_strategy = PlacementStrategy.BALANCED    # rich enough to loosen up
    
    strategy_changed = new_strategy is not self.strategy
    if strategy_changed:
        self.strategy = new_strategy

    # --- Level 2: Should we change SquadTarget? ---
    new_target = self.target
    
    # Find nearest visible enemy king
    placed = [p for p in self.pieces if p.place is not None]
    visible = set()
    for p in placed: visible |= p.sight().hexes
    
    enemy_king_dist = None
    nearest_oasis_idx = None
    
    for sq in parts.squads:
        if sq is self: continue
        for p in sq.pieces:
            if p.place is None: continue
            idx = parts.hp2i(p.place.location)
            if p.type == PieceType.KING and idx in visible:
                d = p.place.location.distance(
                    next(k.place.location for k in self.pieces 
                         if k.type == PieceType.KING and k.place is not None))
                if enemy_king_dist is None or d < enemy_king_dist:
                    enemy_king_dist = d
                    new_king_target = SquadTarget(TargetType.KING, idx)

    # Find nearest oasis if consolidating
    if self.target.type == TargetType.CONSOLIDATE:
        for oasis_idx, richness in parts.fy.oases:
            if oasis_idx in visible:
                nearest_oasis_idx = oasis_idx
                break

    if ctx.threat >= 3.0 and enemy_king_dist is not None and enemy_king_dist <= 6:
        new_target = new_king_target                          # attack visible king
    elif ctx.stance == "expand" and nearest_oasis_idx is not None:
        new_target = SquadTarget(TargetType.OASIS, nearest_oasis_idx)  # head for oasis
    elif ctx.threat == 0.0 and ctx.stance != "attack":
        new_target = SquadTarget(TargetType.CONSOLIDATE)     # peace, build up

    target_changed = new_target.type != self.target.type or new_target.hex_idx != self.target.hex_idx
    if target_changed:
        self.target = new_target

    # --- Replan if anything changed or queue is empty/stale ---
    self.replan(parts)
    return True
```

The key structure:

- **Drift check first** — bail early if nothing changed and queue is healthy
- **Strategy swap** — threat level and pantry richness drive shifts between presets
- **Target swap** — visible enemy king, spotted oasis, or peace drives `SquadTarget` changes
- Always calls `replan()` if we reach this point, which sets a fresh `plan_ctx`

The `TurnProcessor` can then just call `squad.reconsider(parts)` at the top of each turn before popping from the queue.

@patch
def reconsider(self: Squad, parts: GameParts) -> bool:
    """Re-evaluate strategy and target. Replans queue if anything changed.
    Returns True if a replan was triggered."""
    
    ctx = self.current_ctx(parts)
    
    # --- Check if we even need to reconsider ---
    if self.plan_ctx is not None and not self.plan_ctx.has_drifted(ctx) and self.queue:
        return False

    # --- Cache oasis positions on parts (expensive to recompute) ---
    if not hasattr(parts, '_oasis_idxs'):
        try:
            parts._oasis_idxs = parts.fy.find_oasis(3, min_distance=6)
        except Exception:
            parts._oasis_idxs = []

    # --- Level 1: Should we change PlacementStrategy? ---
    flow = SquadFlow.topology_only(self)
    needs = BoardNeeds.assess(self, parts, flow)
    new_strategy = self.strategy

    # Graduate if target army is complete
    if _army_complete(self, self.strategy):
        new_strategy = _graduate(self.strategy, needs)
        if new_strategy is not self.strategy:
            self.strategy = new_strategy
            self.replan(parts)
            return True

    # Reactive strategy shifts
    if ctx.threat >= 3.0 and self.strategy.name not in ("aggressive", "balanced"):
        new_strategy = PlacementStrategy.AGGRESSIVE
    elif ctx.threat == 0.0 and ctx.stance == "expand" and self.strategy.name == "aggressive":
        new_strategy = PlacementStrategy.EXPANSIVE
    elif (ctx.pantry > self.strategy.attack_pantry 
          and ctx.placed >= self.strategy.attack_placed):
        new_strategy = PlacementStrategy.BALANCED
    
    strategy_changed = new_strategy is not self.strategy
    if strategy_changed:
        self.strategy = new_strategy

    # --- Level 2: Should we change SquadTarget? ---
    new_target = self.target
    
    placed = [p for p in self.pieces if p.place is not None]
    visible = set()
    for p in placed:
        visible |= p.sight().hexes
    
    enemy_king_dist = None
    new_king_target = None
    
    for sq in parts.squads:
        if sq is self:
            continue
        for p in sq.pieces:
            if p.place is None:
                continue
            idx = parts.hp2i(p.place.location)
            if p.type == PieceType.KING and idx in visible:
                king_hp = next(k.place.location for k in self.pieces 
                               if k.type == PieceType.KING and k.place is not None)
                d = p.place.location.distance(king_hp)
                if enemy_king_dist is None or d < enemy_king_dist:
                    enemy_king_dist = d
                    new_king_target = SquadTarget(TargetType.KING, idx)

    nearest_oasis_idx = None
    if self.target.type == TargetType.CONSOLIDATE:
        for oasis_idx in parts._oasis_idxs:
            if oasis_idx in visible:
                nearest_oasis_idx = oasis_idx
                break

    if ctx.threat >= 3.0 and new_king_target is not None and enemy_king_dist <= 6:
        new_target = new_king_target
    elif ctx.stance == "expand" and nearest_oasis_idx is not None:
        new_target = SquadTarget(TargetType.OASIS, nearest_oasis_idx)
    elif ctx.threat == 0.0 and ctx.stance != "attack":
        new_target = SquadTarget(TargetType.CONSOLIDATE)

    target_changed = (new_target.type != self.target.type 
                      or new_target.hex_idx != self.target.hex_idx)
    if target_changed:
        self.target = new_target

    # --- Rebuild queue ---
    self.replan(parts)
    return True


Can you build a demo that tests these?

# ── Fresh game ──
myStuff = GameParts()
myStuff.grid.adjustRadius(20)
playerCount = 3
myStuff.loadSquads(year=1950, seed=23, levels=playerCount)

for squad in myStuff.squads:
    for p in squad.pieces:
        p.place = None

starts = myStuff.find_starts(n=playerCount, ring_radius=3, min_distance=6)
for squad, start_idx in zip(myStuff.squads, starts):
    king = next(p for p in squad.pieces if p.type == PieceType.KING)
    loc  = myStuff.location_by_index(start_idx)
    Move.from_squad(squad, placement=Placement(king, loc)).apply(myStuff)
    king.place.pantry = 100

tp = TurnProcessor(myStuff, myStuff.squads)

# Place initial pawns
for squad in myStuff.squads:
    squad.place_optimal_pawns(myStuff, tp, k=3)

print("=== STRATEGY DEMO ===\n")

for round_num in range(10):
    print(f"── Turn {round_num} ──")
    for squad in myStuff.squads:
        if not squad.king_alive(): continue

        # Reconsider / replan
        replanned = squad.reconsider(myStuff)

        king = next(p for p in squad.pieces 
                    if p.type == PieceType.KING and p.place is not None)
        ctx = squad.plan_ctx

        print(f"  {squad.name:22s} | strategy={squad.strategy.name:10s} "
              f"| target={squad.target.type.name:12s} "
              f"| stance={ctx.stance:7s} "
              f"| pantry={ctx.pantry:4d} "
              f"| threat={ctx.threat:.1f} "
              f"| queue={len(squad.queue)} "
              f"| replanned={replanned}")

        # Show queue contents
        for i, m in enumerate(squad.queue):
            pt  = m.purchase.piece_type.name if m.purchase else "?"
            loc = m.purchase.location if m.purchase else "?"
            print(f"    [{i}] buy {pt:7s} @ {loc}")

        # Try to consume one queue entry
        if squad.queue:
            move = squad.queue.pop(0)
            try:
                move.apply(myStuff)
                print(f"    ✓ purchased {move.purchase.piece_type.name}")
            except MoveError as e:
                print(f"    ✗ MoveError: {e} → clearing queue")
                squad.queue.clear()
                squad.plan_ctx = None  # force replan next turn

    # Food/combat round
    moves = {sq: Move() for sq in myStuff.squads if sq.king_alive()}
    tp.play_round(moves)
    print()

print(tp.status())


!cat ../../HexMagic/magic/food.py

it looks ok, but we never seem to leave turtle

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Two separate issues visible in the output:

**1. Turtle never graduates** — by design, `TURTLE` has `attack_pantry=999, attack_placed=99`, which are intentionally unreachable. But there's no "graduation" path out of turtle once you've built enough. You need an explicit condition like:

```python
# Graduate out of turtle once well-established
elif (self.strategy.name == "turtle" 
      and ctx.placed >= 6 
      and ctx.pantry > 80 
      and ctx.threat == 0.0):
    new_strategy = PlacementStrategy.BALANCED
```

**2. `placed` is stuck at 4** — the final `tp.status()` shows `placed=4` for all squads even after 10 turns of bishop purchases. This means either:
- Purchases are silently failing (no `MoveError` raised, but hexes are already occupied)
- Or `placed` in `ctx` is being computed before the purchase is applied

You can verify which by adding a piece count print:

```python
total = len([p for p in squad.pieces if p.place is not None])
print(f"    actual placed: {total}")
```

The queue also keeps replanning the same hexes (e.g. `(-2, 0, 2)` every other turn for Storm Swans), which strongly suggests the purchases **are** failing silently — the hex never gets marked as occupied from the planner's perspective. Worth checking what `move.apply()` does when a piece already exists at that location.

Which issue do you want to tackle first?

So should we start in a state besides turtle?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yes — turtle as a *starting* state is actually counterproductive because:

1. **Early game is naturally expansive** — you only have a king, you need to build out fast, food is the priority
2. **Turtle's high reserve threshold** (`min_reserve_per_piece=8`) means `can_afford` blocks purchases even when you have plenty of pantry
3. **No enemies in sight early**, so turtle's defend-heavy weights are wasted

A better lifecycle would be:

| Phase | Strategy | Trigger |
|---|---|---|
| **Bootstrap** (1–4 pieces) | `EXPANSIVE` | default start |
| **Consolidate** (4–7 pieces) | `TURTLE` | stable income established |
| **Mature** (7+ pieces, full pantry) | `BALANCED` | pantry > threshold |
| **Combat** (enemy spotted) | `AGGRESSIVE` | threat spike |

So the `Squad` default should be `EXPANSIVE`, and `reconsider` graduates *into* turtle once income is stable, then out again when rich enough to fight.

Does that lifecycle feel right, or do you want some squads to start with different strategies to make games more varied?

We definitely want varied strategies. The only thing that is similar is that you should put a certain number of pawns at the start to build your engine (though the exact number might vary). really a group of many strategies would be great with some sort of rock paper scissors to them. but that is in a while. right now just varied strategies would work

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Simple approach — assign a strategy at load time, either randomly or by index so you always get variety in a 3-player game:

```python
_STRATEGY_ROTATION = [
    PlacementStrategy.EXPANSIVE,
    PlacementStrategy.AGGRESSIVE, 
    PlacementStrategy.BALANCED,
    PlacementStrategy.TURTLE,
]

@patch
def assign_strategy(self: Squad, idx: int, randomize: bool = False) -> None:
    """Assign a starting strategy by rotation index or randomly."""
    if randomize:
        self.strategy = random.choice(_STRATEGY_ROTATION)
    else:
        self.strategy = _STRATEGY_ROTATION[idx % len(_STRATEGY_ROTATION)]
```

Then in your setup loop:

```python
for i, squad in enumerate(myStuff.squads):
    squad.assign_strategy(i)   # or randomize=True for chaos
```

With 3 players you'd always get EXPANSIVE vs AGGRESSIVE vs BALANCED — a natural rock-paper-scissors seed for later.

The pawn bootstrap issue is separate — you could handle that by having `replan` **always** prioritize pawns if `placed < min_pawns` regardless of strategy, where `min_pawns` is a field on `PlacementStrategy` (e.g. AGGRESSIVE=2, EXPANSIVE=4, TURTLE=5). That way the engine-building phase is baked into each strategy's DNA rather than being a separate mode.

Want to add `min_pawns` to `PlacementStrategy` and wire it into `replan`?

so the balance with pawns is that we need a certain number to feed an army. so we need to figure out how big an army we want to create that min pawn number

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Exactly — it's a **food budget** problem. Given a target army composition, you need enough pawn harvest to cover total upkeep:

```
min_pawns = ceil((total_army_diet - pawn_net * n_pawns) / pawn_net)
```

But more precisely, given a target army like "2 bishops + 1 knight", you solve:

```
n_pawns * pawn_net >= bishops * bishop_diet + knights * knight_diet + ...
```

This connects directly to your MIP diet solver — the target composition **determines** `min_pawns`, not the other way around.

So the design would be:

1. `PlacementStrategy` holds a **target army composition** (e.g. `{BISHOP: 2, KNIGHT: 1, QUEEN: 0}`)
2. `min_pawns` is computed from that composition + `FOOD_TABLE` at plan time
3. `replan` fills pawns first until the food budget supports the target composition, *then* starts buying the attack pieces

This also means different strategies naturally get different pawn counts:
- **AGGRESSIVE**: small army, fewer pawns needed  
- **EXPANSIVE**: wide coverage, many bishops → more pawns
- **TURTLE**: large defensive army → most pawns

Does `target_composition` as a `dict[PieceType, int]` on `PlacementStrategy` feel right?

This feels right I put target_composition: dict[PieceType, int] = field(default_factory=dict) on PlacementStrategy

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Now wire it up with a computed `min_pawns` property and update the presets:

```python
@patch(as_prop=True)
def min_pawns(self: PlacementStrategy) -> int:
    """Minimum pawns needed to sustain the target army composition."""
    if not self.target_composition:
        return 2  # sensible floor
    
    pawn_stats = FOOD_TABLE[PieceType.PAWN]
    # Net food a pawn produces at tier 1 (conservative estimate)
    pawn_net = pawn_stats.harvest * 1 - pawn_stats.diet
    if pawn_net <= 0:
        return 4  # fallback
    
    # Total upkeep of non-pawn pieces in target composition
    army_diet = sum(
        FOOD_TABLE[pt].diet * count
        for pt, count in self.target_composition.items()
    )
    
    return max(2, math.ceil(army_diet / pawn_net))
```

Then update the presets with target compositions:

```python
PlacementStrategy.EXPANSIVE = PlacementStrategy(
    "expansive", defend_threat=3, attack_pantry=80, attack_placed=6,
    min_reserve_per_piece=4, min_budget_ratio=0.8, buy_cooldown=0,
    target_composition={PieceType.BISHOP: 4, PieceType.PAWN: 0})

PlacementStrategy.AGGRESSIVE = PlacementStrategy(
    "aggressive", defend_threat=1, attack_pantry=40, attack_placed=3,
    knight_roi_mult=0.7, min_reserve_per_piece=3, min_budget_ratio=0.6, buy_cooldown=0,
    target_composition={PieceType.KNIGHT: 2, PieceType.BISHOP: 1})

PlacementStrategy.BALANCED = PlacementStrategy(
    "balanced", defend_threat=2, attack_pantry=60, attack_placed=4,
    min_reserve_per_piece=5, min_budget_ratio=1.0, buy_cooldown=1,
    target_composition={PieceType.BISHOP: 2, PieceType.KNIGHT: 1, PieceType.QUEEN: 1})

PlacementStrategy.TURTLE = PlacementStrategy(
    "turtle", defend_threat=1, attack_pantry=999, attack_placed=99,
    min_reserve_per_piece=8, min_budget_ratio=1.2, buy_cooldown=2,
    target_composition={PieceType.BISHOP: 3, PieceType.PAWN: 0})
```

Then in `replan`, the pawn bootstrap check becomes simply:

```python
current_pawns = sum(1 for p in placed if p.type == PieceType.PAWN)
pawn_starved = current_pawns < self.strategy.min_pawns

# Force pawn if under minimum, otherwise score normally
if pawn_starved:
    affordable = {PieceType.PAWN: PURCHASE_COST[PieceType.PAWN]}
else:
    affordable = {pt: cost for pt, cost in PURCHASE_COST.items() ...}
```

This means every strategy bootstraps its own pawn floor before buying anything else, and the floor is derived from the army it's actually planning to build.

would it make sense that once it has built its army that it would then do something different (like this is how we get out of turtle) we could even have someone be random among the strategies

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yes — **army completion** is the perfect graduation trigger. Once `target_composition` is satisfied, the squad has "finished building" and needs a new objective. A few natural transitions:

| Strategy | Army complete → | Logic |
|---|---|---|
| `TURTLE` | `BALANCED` | defensive build done, now engage |
| `EXPANSIVE` | `BALANCED` or `AGGRESSIVE` | wide network built, time to press |
| `BALANCED` | `AGGRESSIVE` | already mixed, push harder |
| `AGGRESSIVE` | stays `AGGRESSIVE` | never satisfied |
| `RANDOM` | picks a new random one | chaos agent |

You could add a `graduation: str` field to `PlacementStrategy` pointing to the next strategy name:

```python
PlacementStrategy.TURTLE = PlacementStrategy(
    "turtle", ..., graduation="balanced")

PlacementStrategy.EXPANSIVE = PlacementStrategy(
    "expansive", ..., graduation="aggressive")
```

Then in `reconsider`, add:

```python
# --- Graduate if army is complete ---
def army_complete(squad, strategy):
    counts = {pt: sum(1 for p in squad.pieces 
                      if p.place and p.type == pt)
              for pt in strategy.target_composition}
    return all(counts.get(pt, 0) >= n 
               for pt, n in strategy.target_composition.items())

if army_complete(self, self.strategy) and self.strategy.graduation:
    if self.strategy.graduation == "random":
        new_strategy = random.choice(_STRATEGY_ROTATION)
    else:
        new_strategy = next(s for s in _STRATEGY_ROTATION 
                           if s.name == self.strategy.graduation)
```

The `RANDOM` graduation option is particularly interesting as a wildcard player — it keeps opponents guessing and makes replays feel different. Want to add `graduation` to `PlacementStrategy` and wire it in?

Yes I have added graduation to PlacementStrategy

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Update the presets and wire `army_complete` into `reconsider`:

```python
PlacementStrategy.EXPANSIVE = PlacementStrategy(
    "expansive", defend_threat=3, attack_pantry=80, attack_placed=6,
    min_reserve_per_piece=4, min_budget_ratio=0.8, buy_cooldown=0,
    target_composition={PieceType.BISHOP: 4},
    graduation="aggressive")

PlacementStrategy.AGGRESSIVE = PlacementStrategy(
    "aggressive", defend_threat=1, attack_pantry=40, attack_placed=3,
    knight_roi_mult=0.7, min_reserve_per_piece=3, min_budget_ratio=0.6, buy_cooldown=0,
    target_composition={PieceType.KNIGHT: 2, PieceType.BISHOP: 1},
    graduation="random")

PlacementStrategy.BALANCED = PlacementStrategy(
    "balanced", defend_threat=2, attack_pantry=60, attack_placed=4,
    min_reserve_per_piece=5, min_budget_ratio=1.0, buy_cooldown=1,
    target_composition={PieceType.BISHOP: 2, PieceType.KNIGHT: 1, PieceType.QUEEN: 1},
    graduation="aggressive")

PlacementStrategy.TURTLE = PlacementStrategy(
    "turtle", defend_threat=1, attack_pantry=999, attack_placed=99,
    min_reserve_per_piece=8, min_budget_ratio=1.2, buy_cooldown=2,
    target_composition={PieceType.BISHOP: 3},
    graduation="balanced")
```

```python
def _army_complete(squad: Squad, strategy: PlacementStrategy) -> bool:
    """Check if target_composition is satisfied by currently placed pieces."""
    if not strategy.target_composition:
        return False
    counts = defaultdict(int)
    for p in squad.pieces:
        if p.place is not None:
            counts[p.type] += 1
    return all(counts[pt] >= n for pt, n in strategy.target_composition.items())

def _graduate(strategy: PlacementStrategy) -> PlacementStrategy:
    """Return the next strategy after graduation."""
    if not strategy.graduation:
        return strategy
    if strategy.graduation == "random":
        return random.choice(_STRATEGY_ROTATION)
    return next((s for s in _STRATEGY_ROTATION 
                 if s.name == strategy.graduation), strategy)
```

Then add this block to `reconsider` right before the threat/pantry checks in Level 1:

```python
# --- Graduate if target army is complete ---
if _army_complete(self, self.strategy):
    new_strategy = _graduate(self.strategy)
    if new_strategy is not self.strategy:
        self.strategy = new_strategy
        self.replan(parts)
        return True
```

Putting graduation **first** in `reconsider` means it takes priority over threat reactions — a turtle that finishes its build graduates cleanly even in a quiet game, rather than waiting for a threat that may never come.

Should we also use board needs to assign strategies when you graduate?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yes — graduation is the perfect moment to be context-aware rather than following a fixed chain. Instead of a static `graduation` string, you could pick based on what the board actually needs:

```python
def _graduate(strategy: PlacementStrategy, 
              needs: BoardNeeds | None = None) -> PlacementStrategy:
    if needs is None or not strategy.graduation:
        # Fall back to fixed graduation
        if strategy.graduation == "random":
            return random.choice(_STRATEGY_ROTATION)
        return next((s for s in _STRATEGY_ROTATION
                     if s.name == strategy.graduation), strategy)
    
    # Context-aware graduation
    if needs.king_threat >= 2.0:
        return PlacementStrategy.AGGRESSIVE      # under pressure → fight
    elif needs.food_gap > 3:
        return PlacementStrategy.TURTLE          # starving → consolidate
    elif needs.frontier_need > 0.6:
        return PlacementStrategy.EXPANSIVE       # lots unseen → explore
    else:
        return PlacementStrategy.BALANCED        # stable → balanced push
```

Then pass `needs` at the graduation call site in `reconsider`:

```python
if _army_complete(self, self.strategy):
    flow = SquadFlow.topology_only(self)
    needs = BoardNeeds.assess(self, parts, flow)
    new_strategy = _graduate(self.strategy, needs)
    ...
```

The `graduation` field on `PlacementStrategy` then becomes a **fallback** hint rather than a hard rule — used only when `needs` can't distinguish (e.g. everything is low). This also means the same turtle squad graduates differently depending on whether there's an enemy nearby or open frontier to explore.

How are we on parts?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Here's a quick audit of where things stand:

**✅ Done:**
- `PlanContext` named tuple + `has_drifted()`
- `Squad` updated with `strategy`, `plan_ctx`, `queue`
- `BoardNeeds.assess()`
- `PURCHASE_OBJ` weight vectors
- `current_ctx()`, `replan()`, `reconsider()`
- `_army_complete()`, `_graduate()` with `BoardNeeds` awareness
- `PlacementStrategy` presets with `target_composition` + `graduation`
- `assign_strategy()` for varied starts
- Demo running end-to-end

**⚠️ Known issues:**
- Purchases appear to silently fail (placed count stuck at 4) — the `replan` sim isn't marking hexes as truly occupied
- All squads start as `TURTLE` — `assign_strategy` exists but isn't wired into `loadSquads`
- `min_pawns` property not yet wired into `replan` to enforce pawn bootstrap

**🔲 Not yet done:**
- `replan` doesn't enforce `min_pawns` before other purchases
- `assign_strategy` not called in setup
- The silent `MoveError` issue needs debugging
- `TurnProcessor` doesn't yet call `reconsider` automatically — it's manual in the demo

Which of the open issues do you want to tackle next?

can you rewrite replan and reconsider? things like _army_complete aren't in

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

```python
def _army_complete(squad: Squad, strategy: PlacementStrategy) -> bool:
    """Check if target_composition is satisfied by currently placed pieces."""
    if not strategy.target_composition:
        return False
    counts = defaultdict(int)
    for p in squad.pieces:
        if p.place is not None:
            counts[p.type] += 1
    return all(counts[pt] >= n for pt, n in strategy.target_composition.items())

def _graduate(strategy: PlacementStrategy, 
              needs: BoardNeeds | None = None) -> PlacementStrategy:
    if needs is None or not strategy.graduation:
        # Fall back to fixed graduation
        if strategy.graduation == "random":
            return random.choice(_STRATEGY_ROTATION)
        return next((s for s in _STRATEGY_ROTATION
                     if s.name == strategy.graduation), strategy)
    
    # Context-aware graduation
    if needs.king_threat >= 2.0:
        return PlacementStrategy.AGGRESSIVE      # under pressure → fight
    elif needs.food_gap > 3:
        return PlacementStrategy.TURTLE          # starving → consolidate
    elif needs.frontier_need > 0.6:
        return PlacementStrategy.EXPANSIVE       # lots unseen → explore
    else:
        return PlacementStrategy.BALANCED        # stable → balanced push

_STRATEGY_ROTATION = [
    PlacementStrategy.EXPANSIVE,
    PlacementStrategy.AGGRESSIVE,
    PlacementStrategy.BALANCED,
    PlacementStrategy.TURTLE,
]
```

```python
@patch
def current_ctx(self: Squad, parts: GameParts) -> PlanContext:
    """Snapshot current board conditions for drift detection."""
    king = next((p for p in self.pieces 
                 if p.type == PieceType.KING and p.place is not None), None)
    pantry = king.place.pantry if king else 0
    placed = len([p for p in self.pieces if p.place is not None])
    flow = SquadFlow.topology_only(self)
    needs = BoardNeeds.assess(self, parts, flow)
    stance = self.strategy.assess(self, parts)
    return PlanContext(pantry=pantry, placed=placed, 
                       threat=needs.king_threat, stance=stance)

@patch
def replan(self: Squad, parts: GameParts, n: int = 3) -> None:
    """Rebuild the build queue using current strategy + target."""
    self.queue.clear()

    king = next((p for p in self.pieces 
                 if p.type == PieceType.KING and p.place is not None), None)
    if not king:
        return

    ctx = self.current_ctx(parts)
    stance = ctx.stance
    weights_vec = PURCHASE_OBJ.get(self.strategy.name, PURCHASE_OBJ["balanced"])[stance]
    w_food, w_drain, w_connect, w_frontier, w_threat = weights_vec

    flow = SquadFlow.topology_only(self)
    needs = BoardNeeds.assess(self, parts, flow)

    placed = [p for p in self.pieces if p.place is not None]
    occupied = {parts.hp2i(p.place.location)
                for sq in parts.squads for p in sq.pieces if p.place is not None}
    enemy_idxs = {parts.hp2i(p.place.location)
                  for sq in parts.squads if sq is not self
                  for p in sq.pieces if p.place is not None}
    ally_idxs = {parts.hp2i(p.place.location) for p in placed}

    current_visible = set()
    for p in placed:
        current_visible |= p.sight().hexes

    land_set = {i for i in range(len(parts.terr.elevations)) 
                if parts.terr.elevations[i] > 0}
    dirs = HexPosition.directions()

    # --- Pawn bootstrap: how many more pawns do we need? ---
    current_pawns = sum(1 for p in placed if p.type == PieceType.PAWN)
    pawns_needed = max(0, self.strategy.min_pawns - current_pawns)

    # Simulate queue build-out
    sim_pantry = king.place.pantry
    sim_occupied = set(occupied)
    sim_visible = set(current_visible)
    sim_ally_idxs = set(ally_idxs)
    sim_pawns_placed = 0

    for step in range(n):
        valid_hexes = {idx for idx in sim_visible
                       if idx in land_set and idx not in sim_occupied}
        if not valid_hexes:
            break

        # Decide which piece types to consider this step
        if sim_pawns_placed < pawns_needed:
            # Force pawn until food engine is built
            candidates = {PieceType.PAWN: PURCHASE_COST[PieceType.PAWN]}
        else:
            candidates = dict(PURCHASE_COST)

        # Filter by affordability with reserve
        reserve = (len(placed) + step) * self.strategy.min_reserve_per_piece
        candidates = {pt: cost for pt, cost in candidates.items()
                      if cost <= sim_pantry - reserve}
        if not candidates:
            break

        best_score = -999
        best_move = None
        best_pt = None
        best_idx = None

        for pt, cost in candidates.items():
            stats = FOOD_TABLE[pt]

            for idx in valid_hexes:
                hp = parts.i2hp(idx)
                loc = parts.location_by_index(idx)
                tier = int(parts.fy.tiers[idx])
                eff_sight = max(2, loc.level) + stats.bonus_sight
                king_dist = hp.distance(king.place.location)

                for d in dirs:
                    cone = set(parts.hp2i(h) 
                               for h in hp.field_of_view(d, eff_sight))

                    harvest_roi  = stats.harvest * tier - stats.diet
                    new_land     = len((cone & land_set) - sim_visible)
                    enemies_hit  = len(enemy_idxs & cone)
                    allies_seen  = len(sim_ally_idxs & cone)
                    king_visible = 1 if parts.hp2i(king.place.location) in cone else 0

                    score = (
                        w_food     * harvest_roi
                        + w_drain    * (enemies_hit * stats.drain)
                        + w_connect  * (allies_seen + king_visible * 2)
                        + w_frontier * new_land
                        + w_threat   * (enemies_hit if needs.king_threat > 0 else 0)
                        - king_dist  * 0.5
                        + loc.level  * 0.3
                    )

                    if score > best_score:
                        best_score = score
                        best_pt    = pt
                        best_idx   = idx
                        best_move  = Move(purchase=Purchase(best_pt, loc, d))

        if best_move is None:
            break

        self.queue.append(best_move)

        # Advance simulation state
        stats = FOOD_TABLE[best_pt]
        cost  = PURCHASE_COST[best_pt]
        sim_pantry  -= cost
        sim_occupied.add(best_idx)
        sim_ally_idxs.add(best_idx)
        if best_pt == PieceType.PAWN:
            sim_pawns_placed += 1
        hp = parts.i2hp(best_idx)
        loc = parts.location_by_index(best_idx)
        eff_sight = max(2, loc.level) + FOOD_TABLE[best_pt].bonus_sight
        sim_visible |= set(parts.hp2i(h) 
                           for h in hp.field_of_view(dirs[0], eff_sight))

    self.plan_ctx = ctx
```

```python
@patch
def reconsider(self: Squad, parts: GameParts) -> bool:
    """Re-evaluate strategy and target. Replans queue if anything changed.
    Returns True if a replan was triggered."""
    
    ctx = self.current_ctx(parts)
    
    # --- Check if we even need to reconsider ---
    if self.plan_ctx is not None and not self.plan_ctx.has_drifted(ctx) and self.queue:
        return False

    # --- Cache oasis positions on parts (expensive to recompute) ---
    if not hasattr(parts, '_oasis_idxs'):
        try:
            parts._oasis_idxs = parts.fy.find_oasis(3, min_distance=6)
        except Exception:
            parts._oasis_idxs = []

    # --- Level 1: Should we change PlacementStrategy? ---
    flow = SquadFlow.topology_only(self)
    needs = BoardNeeds.assess(self, parts, flow)
    new_strategy = self.strategy

    # Graduate if target army is complete
    if _army_complete(self, self.strategy):
        new_strategy = _graduate(self.strategy, needs)
        if new_strategy is not self.strategy:
            self.strategy = new_strategy
            self.replan(parts)
            return True

    # Reactive strategy shifts
    if ctx.threat >= 3.0 and self.strategy.name not in ("aggressive", "balanced"):
        new_strategy = PlacementStrategy.AGGRESSIVE
    elif ctx.threat == 0.0 and ctx.stance == "expand" and self.strategy.name == "aggressive":
        new_strategy = PlacementStrategy.EXPANSIVE
    elif (ctx.pantry > self.strategy.attack_pantry 
          and ctx.placed >= self.strategy.attack_placed):
        new_strategy = PlacementStrategy.BALANCED
    
    strategy_changed = new_strategy is not self.strategy
    if strategy_changed:
        self.strategy = new_strategy

    # --- Level 2: Should we change SquadTarget? ---
    new_target = self.target
    
    placed = [p for p in self.pieces if p.place is not None]
    visible = set()
    for p in placed:
        visible |= p.sight().hexes
    
    enemy_king_dist = None
    new_king_target = None
    
    for sq in parts.squads:
        if sq is self:
            continue
        for p in sq.pieces:
            if p.place is None:
                continue
            idx = parts.hp2i(p.place.location)
            if p.type == PieceType.KING and idx in visible:
                king_hp = next(k.place.location for k in self.pieces 
                               if k.type == PieceType.KING and k.place is not None)
                d = p.place.location.distance(king_hp)
                if enemy_king_dist is None or d < enemy_king_dist:
                    enemy_king_dist = d
                    new_king_target = SquadTarget(TargetType.KING, idx)

    nearest_oasis_idx = None
    if self.target.type == TargetType.CONSOLIDATE:
        for oasis_idx in parts._oasis_idxs:
            if oasis_idx in visible:
                nearest_oasis_idx = oasis_idx
                break

    if ctx.threat >= 3.0 and new_king_target is not None and enemy_king_dist <= 6:
        new_target = new_king_target
    elif ctx.stance == "expand" and nearest_oasis_idx is not None:
        new_target = SquadTarget(TargetType.OASIS, nearest_oasis_idx)
    elif ctx.threat == 0.0 and ctx.stance != "attack":
        new_target = SquadTarget(TargetType.CONSOLIDATE)

    target_changed = (new_target.type != self.target.type 
                      or new_target.hex_idx != self.target.hex_idx)
    if target_changed:
        self.target = new_target

    # --- Rebuild queue ---
    self.replan(parts)
    return True
```

Key changes from the earlier versions:

- **`_army_complete` + `_graduate`** integrated into `reconsider` as the first Level 1 check
- **`BoardNeeds` passed to `_graduate`** for context-aware transitions
- **`replan` enforces `min_pawns`** — forces pawn purchases until food engine is built before buying other piece types
- **`sim_ally_idxs` tracked** in replan so later queue entries see earlier ones as allies for connectivity scoring
- **Reserve scales with step** in the queue so later purchases account for a larger army

I put your changes in. Does it look better?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

I don't have the output yet — could you run the demo cell and paste the results? Key things to look for:

1. **`placed` count growing** beyond 4 as turns progress
2. **Strategy transitions** — squads graduating out of their starting strategy
3. **Pawn bootstrap** — early queue entries should be `PAWN` if `min_pawns` not yet met
4. **Varied starting strategies** — are squads starting with different strategies now?

# ── Fresh game ──
myStuff = GameParts()
myStuff.grid.adjustRadius(20)
playerCount = 3
myStuff.loadSquads(year=1950, seed=23, levels=playerCount)

for squad in myStuff.squads:
    for p in squad.pieces:
        p.place = None

starts = myStuff.find_starts(n=playerCount, ring_radius=3, min_distance=6)
for squad, start_idx in zip(myStuff.squads, starts):
    king = next(p for p in squad.pieces if p.type == PieceType.KING)
    loc  = myStuff.location_by_index(start_idx)
    Move.from_squad(squad, placement=Placement(king, loc)).apply(myStuff)
    king.place.pantry = 100

tp = TurnProcessor(myStuff, myStuff.squads)

# Place initial pawns
for squad in myStuff.squads:
    squad.place_optimal_pawns(myStuff, tp, k=3)

print("=== STRATEGY DEMO ===\n")

for round_num in range(10):
    print(f"── Turn {round_num} ──")
    for squad in myStuff.squads:
        if not squad.king_alive(): continue

        # Reconsider / replan
        replanned = squad.reconsider(myStuff)

        king = next(p for p in squad.pieces 
                    if p.type == PieceType.KING and p.place is not None)
        ctx = squad.plan_ctx

        print(f"  {squad.name:22s} | strategy={squad.strategy.name:10s} "
              f"| target={squad.target.type.name:12s} "
              f"| stance={ctx.stance:7s} "
              f"| pantry={ctx.pantry:4d} "
              f"| threat={ctx.threat:.1f} "
              f"| queue={len(squad.queue)} "
              f"| replanned={replanned}")

        # Show queue contents
        for i, m in enumerate(squad.queue):
            pt  = m.purchase.piece_type.name if m.purchase else "?"
            loc = m.purchase.location if m.purchase else "?"
            print(f"    [{i}] buy {pt:7s} @ {loc}")

        # Try to consume one queue entry
        if squad.queue:
            move = squad.queue.pop(0)
            try:
                move.apply(myStuff)
                print(f"    ✓ purchased {move.purchase.piece_type.name}")
            except MoveError as e:
                print(f"    ✗ MoveError: {e} → clearing queue")
                squad.queue.clear()
                squad.plan_ctx = None  # force replan next turn

    # Food/combat round
    moves = {sq: Move() for sq in myStuff.squads if sq.king_alive()}
    tp.play_round(moves)
    print()

print(tp.status())


Thoughts

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Three clear problems visible in the output:

**1. Purchases aren't sticking (`placed` stuck at 4)**
Every turn shows `✓ purchased PAWN` but `placed` never grows. The queue keeps replanning the **same hexes** (e.g. `(-2, 0, 2)` reappears every 2 turns for Titanium Toucans) — dead giveaway that pieces placed there are dying between turns. Most likely cause: the purchased pawns are **starving immediately** — their diet exceeds harvest at those locations, and with pantry=0 on a new piece they die in the same turn they're bought.

**2. `Move(purchase=...)` vs `Move.from_squad(...)`**
The replan builds `Move(purchase=Purchase(...))` without a squad reference, whereas elsewhere you use `Move.from_squad(squad, purchase=...)`. This may mean the piece isn't being properly registered to the squad, leaving it unable to participate in food flow.

**3. All squads still starting as turtle**
`assign_strategy` exists but is never called in setup — all three squads default to `PlacementStrategy.TURTLE`.

The quickest diagnostic — add this after each purchase attempt:

```python
total = len([p for p in squad.pieces if p.place is not None])
print(f"    actual placed after purchase: {total}")
```

And check what `FOOD_TABLE[PieceType.PAWN].diet` vs `harvest` looks like at tier 0 locations — if `(-3, 1, 2)` is tier 0, the pawn is immediately net-negative and starves.

## best Moves

In [ ]:
#| export
@patch
def current_ctx(self: Squad, parts: GameParts) -> PlanContext:
    """Snapshot current board conditions for drift detection."""
    king = next((p for p in self.pieces 
                 if p.type == PieceType.KING and p.place is not None), None)
    pantry = king.place.pantry if king else 0
    placed = len([p for p in self.pieces if p.place is not None])
    flow = SquadFlow.topology_only(self)
    needs = BoardNeeds.assess(self, parts, flow)
    stance = self.strategy.assess(self, parts)
    return PlanContext(pantry=pantry, placed=placed, 
                       threat=needs.king_threat, stance=stance)

In [ ]:
#| export
@patch
def best_pawn_move(self: Squad, parts: GameParts) -> Move | None:
    """Find the best pawn placement: highest yield hex in or near ally sight.
    
    Returns a Move with optional rotation + pawn placement, or None if no pawns available.
    """
    # Find an unplaced pawn
    pawn = next((p for p in self.pieces 
                 if p.type == PieceType.PAWN and p.place is None), None)
    if pawn is None:
        return None
    
    placed = [p for p in self.pieces if p.place is not None]
    if not placed:
        return None
    
    grid = parts.grid
    terr = parts.terr
    tiers = parts.fy.tiers
    occupied = {parts.hp2i(p.place.location) 
                for sq in parts.squads for p in sq.pieces if p.place is not None}
    
    # First: check all hexes visible from current sight (no rotation needed)
    current_visible = set()
    for ally in placed:
        current_visible |= ally.sight().hexes
    
    # Filter to valid land hexes not already occupied
    candidates = {idx for idx in current_visible 
                  if 0 <= idx < len(terr.elevations) 
                  and terr.elevations[idx] > 0 
                  and idx not in occupied}
    
    best_idx = None
    best_yield = -1
    best_rotation = None  # (piece, new_facing) if rotation needed
    
    # Check current sight first
    for idx in candidates:
        if tiers[idx] > best_yield:
            best_yield = tiers[idx]
            best_idx = idx
            best_rotation = None
    
    # Now try each piece × each facing to find even better options
    dirs = HexPosition.directions()
    for ally in placed:
        original_facing = ally.place.facing
        for new_facing in dirs:
            if new_facing == original_facing:
                continue
            # Temporarily compute sight with new facing
            bonus = ally.food.bonus_sight
            effective_sight = max(2, ally.place.level) + bonus
            new_sight = set(
                parts.hp2i(hp) 
                for hp in ally.place.location.field_of_view(new_facing, effective_sight)
            )
            for idx in new_sight:
                if (0 <= idx < len(terr.elevations) 
                    and terr.elevations[idx] > 0 
                    and idx not in occupied
                    and tiers[idx] > best_yield):
                    best_yield = tiers[idx]
                    best_idx = idx
                    best_rotation = Rotation(ally, new_facing)
    
    if best_idx is None:
        return None
    
    loc = parts.location_by_index(best_idx)
    rotations = [best_rotation] if best_rotation else []
    
    return Move.from_squad(self, rotations=rotations, 
                           placement=Placement(pawn, loc))


In [ ]:
#| export
@patch
def best_queen_move(self: Squad, parts: GameParts, 
                    stance: str = "defend") -> Move | None:
    """Place a queen strategically based on stance.
    
    stance='defend': prioritize king protection + ally coverage
    stance='attack': prioritize draining enemy pieces
    """
    queen = next((p for p in self.pieces 
                  if p.type == PieceType.QUEEN and p.place is None), None)
    if queen is None:
        return None
    
    placed = [p for p in self.pieces if p.place is not None]
    if not placed:
        return None
    
    king = next((p for p in placed if p.type == PieceType.KING), None)
    if king and king.place.pantry < queen.food.cost + 10:
        return None
    
    terr = parts.terr
    dirs = HexPosition.directions()
    occupied = {parts.hp2i(p.place.location) 
                for sq in parts.squads for p in sq.pieces if p.place is not None}
    
    # Ally + enemy positions
    ally_idxs = {parts.hp2i(p.place.location) for p in placed}
    enemy_idxs = set()
    for sq in parts.squads:
        if sq is self: continue
        for p in sq.pieces:
            if p.place is not None:
                enemy_idxs.add(parts.hp2i(p.place.location))
    
    king_hp = king.place.location if king else None
    
    # All valid placement hexes
    current_visible = set()
    for ally in placed:
        current_visible |= ally.sight().hexes
    
    candidates = {idx for idx in current_visible
                  if 0 <= idx < len(terr.elevations)
                  and terr.elevations[idx] > 0
                  and idx not in occupied}
    
    best_idx = None
    best_facing = None
    best_score = -999
    
    for idx in candidates:
        hp = parts.i2hp(idx)
        loc = parts.location_by_index(idx)
        bonus = queen.food.bonus_sight
        eff_sight = max(2, loc.level) + bonus
        
        # Distance to king (lower = better for defense)
        king_dist = hp.distance(king_hp) if king_hp else 0
        
        for d in dirs:
            cone = set(parts.hp2i(h) for h in hp.field_of_view(d, eff_sight))
            caught_enemies = len(enemy_idxs & cone)
            allies_covered = len(ally_idxs & cone)
            
            if stance == "defend":
                # Fortress: protect the king, cover allies, still punish nearby enemies
                score = (
                    allies_covered * 3.0
                    - king_dist * 4.0
                    + caught_enemies * queen.food.drain * 1.0
                    + (5.0 if king_hp and parts.hp2i(king_hp) in cone else 0)  # bonus: can see king
                )
            else:  # "attack" — you miss 100% of the shots you don't take
                score = (
                    caught_enemies * queen.food.drain * 3.0
                    + allies_covered * 1.0
                    - king_dist * 1.0  # still prefer not to be stranded
                )
            
            if score > best_score:
                best_score = score
                best_idx = idx
                best_facing = d
    
    if best_idx is None:
        return None
    
    loc = parts.location_by_index(best_idx)
    loc.facing = best_facing
    
    # Rotation if target isn't currently visible
    rotations = []
    if best_idx not in current_visible:
        for ally in placed:
            b = ally.food.bonus_sight
            eff = max(2, ally.place.level) + b
            for d in dirs:
                cone = set(parts.hp2i(h) for h in ally.place.location.field_of_view(d, eff))
                if best_idx in cone:
                    rotations = [Rotation(ally, d)]
                    break
            if rotations:
                break
    
    return Move.from_squad(self, rotations=rotations,
                           placement=Placement(queen, loc))


In [ ]:
#| export
@patch
def best_bishop_move(self: Squad, parts: GameParts) -> Move | None:
    """Place a bishop using auto-detected role: relay, harvester, or frontier."""
    bishop = next((p for p in self.pieces 
                   if p.type == PieceType.BISHOP and p.place is None), None)
    if bishop is None:
        return None
    
    placed = [p for p in self.pieces if p.place is not None]
    if not placed:
        return None
    
    king = next((p for p in placed if p.type == PieceType.KING), None)
    if king and king.place.pantry < bishop.food.cost + 10:
        return None
    
    terr = parts.terr
    dirs = HexPosition.directions()
    occupied = {parts.hp2i(p.place.location) 
                for sq in parts.squads for p in sq.pieces if p.place is not None}
    
    enemy_idxs = {parts.hp2i(p.place.location)
                  for sq in parts.squads if sq is not self
                  for p in sq.pieces if p.place is not None}
    
    ally_idxs = {parts.hp2i(p.place.location) for p in placed}
    
    # Current collective sight
    current_visible = set()
    for ally in placed:
        current_visible |= ally.sight().hexes
    
    # Land hex set for frontier scoring
    land_set = {i for i in range(len(terr.elevations)) if terr.elevations[i] > 0}
    
    # --- Auto-detect role weights ---
    G = self.to_nx()
    U = G.to_undirected(as_view=True) if len(G) > 1 else nx.Graph()
    has_bottleneck = len(list(nx.articulation_points(U))) > 0 if len(G) > 1 else False
    isolated = self.isolated_pieces()
    king_pantry = king.place.pantry if king else 0
    food_tight = king_pantry < 30
    
    if has_bottleneck or len(isolated) > 0:
        # Network fragile — bridge it
        w_connect, w_drain, w_harvest, w_frontier = 15, 1, 1, 3
        role = "relay"
    elif food_tight:
        # Running low — earn food
        w_connect, w_drain, w_harvest, w_frontier = 5, 1, 8, 1
        role = "harvester"
    else:
        # Healthy — expand vision for future placements
        w_connect, w_drain, w_harvest, w_frontier = 5, 3, 2, 6
        role = "frontier"
    
    candidates = {idx for idx in current_visible
                  if idx in land_set and idx not in occupied}
    
    best_idx = None
    best_facing = None
    best_score = -999
    
    for idx in candidates:
        hp = parts.i2hp(idx)
        loc = parts.location_by_index(idx)
        bonus = bishop.food.bonus_sight
        eff_sight = max(2, loc.level) + bonus
        tier = int(parts.fy.tiers[idx])
        
        for d in dirs:
            cone = set(parts.hp2i(h) for h in hp.field_of_view(d, eff_sight))
            
            # Connectivity: allies this bishop can see
            allies_seen = len(ally_idxs & cone)
            
            # Drain: enemies caught
            enemies_caught = len(enemy_idxs & cone)
            
            # Harvest: net food over 4-turn lifespan
            harvest_per_turn = bishop.food.harvest * tier - bishop.food.diet
            lifetime_net = harvest_per_turn * LIFESPAN[PieceType.BISHOP] - bishop.food.cost
            
            # Frontier: new land hexes revealed that nobody currently sees
            new_land = len((cone & land_set) - current_visible)
            
            score = (allies_seen * w_connect
                     + enemies_caught * bishop.food.drain * w_drain
                     + lifetime_net * w_harvest
                     + new_land * w_frontier
                     + loc.level * 0.5)  # slight high-ground tiebreak
            
            if score > best_score:
                best_score = score
                best_idx = idx
                best_facing = d
    
    if best_idx is None:
        return None
    
    loc = parts.location_by_index(best_idx)
    loc.facing = best_facing
    
    # Rotation if target isn't in current sight
    rotations = []
    if best_idx not in current_visible:
        for ally in placed:
            b = ally.food.bonus_sight
            eff = max(2, ally.place.level) + b
            for d in dirs:
                cone = set(parts.hp2i(h) for h in ally.place.location.field_of_view(d, eff))
                if best_idx in cone:
                    rotations = [Rotation(ally, d)]
                    break
            if rotations:
                break
    
    return Move.from_squad(self, rotations=rotations,
                           placement=Placement(bishop, loc))


In [ ]:
#| export
@patch
def best_knight_move(self: Squad, parts: GameParts) -> Move | None:
    """Place a knight as a one-turn drain missile targeting enemy clusters."""
    knight = next((p for p in self.pieces 
                   if p.type == PieceType.KNIGHT and p.place is None), None)
    if knight is None:
        return None
    
    placed = [p for p in self.pieces if p.place is not None]
    if not placed:
        return None
    
    king = next((p for p in placed if p.type == PieceType.KING), None)
    if king and king.place.pantry < knight.food.cost + 5:
        return None  # don't fire blanks when we're broke
    
    terr = parts.terr
    dirs = HexPosition.directions()
    occupied = {parts.hp2i(p.place.location) 
                for sq in parts.squads for p in sq.pieces if p.place is not None}
    
    # Build enemy position map with piece value for targeting priority
    PIECE_VALUE = {
        PieceType.KING: 20, PieceType.QUEEN: 10,
        PieceType.BISHOP: 4, PieceType.PAWN: 2, PieceType.KNIGHT: 1,
    }
    enemy_map = {}  # idx -> total value of enemies at that hex
    for sq in parts.squads:
        if sq is self:
            continue
        for p in sq.pieces:
            if p.place is not None:
                idx = parts.hp2i(p.place.location)
                enemy_map[idx] = enemy_map.get(idx, 0) + PIECE_VALUE.get(p.type, 1)
    
    if not enemy_map:
        return None  # no targets, save the food
    
    # Current collective sight
    current_visible = set()
    for ally in placed:
        current_visible |= ally.sight().hexes
    
    land_set = {i for i in range(len(terr.elevations)) if terr.elevations[i] > 0}
    
    candidates = {idx for idx in current_visible
                  if idx in land_set and idx not in occupied}
    
    best_idx = None
    best_facing = None
    best_score = -999
    best_rotation = None
    
    # Also try with one rotation to unlock new hexes
    sight_configs = [(None, current_visible)]  # (rotation, visible_set)
    for ally in placed:
        original = ally.place.facing
        b = ally.food.bonus_sight
        eff = max(2, ally.place.level) + b
        for d in dirs:
            if d == original:
                continue
            new_cone = set(
                parts.hp2i(h) for h in ally.place.location.field_of_view(d, eff))
            # Recompute collective sight with this rotation
            expanded = (current_visible - ally.sight().hexes) | new_cone
            sight_configs.append((Rotation(ally, d), expanded))
    
    for rotation, visible in sight_configs:
        cands = {idx for idx in visible
                 if idx in land_set and idx not in occupied}
        
        for idx in cands:
            hp = parts.i2hp(idx)
            loc = parts.location_by_index(idx)
            eff_sight = max(2, loc.level)  # knight has 0 bonus_sight
            
            for d in dirs:
                cone = set(parts.hp2i(h) for h in hp.field_of_view(d, eff_sight))
                
                # Total drain value: enemies caught weighted by importance
                drain_value = sum(enemy_map.get(eidx, 0) 
                                  for eidx in (cone & set(enemy_map.keys())))
                enemies_hit = len(cone & set(enemy_map.keys()))
                
                # ROI: is draining worth the 6 food cost?
                total_drain = enemies_hit * knight.food.drain
                roi = total_drain - knight.food.cost - knight.food.diet
                
                if roi <= 0:
                    continue  # not worth deploying
                
                score = (drain_value * knight.food.drain * 2.0
                         + roi * 1.5
                         + loc.level * 0.3)  # high ground = bigger cone
                
                if score > best_score:
                    best_score = score
                    best_idx = idx
                    best_facing = d
                    best_rotation = rotation
    
    if best_idx is None:
        return None
    
    loc = parts.location_by_index(best_idx)
    loc.facing = best_facing
    
    rotations = [best_rotation] if best_rotation else []
    
    return Move.from_squad(self, rotations=rotations,
                           placement=Placement(knight, loc))


In [ ]:
#| export
# Module-level scorer functions (shared by overlay + panel)
def _score_pawn(idx, parts, squad, current_visible, occupied, land_set):
    if idx not in land_set or idx in occupied: return -999
    return int(parts.fy.tiers[idx])

def _score_bishop(idx, parts, squad, current_visible, occupied, land_set):
    if idx not in land_set or idx in occupied: return -999
    hp = parts.i2hp(idx)
    loc = parts.location_by_index(idx)
    bonus = FOOD_TABLE[PieceType.BISHOP].bonus_sight
    eff = max(2, loc.level) + bonus
    tier = int(parts.fy.tiers[idx])
    dirs = HexPosition.directions()
    ally_idxs = {parts.hp2i(p.place.location) for p in squad.pieces if p.place is not None}
    best = -999
    for d in dirs:
        cone = set(parts.hp2i(h) for h in hp.field_of_view(d, eff))
        allies_seen = len(ally_idxs & cone)
        new_land = len((cone & land_set) - current_visible)
        harvest_net = FOOD_TABLE[PieceType.BISHOP].harvest * tier - FOOD_TABLE[PieceType.BISHOP].diet
        score = allies_seen * 5 + new_land * 4 + harvest_net * 3 + loc.level * 0.5
        best = max(best, score)
    return best


def _score_queen(idx, parts, squad, current_visible, occupied, land_set):
    if idx not in land_set or idx in occupied: return -999
    hp = parts.i2hp(idx)
    loc = parts.location_by_index(idx)
    bonus = FOOD_TABLE[PieceType.QUEEN].bonus_sight
    eff = max(2, loc.level) + bonus
    dirs = HexPosition.directions()
    king = next((p for p in squad.pieces if p.type == PieceType.KING and p.place is not None), None)
    king_hp = king.place.location if king else None
    ally_idxs = {parts.hp2i(p.place.location) for p in squad.pieces if p.place is not None}
    enemy_idxs = {parts.hp2i(p.place.location)
                  for sq in parts.squads if sq is not squad
                  for p in sq.pieces if p.place is not None}
    king_dist = hp.distance(king_hp) if king_hp else 0
    best = -999
    for d in dirs:
        cone = set(parts.hp2i(h) for h in hp.field_of_view(d, eff))
        allies_covered = len(ally_idxs & cone)
        enemies_caught = len(enemy_idxs & cone)
        sees_king = 1 if king_hp and parts.hp2i(king_hp) in cone else 0
        score = (allies_covered * 3.0 - king_dist * 4.0
                 + enemies_caught * FOOD_TABLE[PieceType.QUEEN].drain
                 + sees_king * 5.0)
        best = max(best, score)
    return best



In [ ]:
#| export
def _score_knight(idx, parts, squad, current_visible, occupied, land_set):
    if idx not in land_set or idx in occupied: return -999
    hp = parts.i2hp(idx)
    loc = parts.location_by_index(idx)
    eff_sight = max(2, loc.level)
    dirs = HexPosition.directions()
    enemy_idxs = {parts.hp2i(p.place.location)
                  for sq in parts.squads if sq is not squad
                  for p in sq.pieces if p.place is not None}

    if not enemy_idxs:
        target = getattr(squad, 'target', None)
        king = next((p for p in squad.pieces 
                     if p.type == PieceType.KING and p.place is not None), None)
        if target and target.hex_idx >= 0 and king:
            target_hp = parts.i2hp(target.hex_idx)
            king_dist = king.place.location.distance(target_hp)
            new_dist = hp.distance(target_hp)
            approach_bonus = (king_dist - new_dist) * 2.0
            return loc.level * 1.5 + approach_bonus
        return loc.level * 1.5

    # Normal drain scoring (unchanged)
    best = -999
    for d in dirs:
        cone = set(parts.hp2i(h) for h in hp.field_of_view(d, eff_sight))
        enemies_hit = len(cone & enemy_idxs)
        roi = enemies_hit * FOOD_TABLE[PieceType.KNIGHT].drain - FOOD_TABLE[PieceType.KNIGHT].cost
        if roi <= 0: continue
        best = max(best, roi + loc.level * 0.3)
    return best if best > -999 else loc.level * 0.5


In [ ]:
#| export
PLACEMENT_SCORERS = {
    PieceType.PAWN:   _score_pawn,
    PieceType.BISHOP: _score_bishop,
    PieceType.KNIGHT: _score_knight,
    PieceType.QUEEN:  _score_queen,
}


I just added some functions like best queen fuction. score for things. But I don't know if it makes sense to integrate them or how. Will these help our dilemmas

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Good news — these are actually complementary to `replan`, not redundant. The issue is they're solving at **different levels**, and right now `replan` is trying to do everything itself (badly). Here's how they fit together:

**`replan`** should decide **what** to buy and **when** (the queue order):
- "pawn, pawn, bishop, knight" based on strategy weights

**`best_*_move` / `PLACEMENT_SCORERS`** should decide **where** to place it (at execution time, not plan time):
- Uses current board state, not stale sim state from 3 turns ago

This solves the biggest current bug — planned hexes going stale. The queue entries would just store the **piece type**, and at execution time you call `best_bishop_move()` etc. to get the actual placement:

```python
@dataclass
class QueueEntry:
    piece_type: PieceType
    priority: float  # from replan scoring

@patch
def execute_queue(self: Squad, parts: GameParts) -> Move | None:
    """Pop next queue entry and resolve placement at execution time."""
    while self.queue:
        entry = self.queue.pop(0)
        # Delegate to piece-specific placement logic
        move = {
            PieceType.PAWN:   self.best_pawn_move,
            PieceType.BISHOP: self.best_bishop_move,
            PieceType.KNIGHT: self.best_knight_move,
            PieceType.QUEEN:  self.best_queen_move,
        }.get(entry.piece_type, lambda p: None)(parts)
        
        if move is not None:
            return move
        # If placement failed, skip and try next in queue
    return None
```

This way:
- `replan` uses `PURCHASE_OBJ` weights + `BoardNeeds` to decide the **shopping list**
- `best_*_move` handles the tactical **where/facing** decision with fresh board state
- No more stale hex references in the queue
- `MoveError` from occupied hexes goes away entirely

Want to refactor in this direction?

 I put QueueEntry as a queue squad and imported heapq so we can use it.

 ```python
 class QueueEntry(NamedTuple):
    priority: float      # negated so heapq (min-heap) gives highest first
    piece_type: PieceType
    ```

```python
@dataclass
class Squad:
    ...
    queue: list[QueueEntry] = field(default_factory=list)  # heapq
```

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Now `replan` just needs to build the heap with negated priorities, and `execute_queue` pops from it:

```python
@patch
def replan(self: Squad, parts: GameParts, n: int = 3) -> None:
    """Rebuild the build queue using current strategy + target."""
    self.queue.clear()

    king = next((p for p in self.pieces 
                 if p.type == PieceType.KING and p.place is not None), None)
    if not king:
        return

    ctx = self.current_ctx(parts)
    stance = ctx.stance
    weights_vec = PURCHASE_OBJ.get(self.strategy.name, PURCHASE_OBJ["balanced"])[stance]
    w_food, w_drain, w_connect, w_frontier, w_threat = weights_vec

    flow = SquadFlow.topology_only(self)
    needs = BoardNeeds.assess(self, parts, flow)

    placed = [p for p in self.pieces if p.place is not None]
    occupied = {parts.hp2i(p.place.location)
                for sq in parts.squads for p in sq.pieces if p.place is not None}
    enemy_idxs = {parts.hp2i(p.place.location)
                  for sq in parts.squads if sq is not self
                  for p in sq.pieces if p.place is not None}

    current_visible = set()
    for p in placed:
        current_visible |= p.sight().hexes

    land_set = {i for i in range(len(parts.terr.elevations))
                if parts.terr.elevations[i] > 0}

    # --- Pawn bootstrap ---
    current_pawns = sum(1 for p in placed if p.type == PieceType.PAWN)
    pawns_needed = max(0, self.strategy.min_pawns - current_pawns)

    # --- Score each candidate piece type using PLACEMENT_SCORERS ---
    sim_pantry = king.place.pantry
    sim_pawns_planned = 0
    entries = []

    for step in range(n):
        # Which types are we allowed to consider?
        if sim_pawns_planned < pawns_needed:
            candidate_types = [PieceType.PAWN]
        else:
            candidate_types = list(PURCHASE_COST.keys())

        # Filter by affordability
        reserve = (len(placed) + step) * self.strategy.min_reserve_per_piece
        candidate_types = [pt for pt in candidate_types
                           if PURCHASE_COST[pt] <= sim_pantry - reserve]
        if not candidate_types:
            break

        # Score each type by sampling placement scorer over visible hexes
        best_pt = None
        best_priority = -999

        for pt in candidate_types:
            scorer = PLACEMENT_SCORERS.get(pt)
            if scorer is None:
                continue
            # Sample top score across valid hexes
            hex_scores = [
                scorer(idx, parts, self, current_visible, occupied, land_set)
                for idx in current_visible
                if idx in land_set and idx not in occupied
            ]
            if not hex_scores:
                continue
            placement_score = max(hex_scores)

            # Combine with strategy weights
            stats = FOOD_TABLE[pt]
            food_score    = stats.harvest - stats.diet
            drain_score   = stats.drain * len(enemy_idxs)
            connect_score = placement_score  # proxy for connectivity
            frontier_score = placement_score
            threat_score  = stats.drain if needs.king_threat > 0 else 0

            priority = (
                w_food     * food_score
                + w_drain    * drain_score
                + w_connect  * connect_score
                + w_frontier * frontier_score
                + w_threat   * threat_score
            )

            if priority > best_priority:
                best_priority = priority
                best_pt = pt

        if best_pt is None:
            break

        heapq.heappush(entries, QueueEntry(-best_priority, best_pt))

        # Advance sim state
        sim_pantry -= PURCHASE_COST[best_pt]
        if best_pt == PieceType.PAWN:
            sim_pawns_planned += 1

    self.queue = entries
    self.plan_ctx = ctx
```

```python
@patch
def execute_queue(self: Squad, parts: GameParts) -> Move | None:
    """Pop highest-priority entry and resolve placement with fresh board state."""
    while self.queue:
        entry = heapq.heappop(self.queue)
        move = {
            PieceType.PAWN:   self.best_pawn_move,
            PieceType.BISHOP: self.best_bishop_move,
            PieceType.KNIGHT: self.best_knight_move,
            PieceType.QUEEN:  self.best_queen_move,
        }.get(entry.piece_type, lambda p: None)(parts)

        if move is not None:
            return move
        # Placement found no valid hex — try next entry
    return None
```

Key changes:
- `replan` only stores **piece types + priorities** in the heap, no hexes
- Priorities are scored using `PLACEMENT_SCORERS` (sampling max over visible hexes) combined with `PURCHASE_OBJ` weights
- `execute_queue` resolves the actual hex/facing at pop time using `best_*_move` — always fresh board state, no stale hex problem
- The demo loop should now call `squad.execute_queue(myStuff)` instead of manually popping

@patch
def next_purchase_type(self: Squad, target_type, max_army_size=10):
    placed = [p for p in self.pieces if p.place is not None and p.type != PieceType.KING]
    counts = defaultdict(int)
    for p in placed:
        counts[p.type] += 1
    
    if len(placed) >= max_army_size:
        return None
    
    # Current food balance
    income = sum(int(p.food.harvest * p.place.harvest) 
                 for p in self.pieces if p.place is not None)
    upkeep = sum(p.food.diet for p in self.pieces if p.place is not None)
    surplus = income - upkeep
    
    # Force pawns until we have positive surplus + buffer
    min_surplus = 3  # need at least 3/turn headroom before buying attackers
    if surplus < min_surplus:
        return PieceType.PAWN
    
    mix = TARGET_MIX[target_type]
    target_size = max(len(placed) + 1, 6)
    
    biggest_gap = -999
    best_pt = None
    for pt, ratio in mix.items():
        desired = ratio * target_size
        gap = desired - counts[pt]
        stats = FOOD_TABLE[pt]
        
        # Don't buy zero-harvest pieces if surplus can't cover diet
        if stats.harvest == 0 and surplus < stats.diet + 1:
            continue
        
        if gap > biggest_gap:
            biggest_gap = gap
            best_pt = pt
    
    return best_pt


@patch
def next_purchase_type(self: Squad, target_type, max_army_size=10):
    placed = [p for p in self.pieces if p.place is not None and p.type != PieceType.KING]
    counts = defaultdict(int)
    for p in placed:
        counts[p.type] += 1
    
    if len(placed) >= max_army_size:
        return None
    
    # Actual net food via flow network (accounts for connectivity + drain)
    flow = SquadFlow.from_squad(self)
    net_food = flow.net_delivery  # food reaching king per turn
    
    # Force pawns until we have real surplus reaching the king
    min_surplus = 3
    if net_food < min_surplus:
        return PieceType.PAWN
    
    mix = TARGET_MIX[target_type]
    target_size = max(len(placed) + 1, 6)
    
    biggest_gap = -999
    best_pt = None
    for pt, ratio in mix.items():
        desired = ratio * target_size
        gap = desired - counts[pt]
        stats = FOOD_TABLE[pt]
        
        # Can the network absorb this piece's diet?
        if net_food < stats.diet + 1:
            continue
        
        if gap > biggest_gap:
            biggest_gap = gap
            best_pt = pt
    
    return best_pt


@patch
def next_purchase_type(self: Squad, target_type, max_army_size=10):
    placed = [p for p in self.pieces if p.place is not None and p.type != PieceType.KING]
    counts = defaultdict(int)
    for p in placed:
        counts[p.type] += 1
    
    if len(placed) >= max_army_size:
        return None
    
    # Net food from pieces actually connected to king
    flow = SquadFlow.from_squad(self)
    king_comp = flow.king_component
    connected_surplus = sum(
        flow.G.nodes[n]['harvest'] - flow.G.nodes[n]['diet']
        for n in king_comp
    )
    
    # Force pawns until connected network has real surplus
    min_surplus = 3
    if connected_surplus < min_surplus:
        return PieceType.PAWN
    
    mix = TARGET_MIX[target_type]
    target_size = max(len(placed) + 1, 6)
    
    biggest_gap = -999
    best_pt = None
    for pt, ratio in mix.items():
        desired = ratio * target_size
        gap = desired - counts[pt]
        stats = FOOD_TABLE[pt]
        
        # Can the connected network absorb this piece's diet?
        if connected_surplus < stats.diet + 1:
            continue
        
        if gap > biggest_gap:
            biggest_gap = gap
            best_pt = pt
    
    return best_pt


In [ ]:
#| export
@patch
def next_purchase_type(self: Squad, target_type, max_army_size=10):
    placed = [p for p in self.pieces if p.place is not None and p.type != PieceType.KING]
    counts = defaultdict(int)
    for p in placed:
        counts[p.type] += 1
    
    if len(placed) >= max_army_size:
        return None
    
    # Net food from pieces actually connected to king
    flow = SquadFlow.from_squad(self)
    king_comp = flow.king_component
    connected_surplus = sum(
        flow.G.nodes[n]['harvest'] - flow.G.nodes[n]['diet']
        for n in king_comp
    )
    
    # Subtract incoming enemy drain on our pieces
    parts = placed[0].place.parent if placed else None
    if parts:
        for sq in parts.squads:
            if sq is self: continue
            for ep in sq.pieces:
                if ep.place is None or ep.food.drain <= 0: continue
                cone = ep.sight().hexes
                for p in self.pieces:
                    if p.place and p.id in king_comp:
                        idx = parts.hp2i(p.place.location)
                        if idx in cone:
                            connected_surplus -= ep.food.drain
    
    # Force pawns until connected network has real surplus
    min_surplus = 3
    if connected_surplus < min_surplus:
        return PieceType.PAWN
    
    mix = TARGET_MIX[target_type]
    target_size = max(len(placed) + 1, 6)
    
    biggest_gap = -999
    best_pt = None
    for pt, ratio in mix.items():
        desired = ratio * target_size
        gap = desired - counts[pt]
        stats = FOOD_TABLE[pt]
        
        if connected_surplus < stats.diet + 1:
            continue
        
        if gap > biggest_gap:
            biggest_gap = gap
            best_pt = pt
    
    return best_pt


In [ ]:
#| export
@patch
def army_centroid(squad:Squad, parts):
    """Centroid of non-king, non-pawn pieces (the 'attack force')."""
    attack_pieces = [p for p in squad.pieces 
                     if p.place is not None 
                     and p.type not in (PieceType.KING, PieceType.PAWN)]
    if not attack_pieces:
        # Fall back to all placed
        attack_pieces = [p for p in squad.pieces if p.place is not None]
    
    qs = [p.place.location.q for p in attack_pieces]
    rs = [p.place.location.r for p in attack_pieces]
    return HexPosition(round(sum(qs)/len(qs)), round(sum(rs)/len(rs)), 
                        round(-sum(qs)/len(qs) - sum(rs)/len(rs)))

@patch
def centroid_progress(squad:Squad, parts, target):
    """How far is our attack centroid from the target? Lower = better."""
    if target.hex_idx < 0:
        return 0.0
    c = squad.army_centroid( parts)
    target_hp = parts.i2hp(target.hex_idx)
    return c.distance(target_hp)


## Canidates

In [ ]:
#| export
def hexes_in_range(n: int) -> list[HexPosition]:
    """All hex offsets within distance N from origin."""
    results = []
    for q in range(-n, n + 1):
        for r in range(max(-n, -q - n), min(n, -q + n) + 1):
            s = -q - r
            results.append(HexPosition(q, r, s))
    return results

def piece_candidates(pt: PieceType, king_hp: HexPosition, parts: GameParts,
                     current_visible: set, occupied: set,
                     land_set: set, n: int = 8, top_k: int = 15) -> list[int]:
    """Fast candidate hex set per piece type, sorted by placement preference."""
    nearby = [parts.hp2i(king_hp + d) for d in hexes_in_range(n)]
    valid = [idx for idx in nearby
             if idx is not None
             and idx in land_set
             and idx in current_visible
             and idx not in occupied]
    
    if pt == PieceType.BISHOP:
        # High ground first, then food richness
        key = lambda i: (parts.location_by_index(i).level, parts.fy.tiers[i])
    elif pt == PieceType.KNIGHT:
        # Forward deploy: far from king, high ground
        key = lambda i: (parts.location_by_index(i).level,
                         -parts.i2hp(i).distance(king_hp))
    elif pt == PieceType.QUEEN:
        # Close to king, high ground
        key = lambda i: (parts.location_by_index(i).level,
                         parts.i2hp(i).distance(king_hp))
    else:  # PAWN
        # Pure food richness
        key = lambda i: parts.fy.tiers[i]
    
    return sorted(valid, key=key, reverse=True)[:top_k]


In [ ]:
#| export
def bishop_candidates(king_hp: HexPosition, parts: GameParts, 
                      current_visible: set, occupied: set, 
                      land_set: set, n: int = 6, top_k: int = 10) -> list[int]:
    """Fast candidate set for bishop placement: high ground near king."""
    nearby = [
        parts.hp2i(king_hp + d)  # offset from king
        for d in hexes_in_range(n)
    ]
    return sorted(
        [idx for idx in nearby
         if idx is not None
         and idx in land_set
         and idx in current_visible
         and idx not in occupied],
        key=lambda i: (parts.location_by_index(i).level,   # high ground first
                       parts.fy.tiers[i]),                  # then food richness
        reverse=True
    )[:top_k]


In [ ]:
#| export
@patch
def replan(self: Squad, parts: GameParts, n: int = 3) -> None:
    self.queue.clear()
    king = next((p for p in self.pieces 
                 if p.type == PieceType.KING and p.place is not None), None)
    if not king:
        return

    ctx = self.current_ctx(parts)
    sim_pantry = king.place.pantry
    entries = []
    placed = [p for p in self.pieces if p.place is not None]

    for step in range(n):
        # Target-driven: what type are we most behind on?
        pt = self.next_purchase_type(self.target.type)
        if pt is None:
            break

        # Affordability check
        reserve = (len(placed) + step) * self.strategy.min_reserve_per_piece
        cost = PURCHASE_COST.get(pt)
        if cost is None or cost > sim_pantry - reserve:
            break

        # Priority from placement scorer (how good is the best hex?)
        occupied = {parts.hp2i(p.place.location)
                    for sq in parts.squads for p in sq.pieces if p.place is not None}
        current_visible = set()
        for p in placed:
            current_visible |= p.sight().hexes
        land_set = {i for i in range(len(parts.terr.elevations))
                    if parts.terr.elevations[i] > 0}

        scorer = PLACEMENT_SCORERS.get(pt)
        if scorer is None:
            break

        cands = piece_candidates(pt, king.place.location, parts,
                                 current_visible, occupied, land_set)
        if not cands:
            break

        priority = max(scorer(idx, parts, self, current_visible, occupied, land_set)
                       for idx in cands)

        heapq.heappush(entries, QueueEntry(-priority, _next_seq(), pt))
        sim_pantry -= cost

    self.queue = entries
    self.plan_ctx = ctx


In [ ]:
#| export
@patch
def reconsider(self: Squad, parts: GameParts) -> bool:
    """Re-evaluate strategy and target. Replans queue if anything changed.
    Returns True if a replan was triggered."""
    
    ctx = self.current_ctx(parts)
    
    # --- Check if we even need to reconsider ---
    if self.plan_ctx is not None and not self.plan_ctx.has_drifted(ctx) and self.queue:
        return False

    # --- Cache oasis positions on parts (expensive to recompute) ---
    if not hasattr(parts, '_oasis_idxs'):
        try:
            parts._oasis_idxs = parts.fy.find_oasis(3, min_distance=6)
        except Exception:
            parts._oasis_idxs = []

    # --- Level 1: Should we change PlacementStrategy? ---
    flow = SquadFlow.topology_only(self)
    needs = BoardNeeds.assess(self, parts, flow)
    new_strategy = self.strategy

    # Graduate if target army is complete
    if _army_complete(self, self.strategy):
        new_strategy = _graduate(self.strategy, needs)
        if new_strategy is not self.strategy:
            self.strategy = new_strategy
            self.replan(parts)
            return True

    # Reactive strategy shifts
    if ctx.threat >= 3.0 and self.strategy.name not in ("aggressive", "balanced"):
        new_strategy = PlacementStrategy.AGGRESSIVE
    elif ctx.threat == 0.0 and ctx.stance == "expand" and self.strategy.name == "aggressive":
        new_strategy = PlacementStrategy.EXPANSIVE
    elif (ctx.pantry > self.strategy.attack_pantry 
          and ctx.placed >= self.strategy.attack_placed):
        new_strategy = PlacementStrategy.BALANCED
    
    strategy_changed = new_strategy is not self.strategy
    if strategy_changed:
        self.strategy = new_strategy

    # --- Level 2: Should we change SquadTarget? ---
    new_target = self.target
    
    placed = [p for p in self.pieces if p.place is not None]
    visible = set()
    for p in placed:
        visible |= p.sight().hexes
    
    enemy_king_dist = None
    new_king_target = None
    
    for sq in parts.squads:
        if sq is self:
            continue
        for p in sq.pieces:
            if p.place is None:
                continue
            idx = parts.hp2i(p.place.location)
            if p.type == PieceType.KING and idx in visible:
                king_hp = next(k.place.location for k in self.pieces 
                               if k.type == PieceType.KING and k.place is not None)
                d = p.place.location.distance(king_hp)
                if enemy_king_dist is None or d < enemy_king_dist:
                    enemy_king_dist = d
                    new_king_target = SquadTarget(TargetType.KING, idx)

    nearest_oasis_idx = None
    if self.target.type == TargetType.CONSOLIDATE:
        for oasis_idx in parts._oasis_idxs:
            if oasis_idx in visible:
                nearest_oasis_idx = oasis_idx
                break

    if ctx.threat >= 3.0 and new_king_target is not None and enemy_king_dist <= 6:
        new_target = new_king_target
    elif ctx.stance == "expand" and nearest_oasis_idx is not None:
        new_target = SquadTarget(TargetType.OASIS, nearest_oasis_idx)
    elif ctx.threat == 0.0 and ctx.stance != "attack":
        new_target = SquadTarget(TargetType.CONSOLIDATE)

    target_changed = (new_target.type != self.target.type 
                      or new_target.hex_idx != self.target.hex_idx)
    if target_changed:
        self.target = new_target

    # --- Rebuild queue ---
    self.replan(parts)
    return True


In [ ]:
#| export
@patch
def execute_queue(self: Squad, parts: GameParts) -> Move | None:
    """Pop highest-priority entry, find best hex, return a Purchase move."""
    placed = [p for p in self.pieces if p.place is not None]
    if not placed:
        return None

    king = next((p for p in placed if p.type == PieceType.KING), None)
    if not king:
        return None

    occupied = {parts.hp2i(p.place.location)
                for sq in parts.squads for p in sq.pieces if p.place is not None}
    current_visible = set()
    for p in placed:
        current_visible |= p.sight().hexes
    land_set = {i for i in range(len(parts.terr.elevations))
                if parts.terr.elevations[i] > 0}

    while self.queue:
        entry = heapq.heappop(self.queue)
        pt = entry.piece_type
        scorer = PLACEMENT_SCORERS.get(pt)
        if scorer is None:
            continue

        # Use pre-filtered candidates, not all visible hexes
        cands = piece_candidates(pt, king.place.location, parts,
                                 current_visible, occupied, land_set)
        if not cands:
            continue

        best_idx = max(cands, key=lambda i: scorer(
            i, parts, self, current_visible, occupied, land_set))
        best_score = scorer(best_idx, parts, self, current_visible, occupied, land_set)

        if best_score <= -999:
            continue

        loc = parts.location_by_index(best_idx)
        hp = parts.i2hp(best_idx)
        stats = FOOD_TABLE[pt]
        eff_sight = max(2, loc.level) + stats.bonus_sight

        # Best facing
        best_facing = max(
            HexPosition.directions(),
            key=lambda d: len(
                {parts.hp2i(h) for h in hp.field_of_view(d, eff_sight)} & current_visible
            )
        )

        return Move.from_squad(self, purchase=Purchase(pt, loc, best_facing))

    return None


@patch
def __lt__(self: PieceType, other: PieceType) -> bool:
    
    return PURCHASE_COST[self] < PURCHASE_COST[other]


In [ ]:
# ── Fresh game ──
myStuff = GameParts()
myStuff.grid.adjustRadius(20)
playerCount = 3
myStuff.loadSquads(year=1950, seed=23, levels=playerCount)

for squad in myStuff.squads:
    for p in squad.pieces:
        p.place = None

starts = myStuff.find_starts(n=playerCount, ring_radius=3, min_distance=6)
for i, (squad, start_idx) in enumerate(zip(myStuff.squads, starts)):
    king = next(p for p in squad.pieces if p.type == PieceType.KING)
    loc  = myStuff.location_by_index(start_idx)
    Move.from_squad(squad, placement=Placement(king, loc)).apply(myStuff)
    king.place.pantry = 100
    squad.assign_strategy(i)  # varied strategies

tp = TurnProcessor(myStuff, myStuff.squads)

# Place initial pawns
for squad in myStuff.squads:
    squad.place_optimal_pawns(myStuff, tp, k=3)

print("=== STRATEGY DEMO ===\n")
print(f"{'Squad':22s} | {'Strategy':10s} | {'Target':12s} | {'Stance':7s} "
      f"| {'Pantry':>6s} | {'Threat':>6s} | {'Queue':>5s} | {'Placed':>6s} | Action")
print("-" * 110)

for round_num in range(15):
    print(f"\n── Turn {round_num} ──")
    for squad in myStuff.squads:
        if not squad.king_alive():
            continue

        # Reconsider / replan
        replanned = squad.reconsider(myStuff)
        ctx = squad.plan_ctx
        placed_count = len([p for p in squad.pieces if p.place is not None])

        # Show queue
        queue_str = ", ".join(f"{e.piece_type.name}({-e.priority:.0f})" 
                              for e in sorted(squad.queue))

        unplaced = [(p.type.name, p.id) for p in squad.pieces if p.place is None]
        print(f"    unplaced pieces: {len(unplaced)} → {unplaced[:5]}")

        # Try to execute one purchase
        action = ""
        move = squad.execute_queue(myStuff)
        if move is not None:
            try:
                summary = tp.resolve_move(squad, move)
                actual = len([p for p in squad.pieces if p.place is not None])
                king_p = next(p for p in squad.pieces if p.type == PieceType.KING and p.place)
                print(f"    ✓ bought {move.purchase.piece_type.name}")
                print(f"placed={actual}, king_pantry={king_p.place.pantry}")
                placed_count = len([p for p in squad.pieces if p.place is not None])
            except MoveError as e:
                action = f"✗ {e}"
                squad.queue.clear()
                squad.plan_ctx = None
        elif replanned:
            action = "replanned (queue empty or no valid placement)"
        else:
            action = "– waiting"

        replan_tag = " [R]" if replanned else ""
        print(f"  {squad.name:22s} | {squad.strategy.name:10s} | "
              f"{squad.target.type.name:12s} | {ctx.stance:7s} | "
              f"{ctx.pantry:6d} | {ctx.threat:6.1f} | {len(squad.queue):5d} | "
              f"{placed_count:6d} | {action}{replan_tag}")
        if queue_str:
            print(f"  {'':22s}   queue: [{queue_str}]")

    # Food/combat round
    moves = {sq: Move() for sq in myStuff.squads if sq.king_alive()}
    tp.play_round(moves)
    for squad in myStuff.squads:
        actual = len([p for p in squad.pieces if p.place is not None])
        print(f"  POST-ROUND {squad.name}: placed={actual}")


    if tp.game_over:
        print(f"\n🏆 Winner: {tp.winner.name if tp.winner else 'draw'}")
        break

print(f"\n{tp.status()}")


@patch
def next_purchase_type(self: Squad, target_type, max_army_size=10):
    

    this needs to make sure we don't starve. should it run the flow network just to make sure?


##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Running the full flow network is overkill here, but the simple `income - upkeep` check is clearly not enough — Swift Sloths went from 7 placed to 0 in one turn. The problem is twofold:

1. **Simple sum ignores connectivity** — a pawn might harvest 5/turn but be disconnected from the king, so that food never reaches the pantry
2. **Drain isn't accounted for** — Swift Sloths had `threat=6.8`, meaning enemies were draining ~7 food/turn on top of upkeep

A middle ground — use the flow network's **actual net delivery to king**, which `SquadFlow` already computes:

```python
@patch
def next_purchase_type(self: Squad, target_type, max_army_size=10):
    placed = [p for p in self.pieces if p.place is not None and p.type != PieceType.KING]
    counts = defaultdict(int)
    for p in placed:
        counts[p.type] += 1
    
    if len(placed) >= max_army_size:
        return None
    
    # Actual net food via flow network (accounts for connectivity + drain)
    flow = SquadFlow.from_squad(self)
    net_food = flow.net_delivery  # food reaching king per turn
    
    # Force pawns until we have real surplus reaching the king
    min_surplus = 3
    if net_food < min_surplus:
        return PieceType.PAWN
    
    mix = TARGET_MIX[target_type]
    target_size = max(len(placed) + 1, 6)
    
    biggest_gap = -999
    best_pt = None
    for pt, ratio in mix.items():
        desired = ratio * target_size
        gap = desired - counts[pt]
        stats = FOOD_TABLE[pt]
        
        # Can the network absorb this piece's diet?
        if net_food < stats.diet + 1:
            continue
        
        if gap > biggest_gap:
            biggest_gap = gap
            best_pt = pt
    
    return best_pt
```

The key changes:

- **`flow.net_delivery`** replaces the naive `income - upkeep` — only counts food that actually reaches the king through connected paths
- **Diet check per piece type** — won't suggest a queen (high diet) if the network can barely sustain what's there
- **Pawn forcing** uses real surplus, not theoretical — if drain is eating your income, you get more pawns before anything else

Does `SquadFlow` already expose a `net_delivery` property, or do we need to add one?

In [ ]:
!cat ../../HexMagic/magic/network.py

So how we redo next purchase?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

`SquadFlow` has `total_surplus` but that's the naive sum — doesn't account for connectivity. The real signal is: **what's the net food of pieces actually connected to the king?** That's easy to compute from what's already there:

```python
@patch
def next_purchase_type(self: Squad, target_type, max_army_size=10):
    placed = [p for p in self.pieces if p.place is not None and p.type != PieceType.KING]
    counts = defaultdict(int)
    for p in placed:
        counts[p.type] += 1
    
    if len(placed) >= max_army_size:
        return None
    
    # Net food from pieces actually connected to king
    flow = SquadFlow.from_squad(self)
    king_comp = flow.king_component
    connected_surplus = sum(
        flow.G.nodes[n]['harvest'] - flow.G.nodes[n]['diet']
        for n in king_comp
    )
    
    # Force pawns until connected network has real surplus
    min_surplus = 3
    if connected_surplus < min_surplus:
        return PieceType.PAWN
    
    mix = TARGET_MIX[target_type]
    target_size = max(len(placed) + 1, 6)
    
    biggest_gap = -999
    best_pt = None
    for pt, ratio in mix.items():
        desired = ratio * target_size
        gap = desired - counts[pt]
        stats = FOOD_TABLE[pt]
        
        # Can the connected network absorb this piece's diet?
        if connected_surplus < stats.diet + 1:
            continue
        
        if gap > biggest_gap:
            biggest_gap = gap
            best_pt = pt
    
    return best_pt
```

Key difference from the broken version:
- Uses `flow.king_component` to only count food from pieces **connected to the king** — disconnected pawns don't count
- `connected_surplus` replaces the non-existent `net_delivery`
- Diet check per piece type still there — won't suggest a queen if surplus barely covers it

The middle one still dies.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The issue is **enemy drain isn't factored into surplus**. Swift Sloths has `threat=6.0` from turn 2, meaning enemies are draining ~6 food/turn from them. But `connected_surplus` only counts `harvest - diet` — it doesn't see the drain coming in. So it happily buys bishops while bleeding out.

Two fixes needed:

**1. Account for incoming drain in `next_purchase_type`:**

```python
@patch
def next_purchase_type(self: Squad, target_type, max_army_size=10):
    placed = [p for p in self.pieces if p.place is not None and p.type != PieceType.KING]
    counts = defaultdict(int)
    for p in placed:
        counts[p.type] += 1
    
    if len(placed) >= max_army_size:
        return None
    
    # Net food from pieces actually connected to king
    flow = SquadFlow.from_squad(self)
    king_comp = flow.king_component
    connected_surplus = sum(
        flow.G.nodes[n]['harvest'] - flow.G.nodes[n]['diet']
        for n in king_comp
    )
    
    # Subtract incoming enemy drain on our pieces
    parts = placed[0].place.parent if placed else None
    if parts:
        for sq in parts.squads:
            if sq is self: continue
            for ep in sq.pieces:
                if ep.place is None or ep.food.drain <= 0: continue
                cone = ep.sight().hexes
                for p in self.pieces:
                    if p.place and p.id in king_comp:
                        idx = parts.hp2i(p.place.location)
                        if idx in cone:
                            connected_surplus -= ep.food.drain
    
    # Force pawns until connected network has real surplus
    min_surplus = 3
    if connected_surplus < min_surplus:
        return PieceType.PAWN
    
    mix = TARGET_MIX[target_type]
    target_size = max(len(placed) + 1, 6)
    
    biggest_gap = -999
    best_pt = None
    for pt, ratio in mix.items():
        desired = ratio * target_size
        gap = desired - counts[pt]
        stats = FOOD_TABLE[pt]
        
        if connected_surplus < stats.diet + 1:
            continue
        
        if gap > biggest_gap:
            biggest_gap = gap
            best_pt = pt
    
    return best_pt
```

**2. Fix the `reconsider` strategy shift** — Swift Sloths has `threat=6.0` but stays `balanced` because the condition is:

```python
if ctx.threat >= 3.0 and self.strategy.name not in ("aggressive", "balanced"):
```

`"balanced"` is excluded! Under heavy drain they should shift to aggressive to counter-drain. Change to:

```python
if ctx.threat >= 3.0 and self.strategy.name != "aggressive":
    new_strategy = PlacementStrategy.AGGRESSIVE
```

The drain-aware surplus will also make them buy pawns instead of bishops when under fire, which buys time to survive.

!cat ../../HexMagic/magic/pieces.py

!cat ../../HexMagic/magic/turns.py

!cat ../../HexMagic/magic/moves.py

So this is a bunch more context to think about our bug from the demo

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Found it! The bug is in `Move.apply()`:

```python
if self.purchase is not None and squad is not None:
    # Creates new piece and appends to squad
    ...
```

The purchase only creates a piece **if `squad` is passed**. But in the demo loop:

```python
move = squad.execute_queue(myStuff)
if move is not None:
    move.apply(myStuff)  # ← squad=None! Purchase silently skipped!
```

So the pantry gets deducted during `Move.from_squad` validation... actually no, `from_squad` only validates. The deduction happens in `apply`, which skips the whole purchase block because `squad is None`.

**The fix** — either:

1. **Quick**: pass squad in the demo: `move.apply(myStuff, squad=squad)`
2. **Better**: use the turn processor which already handles this correctly:
```python
move = squad.execute_queue(myStuff)
if move is not None:
    try:
        summary = tp.resolve_move(squad, move)
        print(f"    ✓ bought {move.purchase.piece_type.name}")
    except MoveError as e:
        ...
```

Option 2 is better because `resolve_move` also runs the food cycle, drain, starvation check, etc. — so the new piece immediately participates in the economy.

better, but we seem to be pawn heavy. At least we are putting pieces on the board

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Progress! The pawn-heavy issue has two likely causes:

**1. `min_pawns` is probably very high.** Check what it computes:

```python
for s in [PlacementStrategy.EXPANSIVE, PlacementStrategy.AGGRESSIVE, 
          PlacementStrategy.BALANCED, PlacementStrategy.TURTLE]:
    print(f"{s.name}: min_pawns={s.min_pawns}, composition={s.target_composition}")
```

If `min_pawns` is like 8+, the pawn bootstrap phase in `replan` never ends because the queue only plans `n=3` at a time, so it takes many turns of pure pawn buying.

**2. Even after min_pawns is met, scoring still favors pawns.** In `replan`, `connect_score = placement_score` uses the raw scorer output — but `_score_pawn` returns just `tiers[idx]` (0–7) while `_score_bishop` returns much larger composites. This means the weighting isn't normalized, so the `PURCHASE_OBJ` weights don't actually create comparable tradeoffs across piece types.

A quick fix for both:

```python
# 1. Cap min_pawns at something reasonable
return max(2, min(5, math.ceil(army_diet / pawn_net)))

# 2. Normalize placement scores before weighting
if hex_scores:
    placement_score = max(hex_scores)
    # Normalize to 0-10 range so piece types are comparable
    placement_score = min(10.0, max(0.0, placement_score / max(1, abs(placement_score)) * 5))
```

Which approach do you want to try first — capping `min_pawns`, or normalizing scores?

## Balancing

In [ ]:
#| export
import optuna

I think we are at a point where we should consider balancing things. It looks functional but not tuned. Thoughts? The purchasing ratios seem ok. Something else?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Good instinct — and you imported optuna, which is exactly the right tool here. There are too many interacting knobs to hand-tune. The key question is: **what's the objective function?**

**What to optimize:** Run N simulated games (e.g. 3-player, 20 turns each) and score by:
- Turns survived (minimum viable)
- Pieces placed at end (army building efficiency)
- King pantry at end (economic health)
- Enemies killed (combat effectiveness)

**Which knobs to tune first** (biggest impact, most uncertain):

| Priority | Parameter | Why |
|---|---|---|
| 🔴 High | `target_composition` per strategy | Determines army shape + min_pawns |
| 🔴 High | `PURCHASE_OBJ` weight vectors | Controls what gets bought when |
| 🟡 Medium | `min_reserve_per_piece` | Too high = never buy, too low = starve |
| 🟡 Medium | `_graduate` thresholds | When to shift gears |
| 🟢 Low | `defend_threat`, `attack_pantry` | Stance thresholds |

I'd suggest starting with just **one strategy at a time** — freeze the other two players as fixed, tune one strategy's `target_composition` + `PURCHASE_OBJ` weights, then rotate. Something like:

```python
def objective(trial):
    # Tune target_composition for EXPANSIVE
    n_bishops = trial.suggest_int("bishops", 2, 6)
    n_pawns = trial.suggest_int("pawns", 2, 6)
    
    # Tune PURCHASE_OBJ weights
    w_food = trial.suggest_float("w_food", 0.5, 5.0)
    w_frontier = trial.suggest_float("w_frontier", 0.5, 5.0)
    # ...
    
    # Run 3 games, return average score
    scores = []
    for seed in range(3):
        score = simulate_game(seed, strategy_overrides=...)
        scores.append(score)
    return np.mean(scores)
```

Want to build the `simulate_game` function first? It's basically your demo loop stripped of print statements and returning a numeric score.

I think I want to tune to see how these strategies are working. so each of them should have a metric
aggressive - we measure how much we are draining from others
turtle = we measure how much we are harvesting
balanced - we measure the net of the harvest vs draining
expansive - we measure hexregion sight.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Those metrics map perfectly to each strategy's intent. One quick clarification before we build the objective:

Should each strategy be measured **in isolation** (e.g. run a game with all-aggressive squads and measure their average drain), or **in competition** (a mixed game where each strategy competes, and you measure how well each one achieves its metric relative to the others)?

The competition version is more realistic but noisier. The isolation version is cleaner for tuning but might over-optimize for homogeneous games.

I think we need a coarse and fine tuning versions. So we can start in isolation of two teams with the same strategy and ultimately work up to the 4x4 matrix of the strategies. The thing is right now we don't know whether they work at all. Alternatively we could measure against the opposite:
1. Aggresive vs  Turtle.
2. expansive vs balanced.

I do think getting things working in Aggressive vs Turtle would give us the best first datapoint.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

That's a smart approach — Aggressive vs Turtle is the ideal first test because they're the most opposing strategies (drain vs harvest), so if the metrics are working you should see a clear signal.

The trial structure would be:

- **1 game** = Aggressive squad vs Turtle squad, N turns
- **Aggressive metric** = total drain dealt over the game
- **Turtle metric** = total food harvested over the game
- **Optuna objective** = maximize the *gap* between each strategy achieving its own metric vs the other

Before we build the trial, two quick questions:

1. How many turns should a trial game run? (or should it run until one king dies?)
2. What parameters do you actually want Optuna to tune — the `PlacementStrategy` fields like `min_reserve_per_piece`, `attack_pantry`, `target_composition` counts? Or the `PURCHASE_OBJ` weights?

I think each game should have four cycles through the placement of the suggested (unless someone dies). We should have a reconsideration (unless someone dies). We should have a reconsider that doesn't switch strategies just to make sure.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Good constraints. Just to make sure I understand:

- **4 placement cycles** = each squad gets 4 purchase opportunities (so ~4 pieces bought per squad, assuming no deaths)
- **reconsider runs** but with strategy switching disabled — so it can update `target`, `stance`, and replan the queue, but won't graduate or reactively shift to a different `PlacementStrategy`

Is that right? And should `reconsider` still update the `SquadTarget` (e.g. switch from CONSOLIDATE to KING if an enemy comes into sight), just not the strategy itself?

so if we look at the stragetgy
```python

PlacementStrategy.AGGRESSIVE = PlacementStrategy(
    "aggressive", defend_threat=1, attack_pantry=40, attack_placed=3,
    knight_roi_mult=0.7, min_reserve_per_piece=3, min_budget_ratio=0.6, buy_cooldown=0,
    target_composition={PieceType.KNIGHT: 2, PieceType.BISHOP: 1})
```
we see that it wants to place 2 knights and a bishop for its target_composition. so that would give us 3 pieces (non pawn) for what I am calling a cycle. we would want 4 cycles so at least 12 rounds (if no pawns were placed). figuring out army composition. num queens, num bishops etc for these strategies is what I am thinking about for optuna. as well as how do we weight them for the priority queue to be placed. 

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

That's a clean framing. So Optuna would tune two things per strategy:

1. **`target_composition`** — how many of each piece type (knights, bishops, queens) per cycle
2. **`PURCHASE_OBJ` weights** — how the queue prioritizes food/drain/connect/frontier/threat when deciding what to buy next

And the trial structure becomes:
- 1 cycle = place all pieces in `target_composition` once
- 4 cycles = army builds up over time, graduating and rebuilding
- ~12-20 turns depending on composition size

Before we wire up Optuna, it's worth sanity-checking the current `target_composition` for AGGRESSIVE manually. With `{KNIGHT: 2, BISHOP: 1}` and `min_pawns` computed from army diet — how many pawns is it currently requiring? That'll tell us if the pawn-heavy problem is in `min_pawns` math or in the queue weighting.

```python
print(f"AGGRESSIVE min_pawns: {PlacementStrategy.AGGRESSIVE.min_pawns}")
print(f"TURTLE min_pawns: {PlacementStrategy.TURTLE.min_pawns}")
```

Can you run that?

from board needs we select a strategy with perhaps some extra parameters using pyomo?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yes — it maps cleanly to a linear scoring model:

```
score(strategy) = W[strategy] · BoardNeeds_vector
best_strategy = argmax(scores)
```

Pyomo could solve for the **optimal weights** `W` given a training objective (e.g. "aggressive should win against turtle when threat is high"). But Optuna might be simpler here since:

- Pyomo needs explicit constraints + a known optimal solution to fit against
- Optuna just needs a **simulation score** — run N games, measure diversity/competitiveness, tune weights

The key design question before either: **what's your training signal?** What does a "good" strategy selection look like in a game outcome you can measure?

perhaps distance to opponent?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Great addition — distance to nearest enemy king is a natural signal that bridges `BoardNeeds` and strategy choice:

- **Close enemy** → aggressive or turtle (fight or fortify)
- **Far enemy** → expansive (spread out, build engine)
- **Mid-range** → balanced (prepare)

You could add it to `BoardNeeds` as `enemy_king_dist: float` — normalized so 0 = adjacent, 1 = far corner.

Does that feel like the right way to capture it, or would you want something coarser like "enemy in sight / not in sight"?

So we need something outside the system to measure against. so we tune and then check against distance?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yes — the loop would be:

1. **Tune** weights (Optuna tweaks `W` matrix)
2. **Simulate** N games with those weights
3. **Measure** outcomes — e.g. did aggressive squads close distance faster? Did expansive squads stay far and build bigger armies?
4. **Score** the trial and repeat

The tricky part is defining the objective function. A few options:

- **Strategy coherence** — does aggressive actually end up closer to enemies than expansive?
- **Win diversity** — does each strategy win roughly equally often across many games?
- **Both** — weighted combination

Which feels more important to nail first?

In [ ]:
print(f"AGGRESSIVE min_pawns: {PlacementStrategy.AGGRESSIVE.min_pawns}")
print(f"TURTLE min_pawns: {PlacementStrategy.TURTLE.min_pawns}")


min_pawns is also a made up number so we could have it as a variable.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Exactly — `min_pawns=28` is the root cause of the pawn-heavy problem. The `min_pawns` math is computing diet/net wrong.

Two fixes:

1. **Short term**: cap it — `return max(2, min(5, math.ceil(army_diet / pawn_net)))`
2. **Long term**: make it an explicit Optuna parameter per strategy, like `target_composition` — so the tuner discovers the right pawn floor organically

Which do you want to do first?

Lets do the long term. right now we are trying to figure out army composition and so pawns are going to be part of all of this.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Perfect — so `min_pawns` becomes just another integer parameter in `target_composition`, sitting alongside knights/bishops/queens:

```python
target_composition={
    PieceType.PAWN: 3,      # ← explicit, tunable
    PieceType.KNIGHT: 2, 
    PieceType.BISHOP: 1
}
```

And drop the computed `min_pawns` property entirely — `replan` just reads `target_composition[PieceType.PAWN]` directly.

Does that feel right before we wire it into Optuna?

So the purpose of min pawns is to prevent starvation. I think making sure we have enough enables tuning. so it is weighted differently.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Right — pawns are a **constraint** (don't starve), not an **objective** (what to build). So the separation would be:

- `min_pawns` = safety floor, computed from the rest of `target_composition` to guarantee net-positive food
- Everything else in `target_composition` = what Optuna tunes

That way Optuna can't accidentally tune itself into starvation. The `min_pawns` formula just needs fixing — what's wrong currently is it's using `pawn_net = harvest * 1 - diet` at tier 1, but real pawns are placed at varying tiers. Can you print `FOOD_TABLE[PieceType.PAWN]` so we can see the actual numbers?

In [ ]:

FOOD_TABLE[PieceType.PAWN] 

Make sense?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yes! With `harvest=1.25, diet=1`, pawn net at tier 1 = `1.25 * 1 - 1 = 0.25` per turn. That's tiny, which is why `min_pawns` explodes to 28 — it's computing against the diet of the whole army.

The fix: compute `min_pawns` against **average tier** (say tier 2), not tier 1:

```python
pawn_net = pawn_stats.harvest * 2 - pawn_stats.diet  # = 1.5 per turn at tier 2
```

That would bring AGGRESSIVE down to roughly `ceil(army_diet / 1.5)` which is much more reasonable. Want to check what the army diets are for each strategy first?

In [ ]:
FOOD_TABLE

!cat ../../HexMagic/magic/board.py
!cat ../../HexMagic/magic/strategy.py
!cat ../../HexMagic/magic/piece.py
!cat ../../HexMagic/magic/turns.py

In [ ]:
#| export
def OptimalPlacementOverlay(top_n: int = 3, **kw) -> OverlaySpec:
    """Show top-N candidate hexes per piece type using flag-colored chess pieces.
    
    Improvements:
    - Uses piece_candidates for fast filtering
    - Filters by affordability
    - Highlights recommended piece type from next_purchase_type
    - Shows facing wedge for best direction
    - Shows numeric score label
    """
    
    def render(ctx) -> str:
        grid = ctx.grid
        N = len(grid.hexes)
        parts_list = []
        
        for squad in ctx.squads:
            placed = [p for p in squad.pieces if p.place is not None]
            if not placed: continue
            parts_obj = placed[0].place.parent
            terr = parts_obj.terr
            flag = squad.flag

            king = next((p for p in placed if p.type == PieceType.KING), None)
            pantry = king.place.pantry if king else 0

            current_visible = set()
            for ally in placed:
                current_visible |= ally.sight().hexes
            land_set = {i for i in range(len(terr.elevations)) if terr.elevations[i] > 0}
            occupied = {parts_obj.hp2i(p.place.location)
                        for sq in parts_obj.squads for p in sq.pieces if p.place is not None}

            # What would the AI actually buy next?
            try:
                recommended_pt = squad.next_purchase_type(squad.target.type)
            except Exception:
                recommended_pt = None

            for pt, scorer in PLACEMENT_SCORERS.items():
                # Skip if can't afford
                cost = PURCHASE_COST.get(pt, 999)
                if cost > pantry:
                    continue

                # Skip if no unplaced pieces of this type
                unplaced = [p for p in squad.pieces if p.type == pt and p.place is None]
                if not unplaced:
                    continue

                # Fast candidate set
                cands = piece_candidates(pt, king.place.location, parts_obj,
                                         current_visible, occupied, land_set) if king else []
                if not cands:
                    continue

                # Score and rank candidates, also tracking best facing
                dirs = HexPosition.directions()
                scored = []
                for idx in cands:
                    s = scorer(idx, parts_obj, squad, current_visible, occupied, land_set)
                    if s <= -999:
                        continue
                    # Find best facing for this hex
                    hp = parts_obj.i2hp(idx)
                    loc = parts_obj.location_by_index(idx)
                    stats = FOOD_TABLE[pt]
                    eff_sight = max(2, loc.level) + stats.bonus_sight
                    best_facing = max(
                        dirs,
                        key=lambda d: len(
                            {parts_obj.hp2i(h) for h in hp.field_of_view(d, eff_sight)}
                            & current_visible
                        )
                    )
                    scored.append((idx, s, best_facing))

                if not scored:
                    continue

                scored.sort(key=lambda x: -x[1])
                top = scored[:top_n]

                scale = grid.radius * 0.7 / 22.5
                is_recommended = (pt == recommended_pt)

                for rank, (idx, score, best_facing) in enumerate(top):
                    # Recommended type is full opacity; others are dimmed
                    base_opacity = (0.85 - rank * 0.15) if is_recommended else (0.35 - rank * 0.08)
                    base_opacity = max(0.1, base_opacity)

                    for fi in ctx.fine_indices(idx):
                        if 0 <= fi < N:
                            c = grid.hexes[fi].center
                            r = grid.radius * 0.6

                            # Highlight ring — solid for recommended, dashed for others
                            dash = "" if is_recommended else 'stroke-dasharray="3,2"'
                            sw = 2.5 if is_recommended else 1.5
                            parts_list.append(
                                f'<circle cx="{c.x:.1f}" cy="{c.y:.1f}" r="{r:.1f}" '
                                f'fill="{flag.lightPrimary}" fill-opacity="{base_opacity * 0.4:.2f}" '
                                f'stroke="{flag.darkPrimary}" stroke-opacity="{base_opacity:.2f}" '
                                f'stroke-width="{sw}" {dash}/>')

                            # Facing wedge — show direction the piece would look
                            fd = best_facing
                            fx = c.x + fd.q * r * 0.7
                            fy = c.y + (fd.r - fd.s) * r * 0.4
                            parts_list.append(
                                f'<line x1="{c.x:.1f}" y1="{c.y:.1f}" '
                                f'x2="{fx:.1f}" y2="{fy:.1f}" '
                                f'stroke="{flag.primary}" stroke-opacity="{base_opacity:.2f}" '
                                f'stroke-width="2" stroke-linecap="round"/>')

                            # Chess piece silhouette
                            svg_str, pat_def = flag.piece_svg(
                                pt, MapCord(c.x, c.y),
                                scale=scale, size='board',
                                piece_id=f"opt_{squad.name}_{pt.value}_{rank}_{fi}")
                            if pat_def is not None:
                                ctx.builder.add_definition(pat_def)
                            parts_list.append(
                                f'<g opacity="{base_opacity:.2f}">{svg_str}</g>')

                            # Rank badge (top-right)
                            parts_list.append(
                                f'<text x="{c.x + r * 0.7:.1f}" y="{c.y - r * 0.7:.1f}" '
                                f'text-anchor="middle" dominant-baseline="central" '
                                f'font-size="{max(9, grid.radius * 0.35):.0f}" '
                                f'fill="{flag.darkPrimary}" font-weight="bold" '
                                f'opacity="{base_opacity:.2f}">{rank + 1}</text>')

                            # Score label (bottom) — useful during tuning
                            parts_list.append(
                                f'<text x="{c.x:.1f}" y="{c.y + r * 1.1:.1f}" '
                                f'text-anchor="middle" dominant-baseline="central" '
                                f'font-size="{max(7, grid.radius * 0.28):.0f}" '
                                f'fill="{flag.darkPrimary}" '
                                f'opacity="{base_opacity * 0.8:.2f}">{score:.0f}</text>')

                            # "AI pick" star for top recommended hex
                            if is_recommended and rank == 0:
                                parts_list.append(
                                    f'<text x="{c.x - r * 0.7:.1f}" y="{c.y - r * 0.7:.1f}" '
                                    f'text-anchor="middle" dominant-baseline="central" '
                                    f'font-size="{max(10, grid.radius * 0.4):.0f}" '
                                    f'opacity="{base_opacity:.2f}">⭐</text>')

        return '\n'.join(parts_list)

    return OverlaySpec("optimal_placement", render, requires={'squads'}, priority=75)


Is there anything we should do to improve OPtimalPlacementOverlay?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

A few things jump out:

**1. It's slow — iterates all visible hexes instead of using `piece_candidates`:**
```python
# Current: scores every visible hex
for idx in current_visible:
    s = scorer(idx, ...)

# Better: use the pre-filtered candidate set
cands = piece_candidates(pt, king.place.location, parts_obj,
                         current_visible, occupied, land_set)
for idx in cands:
    s = scorer(idx, ...)
```

**2. It shows pieces the squad can't afford.** Filter by pantry:
```python
king = next((p for p in placed if p.type == PieceType.KING), None)
pantry = king.place.pantry if king else 0

for pt, scorer in PLACEMENT_SCORERS.items():
    cost = PURCHASE_COST.get(pt, 999)
    if cost > pantry: continue  # can't afford, don't show
```

**3. It doesn't reflect what the AI would actually buy.** The overlay shows all piece types equally, but `next_purchase_type` might only want pawns right now. You could highlight the recommended type differently:
```python
recommended_pt = squad.next_purchase_type(squad.target.type)
# Use full opacity for recommended, dim for others
base_opacity = 0.8 if pt == recommended_pt else 0.3
```

**4. No facing indicator.** The scorers pick the best facing internally but don't expose it. The overlay just shows a hex with no direction — adding a small facing wedge would show *where* the piece would look.

**5. No score label.** During tuning, seeing the actual numeric score next to each candidate would help spot when scoring is broken (e.g. all bishops scoring identically at 233).

The biggest wins are #1 (performance), #2 (relevance), and #3 (showing what the AI actually thinks). Want me to rewrite it with those changes?

Please rewrite

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

```python
def OptimalPlacementOverlay(top_n: int = 3, **kw) -> OverlaySpec:
    """Show top-N candidate hexes per piece type using flag-colored chess pieces.
    
    Improvements:
    - Uses piece_candidates for fast filtering
    - Filters by affordability
    - Highlights recommended piece type from next_purchase_type
    - Shows facing wedge for best direction
    - Shows numeric score label
    """
    
    def render(ctx) -> str:
        grid = ctx.grid
        N = len(grid.hexes)
        parts_list = []
        
        for squad in ctx.squads:
            placed = [p for p in squad.pieces if p.place is not None]
            if not placed: continue
            parts_obj = placed[0].place.parent
            terr = parts_obj.terr
            flag = squad.flag

            king = next((p for p in placed if p.type == PieceType.KING), None)
            pantry = king.place.pantry if king else 0

            current_visible = set()
            for ally in placed:
                current_visible |= ally.sight().hexes
            land_set = {i for i in range(len(terr.elevations)) if terr.elevations[i] > 0}
            occupied = {parts_obj.hp2i(p.place.location)
                        for sq in parts_obj.squads for p in sq.pieces if p.place is not None}

            # What would the AI actually buy next?
            try:
                recommended_pt = squad.next_purchase_type(squad.target.type)
            except Exception:
                recommended_pt = None

            for pt, scorer in PLACEMENT_SCORERS.items():
                # Skip if can't afford
                cost = PURCHASE_COST.get(pt, 999)
                if cost > pantry:
                    continue

                # Skip if no unplaced pieces of this type
                unplaced = [p for p in squad.pieces if p.type == pt and p.place is None]
                if not unplaced:
                    continue

                # Fast candidate set
                cands = piece_candidates(pt, king.place.location, parts_obj,
                                         current_visible, occupied, land_set) if king else []
                if not cands:
                    continue

                # Score and rank candidates, also tracking best facing
                dirs = HexPosition.directions()
                scored = []
                for idx in cands:
                    s = scorer(idx, parts_obj, squad, current_visible, occupied, land_set)
                    if s <= -999:
                        continue
                    # Find best facing for this hex
                    hp = parts_obj.i2hp(idx)
                    loc = parts_obj.location_by_index(idx)
                    stats = FOOD_TABLE[pt]
                    eff_sight = max(2, loc.level) + stats.bonus_sight
                    best_facing = max(
                        dirs,
                        key=lambda d: len(
                            {parts_obj.hp2i(h) for h in hp.field_of_view(d, eff_sight)}
                            & current_visible
                        )
                    )
                    scored.append((idx, s, best_facing))

                if not scored:
                    continue

                scored.sort(key=lambda x: -x[1])
                top = scored[:top_n]

                scale = grid.radius * 0.7 / 22.5
                is_recommended = (pt == recommended_pt)

                for rank, (idx, score, best_facing) in enumerate(top):
                    # Recommended type is full opacity; others are dimmed
                    base_opacity = (0.85 - rank * 0.15) if is_recommended else (0.35 - rank * 0.08)
                    base_opacity = max(0.1, base_opacity)

                    for fi in ctx.fine_indices(idx):
                        if 0 <= fi < N:
                            c = grid.hexes[fi].center
                            r = grid.radius * 0.6

                            # Highlight ring — solid for recommended, dashed for others
                            dash = "" if is_recommended else 'stroke-dasharray="3,2"'
                            sw = 2.5 if is_recommended else 1.5
                            parts_list.append(
                                f'<circle cx="{c.x:.1f}" cy="{c.y:.1f}" r="{r:.1f}" '
                                f'fill="{flag.lightPrimary}" fill-opacity="{base_opacity * 0.4:.2f}" '
                                f'stroke="{flag.darkPrimary}" stroke-opacity="{base_opacity:.2f}" '
                                f'stroke-width="{sw}" {dash}/>')

                            # Facing wedge — show direction the piece would look
                            fd = best_facing
                            fx = c.x + fd.q * r * 0.7
                            fy = c.y + (fd.r - fd.s) * r * 0.4
                            parts_list.append(
                                f'<line x1="{c.x:.1f}" y1="{c.y:.1f}" '
                                f'x2="{fx:.1f}" y2="{fy:.1f}" '
                                f'stroke="{flag.primary}" stroke-opacity="{base_opacity:.2f}" '
                                f'stroke-width="2" stroke-linecap="round"/>')

                            # Chess piece silhouette
                            svg_str, pat_def = flag.piece_svg(
                                pt, MapCord(c.x, c.y),
                                scale=scale, size='board',
                                piece_id=f"opt_{squad.name}_{pt.value}_{rank}_{fi}")
                            if pat_def is not None:
                                ctx.builder.add_definition(pat_def)
                            parts_list.append(
                                f'<g opacity="{base_opacity:.2f}">{svg_str}</g>')

                            # Rank badge (top-right)
                            parts_list.append(
                                f'<text x="{c.x + r * 0.7:.1f}" y="{c.y - r * 0.7:.1f}" '
                                f'text-anchor="middle" dominant-baseline="central" '
                                f'font-size="{max(9, grid.radius * 0.35):.0f}" '
                                f'fill="{flag.darkPrimary}" font-weight="bold" '
                                f'opacity="{base_opacity:.2f}">{rank + 1}</text>')

                            # Score label (bottom) — useful during tuning
                            parts_list.append(
                                f'<text x="{c.x:.1f}" y="{c.y + r * 1.1:.1f}" '
                                f'text-anchor="middle" dominant-baseline="central" '
                                f'font-size="{max(7, grid.radius * 0.28):.0f}" '
                                f'fill="{flag.darkPrimary}" '
                                f'opacity="{base_opacity * 0.8:.2f}">{score:.0f}</text>')

                            # "AI pick" star for top recommended hex
                            if is_recommended and rank == 0:
                                parts_list.append(
                                    f'<text x="{c.x - r * 0.7:.1f}" y="{c.y - r * 0.7:.1f}" '
                                    f'text-anchor="middle" dominant-baseline="central" '
                                    f'font-size="{max(10, grid.radius * 0.4):.0f}" '
                                    f'opacity="{base_opacity:.2f}">⭐</text>')

        return '\n'.join(parts_list)

    return OverlaySpec("optimal_placement", render, requires={'squads'}, priority=75)
```

Key changes:
- **`piece_candidates`** replaces full visible-hex scan — much faster
- **Affordability filter** — skips piece types the squad can't buy
- **`next_purchase_type` integration** — recommended type shown at full opacity, others dimmed
- **Facing wedge** — line from center toward best facing direction
- **Score label** — numeric score shown below each hex for tuning visibility
- **⭐ marker** on the single top AI-recommended hex

In [ ]:
def run_trial(parts: GameParts, 
              agg_comp: dict[PieceType, int],
              turtle_comp: dict[PieceType, int],
              agg_weights: tuple[float, ...],
              turtle_weights: tuple[float, ...],
              starting_pantry: int = 100,
              seed: int = 42,
              verbose: bool = False) -> dict:
    """Run one Aggressive vs Turtle game. Returns metrics dict."""
    
    # --- Cache oasis (expensive) ---
    if not hasattr(parts, '_oasis_idxs'):
        parts._oasis_idxs = parts.fy.find_oasis(3, min_distance=6)
    
    # --- Reset squads on existing board ---
    parts.squads.clear()
    parts.loadSquads(year=1950, seed=seed, levels=2)
    
    for squad in parts.squads:
        for p in squad.pieces:
            p.place = None
    
    starts = parts.find_starts(n=2, ring_radius=3, min_distance=9)
    for squad, start_idx in zip(parts.squads, starts):
        king = next(p for p in squad.pieces if p.type == PieceType.KING)
        loc = parts.location_by_index(start_idx)
        Move.from_squad(squad, placement=Placement(king, loc)).apply(parts)
        king.place.pantry = starting_pantry
    
    agg_squad, turtle_squad = parts.squads[0], parts.squads[1]
    
    # Assign strategies with trial compositions
    agg_strategy = PlacementStrategy(
        "aggressive", defend_threat=1, attack_pantry=40, attack_placed=3,
        knight_roi_mult=0.7, min_reserve_per_piece=3, min_budget_ratio=0.6,
        buy_cooldown=0, target_composition=agg_comp, graduation="random")
    
    turtle_strategy = PlacementStrategy(
        "turtle", defend_threat=1, attack_pantry=999, attack_placed=99,
        min_reserve_per_piece=4, min_budget_ratio=1.0,
        buy_cooldown=0, target_composition=turtle_comp, graduation="balanced")
    
    agg_squad.strategy = agg_strategy
    turtle_squad.strategy = turtle_strategy
    
    # Override PURCHASE_OBJ weights for this trial
    trial_weights = {
        "aggressive": {
            "expand": agg_weights, "defend": agg_weights, "attack": agg_weights,
        },
        "turtle": {
            "expand": turtle_weights, "defend": turtle_weights, "attack": turtle_weights,
        },
    }
    
    tp = TurnProcessor(parts, parts.squads)
    
    # --- Bootstrap pawns ---
    for squad in parts.squads:
        squad.place_optimal_pawns(parts, tp, k=3)
    
    # --- Compute max rounds ---
    total_pieces = sum(agg_comp.values()) + sum(turtle_comp.values())
    max_rounds = total_pieces * 4 + 20
    
    # --- Accumulators ---
    total_drain = 0.0
    total_harvest = 0.0
    purchases = {'aggressive': 0, 'turtle': 0}
    purchase_log = []  # (turn, squad_name, piece_type)
    errors = 0
    
    # Temporarily patch PURCHASE_OBJ
    old_obj = dict(PURCHASE_OBJ)
    PURCHASE_OBJ.update(trial_weights)
    
    try:
        for turn in range(max_rounds):
            if tp.game_over:
                break
            
            for squad in parts.squads:
                if not squad.king_alive():
                    continue
                
                tag = 'aggressive' if squad is agg_squad else 'turtle'
                
                # Reconsider but don't switch strategy
                old_strat = squad.strategy
                squad.reconsider(parts)
                squad.strategy = old_strat  # pin strategy
                
                # Execute one purchase
                move = squad.execute_queue(parts)
                if move is not None:
                    try:
                        summary = tp.resolve_move(squad, move)
                        purchases[tag] += 1
                        pt_name = move.purchase.piece_type.name if move.purchase else "?"
                        purchase_log.append((turn, tag, pt_name))
                        
                        if squad is agg_squad:
                            total_drain += summary.get('food_drain_dealt', 0)
                        else:
                            total_harvest += summary.get('food_harvested', 0)
                        
                        if verbose:
                            placed = len([p for p in squad.pieces if p.place is not None])
                            print(f"  T{turn} {tag}: ✓ {pt_name} (placed={placed})")
                    except MoveError as e:
                        errors += 1
                        if verbose:
                            print(f"  T{turn} {tag}: ✗ {e}")
                        squad.queue.clear()
                        squad.plan_ctx = None
                elif verbose and squad.queue:
                    print(f"  T{turn} {tag}: queue={len(squad.queue)} but execute returned None")
            
            # Food/combat round
            if tp.game_over:
                break
            
            alive = {sq: Move() for sq in parts.squads if sq.king_alive()}
            if not alive:
                break
            
            try:
                summaries = tp.play_round(alive)
            except MoveError:
                errors += 1
                break
            
            for s in summaries:
                if s['squad'] == agg_squad.name:
                    total_drain += s.get('food_drain_dealt', 0)
                elif s['squad'] == turtle_squad.name:
                    total_harvest += s.get('food_harvested', 0)
    finally:
        PURCHASE_OBJ.clear()
        PURCHASE_OBJ.update(old_obj)
    
    # --- Metrics ---
    agg_alive = agg_squad.king_alive()
    turtle_alive = turtle_squad.king_alive()
    agg_placed = len([p for p in agg_squad.pieces if p.place is not None])
    turtle_placed = len([p for p in turtle_squad.pieces if p.place is not None])
    agg_king = next((p for p in agg_squad.pieces 
                     if p.type == PieceType.KING and p.place is not None), None)
    trt_king = next((p for p in turtle_squad.pieces 
                     if p.type == PieceType.KING and p.place is not None), None)
    
    return {
        'agg_drain': total_drain,
        'turtle_harvest': total_harvest,
        'agg_alive': agg_alive,
        'turtle_alive': turtle_alive,
        'agg_placed': agg_placed,
        'turtle_placed': turtle_placed,
        'agg_pantry': agg_king.place.pantry if agg_king else 0,
        'turtle_pantry': trt_king.place.pantry if trt_king else 0,
        'turns': tp.turn,
        'game_over': tp.game_over,
        'purchases': purchases,
        'purchase_log': purchase_log,
        'errors': errors,
    }


In [ ]:
# --- Diagnostic: single run with defaults ---
board = GameParts()
board.grid.adjustRadius(20)

result = run_trial(board, 
    agg_comp={PieceType.PAWN: 3, PieceType.KNIGHT: 2, PieceType.BISHOP: 1},
    turtle_comp={PieceType.PAWN: 4, PieceType.BISHOP: 3},
    agg_weights=(0.5, 3.0, 1.0, 1.0, 0.5),
    turtle_weights=(3.0, 0.3, 4.0, 1.0, 1.0),
    seed=42, verbose=True)

print(f"Turns played: {result['turns']}")
print(f"Game over:    {result['game_over']}")
print(f"")
print(f"AGGRESSIVE:   alive={result['agg_alive']}  placed={result['agg_placed']}  "
      f"pantry={result['agg_pantry']}  drain_dealt={result['agg_drain']:.1f}")
print(f"TURTLE:       alive={result['turtle_alive']}  placed={result['turtle_placed']}  "
      f"pantry={result['turtle_pantry']}  harvested={result['turtle_harvest']:.1f}")


In [ ]:
# --- Build board once ---
board = GameParts()
board.grid.adjustRadius(20)

def objective(trial: optuna.Trial) -> tuple[float, float]:
    """Multi-objective: maximize agg_drain AND turtle_harvest."""
    
    # --- Aggressive composition ---
    agg_pawns   = trial.suggest_int("agg_pawns", 2, 5)
    agg_knights = trial.suggest_int("agg_knights", 1, 4)
    agg_bishops = trial.suggest_int("agg_bishops", 0, 3)
    agg_queens  = trial.suggest_int("agg_queens", 0, 2)
    agg_comp = {PieceType.PAWN: agg_pawns, PieceType.KNIGHT: agg_knights,
                PieceType.BISHOP: agg_bishops, PieceType.QUEEN: agg_queens}
    
    # --- Turtle composition ---
    trt_pawns   = trial.suggest_int("trt_pawns", 3, 6)
    trt_bishops = trial.suggest_int("trt_bishops", 1, 5)
    trt_queens  = trial.suggest_int("trt_queens", 0, 2)
    trt_comp = {PieceType.PAWN: trt_pawns, PieceType.BISHOP: trt_bishops,
                PieceType.QUEEN: trt_queens}
    
    # --- Queue priority weights (food, drain, connect, frontier, threat) ---
    agg_w = tuple(trial.suggest_float(f"agg_w{i}", 0.1, 5.0) for i in range(5))
    trt_w = tuple(trial.suggest_float(f"trt_w{i}", 0.1, 5.0) for i in range(5))
    
    # --- Run across multiple seeds for stability ---
    agg_drains, trt_harvests = [], []
    for seed in range(3):
        result = run_trial(board, agg_comp, trt_comp, agg_w, trt_w, seed=seed) #,verbose=True)
        agg_drains.append(result['agg_drain'])
        trt_harvests.append(result['turtle_harvest'])
    
    return np.mean(agg_drains), np.mean(trt_harvests)

study = optuna.create_study(
    directions=["maximize", "maximize"],
    study_name="agg_vs_turtle_v1",
)
study.optimize(objective, n_trials=50)


We have some code that shows how the game progresses over turns and I think we could make this into some sort of a class that we could use to evaluate strategies
```python
@patch
def _chart_controls(GS:GlobalGameState):
    """Two toggle rows: columns and squads."""
    # Row 1: metric columns
    col_toggles = DivLAligned(
        *[Button(label,
            cls=(ButtonT.primary if key in GS.active_chart_cols else ButtonT.ghost, "text-xs"),
            hx_post="/toggle_chart_col",
            hx_vals=json.dumps({"col": key}),
            hx_target="#chart-panel")
          for key, label in CHART_COLS],
        cls="gap-1 flex-wrap",
    )

    # Row 2: squads (flag color dots + name)
    squad_toggles = []
    all_squads = GS.parts.squads if GS.parts else []
    active = GS.active_chart_squads or {s.name for s in all_squads}  # empty = all shown

    for sq in all_squads:
        is_on = sq.name in active or not GS.active_chart_squads
        is_player = sq is GS.player_squad
        label = DivLAligned(
            Span("●", style=f"color:{sq.flag.primary}; font-size:1.1em"),
            Small(sq.name.split()[0], cls="font-medium text-xs" if is_player else "text-xs"),
        )
        squad_toggles.append(
            Button(label,
                cls=(ButtonT.primary if is_on else ButtonT.ghost, "text-xs px-2"),
                hx_post="/toggle_chart_squad",
                hx_vals=json.dumps({"squad": sq.name}),
                hx_target="#chart-panel")
        )

    return Div(
        DivLAligned(*squad_toggles, cls="gap-1 flex-wrap"),
        col_toggles,
        cls="space-y-1 pb-2 border-b",
    )

@patch
def panel_charts(GS:GlobalGameState):
    """Time-series sparklines — controlled by active_chart_cols + active_chart_squads."""
    if GS.tp is None or GS.tp.record.df.empty:
        return Div(
            _chart_controls(),
            P("No history yet — complete a turn.", cls=TextPresets.muted_sm),
            id="chart-panel", cls="space-y-3",
        )

    df = GS.tp.record.df
    all_squads = GS.parts.squads
    
    # Which squads to plot
    active_squads = GS.active_chart_squads or {s.name for s in all_squads}
    colors = {sq.name: sq.flag.primary for sq in all_squads}
    
    cols = sorted(GS.active_chart_cols)
    n = len(cols)
    ncols = min(3, n)
    nrows = math.ceil(n / ncols)
    
    fig, axes = plt.subplots(nrows, ncols,
                              figsize=(4.5 * ncols, 3.2 * nrows),
                              facecolor='none', squeeze=False)
    fig.patch.set_alpha(0)

    col_labels = dict(CHART_COLS)

    for i, col in enumerate(cols):
        ax = axes[i // ncols][i % ncols]
        
        for sq in all_squads:
            if sq.name not in active_squads:
                continue
            sub = df[df.squad == sq.name].sort_values('turn')
            if sub.empty or col not in sub.columns:
                continue
            is_player = sq is GS.player_squad
            is_alive = sub.iloc[-1]['alive']
            color = colors[sq.name]
            ax.plot(sub['turn'], sub[col],
                    color=color,
                    lw=2.5 if is_player else 1.2,
                    ls='-' if is_alive else '--',
                    alpha=1.0 if is_player else (0.55 if is_alive else 0.2),
                    marker='o', markersize=3 if is_player else 1.5,
                    label=sq.name.split()[0])
            if is_player and not sub.empty:
                last = sub.iloc[-1]
                ax.annotate(f"{last[col]:.0f}",
                            xy=(last['turn'], last[col]),
                            fontsize=7, color=color,
                            xytext=(4, 0), textcoords='offset points')

        ax.set_title(col_labels.get(col, col), fontsize=10, fontweight='bold')
        ax.tick_params(labelsize=7)
        ax.grid(alpha=0.2, lw=0.5)
        ax.spines[['top', 'right']].set_visible(False)
        ax.set_xlabel("Turn", fontsize=7)

    # Hide unused axes
    for j in range(n, nrows * ncols):
        axes[j // ncols][j % ncols].set_visible(False)

    # Shared legend
    handles, labels = axes[0][0].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels, loc='lower center',
                   ncol=len(active_squads), fontsize=8,
                   framealpha=0.3, bbox_to_anchor=(0.5, -0.02))

    plt.tight_layout(rect=[0, 0.06, 1, 1])

    buf = io.BytesIO()
    fig.savefig(buf, format='svg', bbox_inches='tight', transparent=True)
    plt.close(fig)
    buf.seek(0)

    return Div(
        GS._chart_controls(),
        NotStr(buf.read().decode()),
        id="chart-panel",
        cls="space-y-2",
    )
```

for reference this is GlobalGameState
```python
@dataclass
class GlobalGameState:
    parts: GameParts | None = None
    tp: TurnProcessor | None = None
    player_squad: Squad | None = None
    selected_piece: Piece | None = None
    pending_rotations: list[Rotation] = field(default_factory=list)
    pending_placement: Placement | None = None
    pending_purchase: Purchase | None = None
    pending_purchase_type: PieceType | None = None  # add this
    active_overlays: set = field(default_factory=lambda: { 'rivers','lanterns','network','ocean'})
    turn: int = 0
    radius:float = 25
    debug_overlays: bool = False
    rotationClickId: int = None
    ai_strategies: dict[str, PlacementStrategy] = field(default_factory=dict)
    focused_piece: Piece | None = None
    # Add to GlobalGameState
    active_chart_cols: set = field(default_factory=lambda: {'king_pantry', 'pieces_placed'})
    active_chart_squads: set = field(default_factory=lambda: set())  # empty = all
    
    def select_piece(self, piece: Piece):
        """Select an unplaced piece for placement."""
        if piece.place is not None:
            return  # already on board — rotation handled separately
        self.selected_piece = piece
    
    def place_piece(self, hex_idx: int):
        """Place the selected piece at a hex."""
        if self.selected_piece is None:
            return False
        loc = self.parts.location_by_index(hex_idx)
        self.pending_placement = Placement(self.selected_piece, loc)
        self.selected_piece.place = loc  # preview immediately
        self.selected_piece = None
        return True
    
    def unplace_piece(self, piece: Piece):
        """Remove a pending placement (before committing)."""
        if self.pending_placement and self.pending_placement.piece is piece:
            piece.place = None
            self.pending_placement = None
    
    
    @property
    def is_active(self) -> bool:
        return self.parts is not None and self.tp is not None
    
    @property
    def game_over(self) -> bool:
        return self.tp.game_over if self.tp else False

```

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The key insight is that `panel_charts` already does what you want — it just reads from `tp.record.df`. So a `StrategyEvaluator` class just needs to:

1. Run trials and collect `GameRecord` dataframes
2. Reuse the same chart rendering with a lightweight fake `GlobalGameState`

```python
@dataclass
class StrategyEvaluator:
    """Run strategy trials and visualize results using GameRecord data."""
    board: GameParts
    trials: list[dict] = field(default_factory=list)  # raw result dicts
    records: list[pd.DataFrame] = field(default_factory=list)  # tp.record.df per trial
    
    CHART_COLS = [
        ('king_pantry',    'King Pantry'),
        ('pieces_placed',  'Pieces Placed'),
        ('drain_dealt',    'Drain Dealt'),
        ('food_harvested', 'Harvest'),
        ('bottlenecks',    'Bottlenecks'),
    ]
    
    def run(self, agg_comp, turtle_comp, agg_weights, turtle_weights,
            seeds: list[int] = [42, 43, 44]) -> pd.DataFrame:
        """Run trials across seeds, store results + records."""
        for seed in seeds:
            result = run_trial(
                self.board, agg_comp, turtle_comp,
                agg_weights, turtle_weights, seed=seed)
            self.trials.append(result)
            # Capture the record df if run_trial exposes it
            # (we'll need run_trial to return tp.record.df)
        return self.summary()
    
    def summary(self) -> pd.DataFrame:
        """Aggregate metrics across trials."""
        if not self.trials:
            return pd.DataFrame()
        rows = []
        for i, t in enumerate(self.trials):
            rows.append({
                'trial': i,
                'agg_drain':       t['agg_drain'],
                'turtle_harvest':  t['turtle_harvest'],
                'agg_placed':      t['agg_placed'],
                'turtle_placed':   t['turtle_placed'],
                'agg_pantry':      t['agg_pantry'],
                'turtle_pantry':   t['turtle_pantry'],
                'turns':           t['turns'],
                'agg_alive':       t['agg_alive'],
                'turtle_alive':    t['turtle_alive'],
            })
        df = pd.DataFrame(rows)
        return df
    
    def charts(self, cols: set | None = None, width: int = 900) -> FT:
        """Render time-series charts from stored records."""
        if not self.records:
            return P("No trial records yet — call run() first.")
        
        cols = cols or {'king_pantry', 'pieces_placed', 'drain_dealt', 'food_harvested'}
        col_labels = dict(self.CHART_COLS)
        
        # Stack all records with a trial column
        dfs = []
        for i, df in enumerate(self.records):
            d = df.copy()
            d['trial'] = i
            dfs.append(d)
        all_df = pd.concat(dfs, ignore_index=True)
        
        col_list = sorted(cols)
        ncols = min(3, len(col_list))
        nrows = math.ceil(len(col_list) / ncols)
        
        fig, axes = plt.subplots(nrows, ncols,
                                  figsize=(4.5 * ncols, 3.2 * nrows),
                                  facecolor='none', squeeze=False)
        fig.patch.set_alpha(0)
        
        # One color per squad name across all trials
        squad_names = all_df['squad'].unique()
        palette = plt.cm.tab10.colors
        colors = {name: palette[i % 10] for i, name in enumerate(squad_names)}
        
        for i, col in enumerate(col_list):
            ax = axes[i // ncols][i % ncols]
            if col not in all_df.columns:
                ax.set_visible(False)
                continue
            
            for squad_name in squad_names:
                sub = all_df[all_df.squad == squad_name]
                # Plot mean + shaded std across trials
                grouped = sub.groupby('turn')[col]
                mean = grouped.mean()
                std  = grouped.std().fillna(0)
                turns = mean.index
                color = colors[squad_name]
                
                ax.plot(turns, mean, color=color, lw=2,
                        label=squad_name.split()[0])
                ax.fill_between(turns,
                                mean - std, mean + std,
                                color=color, alpha=0.15)
            
            ax.set_title(col_labels.get(col, col), fontsize=10, fontweight='bold')
            ax.tick_params(labelsize=7)
            ax.grid(alpha=0.2, lw=0.5)
            ax.spines[['top', 'right']].set_visible(False)
            ax.set_xlabel("Turn", fontsize=7)
        
        # Hide unused axes
        for j in range(len(col_list), nrows * ncols):
            axes[j // ncols][j % ncols].set_visible(False)
        
        handles, labels = axes[0][0].get_legend_handles_labels()
        if handles:
            fig.legend(handles, labels, loc='lower center',
                       ncol=len(squad_names), fontsize=8,
                       framealpha=0.3, bbox_to_anchor=(0.5, -0.02))
        
        plt.tight_layout(rect=[0, 0.06, 1, 1])
        buf = io.BytesIO()
        fig.savefig(buf, format='svg', bbox_inches='tight', transparent=True)
        plt.close(fig)
        buf.seek(0)
        
        return Div(NotStr(buf.read().decode()), cls="space-y-2")
    
    def __ft__(self):
        """Render summary table + charts."""
        df = self.summary()
        if df.empty:
            return P("No trials yet.")
        
        mean = df.mean(numeric_only=True)
        
        return Card(
            H4("Strategy Evaluation"),
            Grid(
                Div(H5("Mean Metrics"),
                    *[P(f"{k}: {v:.1f}") for k, v in mean.items()]),
                self.charts(),
            ),
        )
```

The key addition needed is that `run_trial` should return `tp.record.df` so the evaluator can store it:

```python
# Add to run_trial return dict:
return {
    ...existing keys...,
    'record_df': tp.record.df.copy(),  # ← add this
}

# And in StrategyEvaluator.run():
self.records.append(result['record_df'])
```

Then usage in the Optuna loop or standalone:

```python
ev = StrategyEvaluator(board)
ev.run(
    agg_comp={PieceType.PAWN: 5, PieceType.KNIGHT: 3, PieceType.BISHOP: 3},
    turtle_comp={PieceType.PAWN: 6, PieceType.BISHOP: 5, PieceType.QUEEN: 2},
    agg_weights=(0.7, 2.7, 3.7, 4.9, 4.6),
    turtle_weights=(1.6, 4.3, 2.6, 2.4, 3.9),
)
ev  # renders in notebook
```

The mean ± std shading is the key addition over `panel_charts` — it shows stability across seeds, not just one run.

In [ ]:
@dataclass
class StrategyEvaluator:
    """Run strategy trials and visualize results using GameRecord data."""
    board: GameParts
    trials: list[dict] = field(default_factory=list)  # raw result dicts
    records: list[pd.DataFrame] = field(default_factory=list)  # tp.record.df per trial
    
    CHART_COLS = [
        ('king_pantry',    'King Pantry'),
        ('pieces_placed',  'Pieces Placed'),
        ('drain_dealt',    'Drain Dealt'),
        ('food_harvested', 'Harvest'),
        ('bottlenecks',    'Bottlenecks'),
    ]
    
    def run(self, agg_comp, turtle_comp, agg_weights, turtle_weights,
            seeds: list[int] = [42, 43, 44]) -> pd.DataFrame:
        """Run trials across seeds, store results + records."""
        for seed in seeds:
            result = run_trial(
                self.board, agg_comp, turtle_comp,
                agg_weights, turtle_weights, seed=seed)
            self.trials.append(result)
            # Capture the record df if run_trial exposes it
            # (we'll need run_trial to return tp.record.df)
        return self.summary()
    
    def summary(self) -> pd.DataFrame:
        """Aggregate metrics across trials."""
        if not self.trials:
            return pd.DataFrame()
        rows = []
        for i, t in enumerate(self.trials):
            rows.append({
                'trial': i,
                'agg_drain':       t['agg_drain'],
                'turtle_harvest':  t['turtle_harvest'],
                'agg_placed':      t['agg_placed'],
                'turtle_placed':   t['turtle_placed'],
                'agg_pantry':      t['agg_pantry'],
                'turtle_pantry':   t['turtle_pantry'],
                'turns':           t['turns'],
                'agg_alive':       t['agg_alive'],
                'turtle_alive':    t['turtle_alive'],
            })
        df = pd.DataFrame(rows)
        return df
    
    def charts(self, cols: set | None = None, width: int = 900) -> FT:
        """Render time-series charts from stored records."""
        if not self.records:
            return P("No trial records yet — call run() first.")
        
        cols = cols or {'king_pantry', 'pieces_placed', 'drain_dealt', 'food_harvested'}
        col_labels = dict(self.CHART_COLS)
        
        # Stack all records with a trial column
        dfs = []
        for i, df in enumerate(self.records):
            d = df.copy()
            d['trial'] = i
            dfs.append(d)
        all_df = pd.concat(dfs, ignore_index=True)
        
        col_list = sorted(cols)
        ncols = min(3, len(col_list))
        nrows = math.ceil(len(col_list) / ncols)
        
        fig, axes = plt.subplots(nrows, ncols,
                                  figsize=(4.5 * ncols, 3.2 * nrows),
                                  facecolor='none', squeeze=False)
        fig.patch.set_alpha(0)
        
        # One color per squad name across all trials
        squad_names = all_df['squad'].unique()
        palette = plt.cm.tab10.colors
        colors = {name: palette[i % 10] for i, name in enumerate(squad_names)}
        
        for i, col in enumerate(col_list):
            ax = axes[i // ncols][i % ncols]
            if col not in all_df.columns:
                ax.set_visible(False)
                continue
            
            for squad_name in squad_names:
                sub = all_df[all_df.squad == squad_name]
                # Plot mean + shaded std across trials
                grouped = sub.groupby('turn')[col]
                mean = grouped.mean()
                std  = grouped.std().fillna(0)
                turns = mean.index
                color = colors[squad_name]
                
                ax.plot(turns, mean, color=color, lw=2,
                        label=squad_name.split()[0])
                ax.fill_between(turns,
                                mean - std, mean + std,
                                color=color, alpha=0.15)
            
            ax.set_title(col_labels.get(col, col), fontsize=10, fontweight='bold')
            ax.tick_params(labelsize=7)
            ax.grid(alpha=0.2, lw=0.5)
            ax.spines[['top', 'right']].set_visible(False)
            ax.set_xlabel("Turn", fontsize=7)
        
        # Hide unused axes
        for j in range(len(col_list), nrows * ncols):
            axes[j // ncols][j % ncols].set_visible(False)
        
        handles, labels = axes[0][0].get_legend_handles_labels()
        if handles:
            fig.legend(handles, labels, loc='lower center',
                       ncol=len(squad_names), fontsize=8,
                       framealpha=0.3, bbox_to_anchor=(0.5, -0.02))
        
        plt.tight_layout(rect=[0, 0.06, 1, 1])
        buf = io.BytesIO()
        fig.savefig(buf, format='svg', bbox_inches='tight', transparent=True)
        plt.close(fig)
        buf.seek(0)
        
        return Div(NotStr(buf.read().decode()), cls="space-y-2")
    
    def __ft__(self):
        """Render summary table + charts."""
        df = self.summary()
        if df.empty:
            return P("No trials yet.")
        
        mean = df.mean(numeric_only=True)
        
        return Card(
            H4("Strategy Evaluation"),
            Grid(
                Div(H5("Mean Metrics"),
                    *[P(f"{k}: {v:.1f}") for k, v in mean.items()]),
                self.charts(),
            ),
        )


## Optuna Results: Aggressive vs Turtle (50 trials)

### Pareto Front Winners

| Trial | Agg Drain | Turtle Harvest | Agg Comp | Turtle Comp |
|-------|-----------|----------------|----------|-------------|
| 29 | **2051** | **5430** | 5P/3K/3B/0Q | 6P/5B/2Q |
| 11 | 1995 | 5177 | 5P/2K/2B/2Q | 5P/5B/2Q |
| 20 | 1995 | 5177 | 5P/4K/3B/0Q | 6P/3B/2Q |
| 0 | 1923 | 4924 | 3P/2K/3B/2Q | 5P/5B/2Q |

### Clear Composition Signals

**Aggressive wants:**
- **5 pawns** — appears in all top trials. The food engine needs to be big to sustain expensive attackers
- **3+ knights** — the drain machines are doing their job
- **3 bishops** — connectivity backbone
- **Queens are optional** — 0Q appears in the best trial (#29). Queens are expensive and the aggressive strategy gets more from knight swarms

**Turtle wants:**
- **5-6 pawns** — maximum harvest engine
- **5 bishops** — wide sight = wide harvest territory
- **2 queens** — defensive drain to punish anyone who comes close

### Problems Exposed

**1. Outcome quantization** — Results cluster into ~8 discrete values (e.g. `(471, 3511)` appears 8 times). This means:
- With only 3 seeds, the game is too deterministic
- Many different weight configurations produce identical outcomes
- The **composition matters far more than the weights**

**2. Weights aren't differentiating** — Trial 29 (best) and Trial 48 (worst aggressive) both have wildly different weights but the gap is driven by composition, not priority ordering. This suggests `PURCHASE_OBJ` weights aren't actually steering placement decisions effectively.

**3. Oasis recomputation** — `find_oasis` is being called on every `loadSquads`, producing massive output spam. The `_oasis_idxs` cache on `parts` gets cleared when squads reset.

### Recommendations

**Next steps:**
1. **Lock in compositions** from the Pareto front and drop them from the search space
2. **Increase seed count** to 5-7 to reduce quantization
3. **Tune weights separately** with fixed compositions — the current search space (16 params) is too wide for 50 trials to find signal in weights
4. **Silence oasis output** — add a `verbose=False` flag or cache oasis at the board level, not the squad level

**Updated presets would be:**

```python
PlacementStrategy.AGGRESSIVE = PlacementStrategy(
    "aggressive", defend_threat=1, attack_pantry=40, attack_placed=3,
    knight_roi_mult=0.7, min_reserve_per_piece=3, min_budget_ratio=0.6,
    buy_cooldown=0,
    target_composition={PieceType.PAWN: 5, PieceType.KNIGHT: 3, PieceType.BISHOP: 3},
    graduation="random")

PlacementStrategy.TURTLE = PlacementStrategy(
    "turtle", defend_threat=1, attack_pantry=999, attack_placed=99,
    min_reserve_per_piece=4, min_budget_ratio=1.0,
    buy_cooldown=0,
    target_composition={PieceType.PAWN: 6, PieceType.BISHOP: 5, PieceType.QUEEN: 2},
    graduation="balanced")
```

Want to lock these in and run a weights-only tuning pass next?

what about balance versus expansive

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Same framework, different metrics:

- **Balanced** = net food economy (harvest + drain_dealt - drain_received)
- **Expansive** = total unique land hexes visible (sight coverage)

```python
def run_trial_bal_exp(parts: GameParts, 
                      bal_comp: dict[PieceType, int],
                      exp_comp: dict[PieceType, int],
                      bal_weights: tuple[float, ...],
                      exp_weights: tuple[float, ...],
                      starting_pantry: int = 100,
                      seed: int = 42,
                      verbose: bool = False) -> dict:
    """Run one Balanced vs Expansive game. Returns metrics dict."""
    
    if not hasattr(parts, '_oasis_idxs'):
        parts._oasis_idxs = parts.fy.find_oasis(3, min_distance=6)
    
    parts.squads.clear()
    parts.loadSquads(year=1950, seed=seed, levels=2)
    
    for squad in parts.squads:
        for p in squad.pieces:
            p.place = None
    
    starts = parts.find_starts(n=2, ring_radius=3, min_distance=9)
    for squad, start_idx in zip(parts.squads, starts):
        king = next(p for p in squad.pieces if p.type == PieceType.KING)
        loc = parts.location_by_index(start_idx)
        Move.from_squad(squad, placement=Placement(king, loc)).apply(parts)
        king.place.pantry = starting_pantry
    
    bal_squad, exp_squad = parts.squads[0], parts.squads[1]
    
    bal_strategy = PlacementStrategy(
        "balanced", defend_threat=2, attack_pantry=60, attack_placed=4,
        min_reserve_per_piece=5, min_budget_ratio=1.0,
        buy_cooldown=0, target_composition=bal_comp, graduation="aggressive")
    
    exp_strategy = PlacementStrategy(
        "expansive", defend_threat=3, attack_pantry=80, attack_placed=6,
        min_reserve_per_piece=4, min_budget_ratio=0.8,
        buy_cooldown=0, target_composition=exp_comp, graduation="aggressive")
    
    bal_squad.strategy = bal_strategy
    exp_squad.strategy = exp_strategy
    
    trial_weights = {
        "balanced": {"expand": bal_weights, "defend": bal_weights, "attack": bal_weights},
        "expansive": {"expand": exp_weights, "defend": exp_weights, "attack": exp_weights},
    }
    
    tp = TurnProcessor(parts, parts.squads)
    
    for squad in parts.squads:
        squad.place_optimal_pawns(parts, tp, k=3)
    
    total_pieces = sum(bal_comp.values()) + sum(exp_comp.values())
    max_rounds = total_pieces * 4 + 20
    
    # Accumulators
    bal_harvest_total = 0.0
    bal_drain_dealt = 0.0
    bal_drain_received = 0.0
    exp_sight_sum = 0.0  # sum of sight coverage each turn (for averaging)
    exp_sight_samples = 0
    errors = 0
    
    land_set = {i for i in range(len(parts.terr.elevations))
                if parts.terr.elevations[i] > 0}
    
    old_obj = dict(PURCHASE_OBJ)
    PURCHASE_OBJ.update(trial_weights)
    
    try:
        for turn in range(max_rounds):
            if tp.game_over:
                break
            
            for squad in parts.squads:
                if not squad.king_alive():
                    continue
                
                old_strat = squad.strategy
                squad.reconsider(parts)
                squad.strategy = old_strat
                
                move = squad.execute_queue(parts)
                if move is not None:
                    try:
                        summary = tp.resolve_move(squad, move)
                        if squad is bal_squad:
                            bal_harvest_total += summary.get('food_harvested', 0)
                            bal_drain_dealt += summary.get('food_drain_dealt', 0)
                            bal_drain_received += summary.get('food_drain_received', 0)
                    except MoveError:
                        errors += 1
                        squad.queue.clear()
                        squad.plan_ctx = None
            
            if tp.game_over:
                break
            
            # Measure expansive sight before food round
            exp_placed = [p for p in exp_squad.pieces if p.place is not None]
            if exp_placed:
                visible = set()
                for p in exp_placed:
                    visible |= p.sight().hexes
                exp_sight_sum += len(visible & land_set)
                exp_sight_samples += 1
            
            alive = {sq: Move() for sq in parts.squads if sq.king_alive()}
            if not alive:
                break
            
            try:
                summaries = tp.play_round(alive)
            except MoveError:
                errors += 1
                break
            
            for s in summaries:
                if s['squad'] == bal_squad.name:
                    bal_harvest_total += s.get('food_harvested', 0)
                    bal_drain_dealt += s.get('food_drain_dealt', 0)
                    bal_drain_received += s.get('food_drain_received', 0)
    finally:
        PURCHASE_OBJ.clear()
        PURCHASE_OBJ.update(old_obj)
    
    bal_net = bal_harvest_total + bal_drain_dealt - bal_drain_received
    exp_avg_sight = exp_sight_sum / max(1, exp_sight_samples)
    
    return {
        'bal_net': bal_net,
        'bal_harvest': bal_harvest_total,
        'bal_drain_dealt': bal_drain_dealt,
        'bal_drain_received': bal_drain_received,
        'exp_avg_sight': exp_avg_sight,
        'exp_max_sight': exp_sight_sum,  # cumulative
        'bal_alive': bal_squad.king_alive(),
        'exp_alive': exp_squad.king_alive(),
        'bal_placed': len([p for p in bal_squad.pieces if p.place is not None]),
        'exp_placed': len([p for p in exp_squad.pieces if p.place is not None]),
        'turns': tp.turn,
        'errors': errors,
    }
```

```python
def objective_bal_exp(trial: optuna.Trial) -> tuple[float, float]:
    """Multi-objective: maximize bal_net AND exp_avg_sight."""
    
    # Balanced composition
    bal_pawns   = trial.suggest_int("bal_pawns", 3, 6)
    bal_bishops = trial.suggest_int("bal_bishops", 1, 4)
    bal_knights = trial.suggest_int("bal_knights", 0, 3)
    bal_queens  = trial.suggest_int("bal_queens", 0, 2)
    bal_comp = {PieceType.PAWN: bal_pawns, PieceType.BISHOP: bal_bishops,
                PieceType.KNIGHT: bal_knights, PieceType.QUEEN: bal_queens}
    
    # Expansive composition
    exp_pawns   = trial.suggest_int("exp_pawns", 3, 6)
    exp_bishops = trial.suggest_int("exp_bishops", 2, 5)
    exp_knights = trial.suggest_int("exp_knights", 0, 2)
    exp_queens  = trial.suggest_int("exp_queens", 0, 2)
    exp_comp = {PieceType.PAWN: exp_pawns, PieceType.BISHOP: exp_bishops,
                PieceType.KNIGHT: exp_knights, PieceType.QUEEN: exp_queens}
    
    # Weights
    bal_w = tuple(trial.suggest_float(f"bal_w{i}", 0.1, 5.0) for i in range(5))
    exp_w = tuple(trial.suggest_float(f"exp_w{i}", 0.1, 5.0) for i in range(5))
    
    bal_nets, exp_sights = [], []
    for seed in range(5):  # more seeds to reduce quantization
        result = run_trial_bal_exp(board, bal_comp, exp_comp, bal_w, exp_w, seed=seed)
        bal_nets.append(result['bal_net'])
        exp_sights.append(result['exp_avg_sight'])
    
    return np.mean(bal_nets), np.mean(exp_sights)

study2 = optuna.create_study(
    directions=["maximize", "maximize"],
    study_name="bal_vs_exp_v1",
)
study2.optimize(objective_bal_exp, n_trials=50)
```

Key differences from the aggressive/turtle study:
- **Balanced metric** = net economy (harvest + drain dealt - drain received) — captures the "balanced" identity of being good at both
- **Expansive metric** = average unique land hexes visible per turn — directly measures territory coverage
- **5 seeds** instead of 3 to reduce the quantization problem
- **Expansive gets higher bishop range** (2-5) since that's its primary tool

In [ ]:
def run_trial_bal_exp(parts: GameParts, 
                      bal_comp: dict[PieceType, int],
                      exp_comp: dict[PieceType, int],
                      bal_weights: tuple[float, ...],
                      exp_weights: tuple[float, ...],
                      starting_pantry: int = 100,
                      seed: int = 42,
                      verbose: bool = False) -> dict:
    """Run one Balanced vs Expansive game. Returns metrics dict."""
    
    if not hasattr(parts, '_oasis_idxs'):
        parts._oasis_idxs = parts.fy.find_oasis(3, min_distance=6)
    
    parts.squads.clear()
    parts.loadSquads(year=1950, seed=seed, levels=2)
    
    for squad in parts.squads:
        for p in squad.pieces:
            p.place = None
    
    starts = parts.find_starts(n=2, ring_radius=3, min_distance=9)
    for squad, start_idx in zip(parts.squads, starts):
        king = next(p for p in squad.pieces if p.type == PieceType.KING)
        loc = parts.location_by_index(start_idx)
        Move.from_squad(squad, placement=Placement(king, loc)).apply(parts)
        king.place.pantry = starting_pantry
    
    bal_squad, exp_squad = parts.squads[0], parts.squads[1]
    
    bal_strategy = PlacementStrategy(
        "balanced", defend_threat=2, attack_pantry=60, attack_placed=4,
        min_reserve_per_piece=5, min_budget_ratio=1.0,
        buy_cooldown=0, target_composition=bal_comp, graduation="aggressive")
    
    exp_strategy = PlacementStrategy(
        "expansive", defend_threat=3, attack_pantry=80, attack_placed=6,
        min_reserve_per_piece=4, min_budget_ratio=0.8,
        buy_cooldown=0, target_composition=exp_comp, graduation="aggressive")
    
    bal_squad.strategy = bal_strategy
    exp_squad.strategy = exp_strategy
    
    trial_weights = {
        "balanced": {"expand": bal_weights, "defend": bal_weights, "attack": bal_weights},
        "expansive": {"expand": exp_weights, "defend": exp_weights, "attack": exp_weights},
    }
    
    tp = TurnProcessor(parts, parts.squads)
    
    for squad in parts.squads:
        squad.place_optimal_pawns(parts, tp, k=3)
    
    total_pieces = sum(bal_comp.values()) + sum(exp_comp.values())
    max_rounds = total_pieces * 4 + 20
    
    # Accumulators
    bal_harvest_total = 0.0
    bal_drain_dealt = 0.0
    bal_drain_received = 0.0
    exp_sight_sum = 0.0  # sum of sight coverage each turn (for averaging)
    exp_sight_samples = 0
    errors = 0
    
    land_set = {i for i in range(len(parts.terr.elevations))
                if parts.terr.elevations[i] > 0}
    
    old_obj = dict(PURCHASE_OBJ)
    PURCHASE_OBJ.update(trial_weights)
    
    try:
        for turn in range(max_rounds):
            if tp.game_over:
                break
            
            for squad in parts.squads:
                if not squad.king_alive():
                    continue
                
                old_strat = squad.strategy
                squad.reconsider(parts)
                squad.strategy = old_strat
                
                move = squad.execute_queue(parts)
                if move is not None:
                    try:
                        summary = tp.resolve_move(squad, move)
                        if squad is bal_squad:
                            bal_harvest_total += summary.get('food_harvested', 0)
                            bal_drain_dealt += summary.get('food_drain_dealt', 0)
                            bal_drain_received += summary.get('food_drain_received', 0)
                    except MoveError:
                        errors += 1
                        squad.queue.clear()
                        squad.plan_ctx = None
            
            if tp.game_over:
                break
            
            # Measure expansive sight before food round
            exp_placed = [p for p in exp_squad.pieces if p.place is not None]
            if exp_placed:
                visible = set()
                for p in exp_placed:
                    visible |= p.sight().hexes
                exp_sight_sum += len(visible & land_set)
                exp_sight_samples += 1
            
            alive = {sq: Move() for sq in parts.squads if sq.king_alive()}
            if not alive:
                break
            
            try:
                summaries = tp.play_round(alive)
            except MoveError:
                errors += 1
                break
            
            for s in summaries:
                if s['squad'] == bal_squad.name:
                    bal_harvest_total += s.get('food_harvested', 0)
                    bal_drain_dealt += s.get('food_drain_dealt', 0)
                    bal_drain_received += s.get('food_drain_received', 0)
    finally:
        PURCHASE_OBJ.clear()
        PURCHASE_OBJ.update(old_obj)
    
    bal_net = bal_harvest_total + bal_drain_dealt - bal_drain_received
    exp_avg_sight = exp_sight_sum / max(1, exp_sight_samples)
    
    return {
        'bal_net': bal_net,
        'bal_harvest': bal_harvest_total,
        'bal_drain_dealt': bal_drain_dealt,
        'bal_drain_received': bal_drain_received,
        'exp_avg_sight': exp_avg_sight,
        'exp_max_sight': exp_sight_sum,  # cumulative
        'bal_alive': bal_squad.king_alive(),
        'exp_alive': exp_squad.king_alive(),
        'bal_placed': len([p for p in bal_squad.pieces if p.place is not None]),
        'exp_placed': len([p for p in exp_squad.pieces if p.place is not None]),
        'turns': tp.turn,
        'errors': errors,
    }


In [ ]:
def objective_bal_exp(trial: optuna.Trial) -> tuple[float, float]:
    """Multi-objective: maximize bal_net AND exp_avg_sight."""
    
    # Balanced composition
    bal_pawns   = trial.suggest_int("bal_pawns", 3, 6)
    bal_bishops = trial.suggest_int("bal_bishops", 1, 4)
    bal_knights = trial.suggest_int("bal_knights", 0, 3)
    bal_queens  = trial.suggest_int("bal_queens", 0, 2)
    bal_comp = {PieceType.PAWN: bal_pawns, PieceType.BISHOP: bal_bishops,
                PieceType.KNIGHT: bal_knights, PieceType.QUEEN: bal_queens}
    
    # Expansive composition
    exp_pawns   = trial.suggest_int("exp_pawns", 3, 6)
    exp_bishops = trial.suggest_int("exp_bishops", 2, 5)
    exp_knights = trial.suggest_int("exp_knights", 0, 2)
    exp_queens  = trial.suggest_int("exp_queens", 0, 2)
    exp_comp = {PieceType.PAWN: exp_pawns, PieceType.BISHOP: exp_bishops,
                PieceType.KNIGHT: exp_knights, PieceType.QUEEN: exp_queens}
    
    # Weights
    bal_w = tuple(trial.suggest_float(f"bal_w{i}", 0.1, 5.0) for i in range(5))
    exp_w = tuple(trial.suggest_float(f"exp_w{i}", 0.1, 5.0) for i in range(5))
    
    bal_nets, exp_sights = [], []
    for seed in range(5):  # more seeds to reduce quantization
        result = run_trial_bal_exp(board, bal_comp, exp_comp, bal_w, exp_w, seed=seed)
        bal_nets.append(result['bal_net'])
        exp_sights.append(result['exp_avg_sight'])
    
    return np.mean(bal_nets), np.mean(exp_sights)

study2 = optuna.create_study(
    directions=["maximize", "maximize"],
    study_name="bal_vs_exp_v1",
)
study2.optimize(objective_bal_exp, n_trials=5)


Can you rewrite?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

```python
def run_trial(parts: GameParts, 
              agg_comp: dict[PieceType, int],
              turtle_comp: dict[PieceType, int],
              agg_weights: tuple[float, ...],
              turtle_weights: tuple[float, ...],
              starting_pantry: int = 100,
              seed: int = 42,
              verbose: bool = False) -> dict:
    """Run one Aggressive vs Turtle game. Returns metrics dict."""
    
    # --- Cache oasis (expensive) ---
    if not hasattr(parts, '_oasis_idxs'):
        parts._oasis_idxs = parts.fy.find_oasis(3, min_distance=6)
    
    # --- Reset squads on existing board ---
    parts.squads.clear()
    parts.loadSquads(year=1950, seed=seed, levels=2)
    
    for squad in parts.squads:
        for p in squad.pieces:
            p.place = None
    
    starts = parts.find_starts(n=2, ring_radius=3, min_distance=10)
    for squad, start_idx in zip(parts.squads, starts):
        king = next(p for p in squad.pieces if p.type == PieceType.KING)
        loc = parts.location_by_index(start_idx)
        Move.from_squad(squad, placement=Placement(king, loc)).apply(parts)
        king.place.pantry = starting_pantry
    
    agg_squad, turtle_squad = parts.squads[0], parts.squads[1]
    
    # Assign strategies with trial compositions
    agg_strategy = PlacementStrategy(
        "aggressive", defend_threat=1, attack_pantry=40, attack_placed=3,
        knight_roi_mult=0.7, min_reserve_per_piece=3, min_budget_ratio=0.6,
        buy_cooldown=0, target_composition=agg_comp, graduation="random")
    
    turtle_strategy = PlacementStrategy(
        "turtle", defend_threat=1, attack_pantry=999, attack_placed=99,
        min_reserve_per_piece=4, min_budget_ratio=1.0,
        buy_cooldown=0, target_composition=turtle_comp, graduation="balanced")
    
    agg_squad.strategy = agg_strategy
    turtle_squad.strategy = turtle_strategy
    
    # Override PURCHASE_OBJ weights for this trial
    trial_weights = {
        "aggressive": {
            "expand": agg_weights, "defend": agg_weights, "attack": agg_weights,
        },
        "turtle": {
            "expand": turtle_weights, "defend": turtle_weights, "attack": turtle_weights,
        },
    }
    
    tp = TurnProcessor(parts, parts.squads)
    
    # --- Bootstrap pawns ---
    for squad in parts.squads:
        squad.place_optimal_pawns(parts, tp, k=3)
    
    # --- Compute max rounds ---
    total_pieces = sum(agg_comp.values()) + sum(turtle_comp.values())
    max_rounds = total_pieces * 4 + 20
    
    # --- Accumulators ---
    total_drain = 0.0
    total_harvest = 0.0
    purchases = {'aggressive': 0, 'turtle': 0}
    purchase_log = []  # (turn, squad_name, piece_type)
    errors = 0
    
    # Temporarily patch PURCHASE_OBJ
    old_obj = dict(PURCHASE_OBJ)
    PURCHASE_OBJ.update(trial_weights)
    
    try:
        for turn in range(max_rounds):
            if tp.game_over:
                break
            
            for squad in parts.squads:
                if not squad.king_alive():
                    continue
                
                tag = 'aggressive' if squad is agg_squad else 'turtle'
                
                # Reconsider but don't switch strategy
                old_strat = squad.strategy
                squad.reconsider(parts)
                squad.strategy = old_strat  # pin strategy
                
                # Execute one purchase
                move = squad.execute_queue(parts)
                if move is not None:
                    try:
                        summary = tp.resolve_move(squad, move)
                        purchases[tag] += 1
                        pt_name = move.purchase.piece_type.name if move.purchase else "?"
                        purchase_log.append((turn, tag, pt_name))
                        
                        if squad is agg_squad:
                            total_drain += summary.get('food_drain_dealt', 0)
                        else:
                            total_harvest += summary.get('food_harvested', 0)
                        
                        if verbose:
                            placed = len([p for p in squad.pieces if p.place is not None])
                            print(f"  T{turn} {tag}: ✓ {pt_name} (placed={placed})")
                    except MoveError as e:
                        errors += 1
                        if verbose:
                            print(f"  T{turn} {tag}: ✗ {e}")
                        squad.queue.clear()
                        squad.plan_ctx = None
                elif verbose and squad.queue:
                    print(f"  T{turn} {tag}: queue={len(squad.queue)} but execute returned None")
            
            # Food/combat round
            if tp.game_over:
                break
            
            alive = {sq: Move() for sq in parts.squads if sq.king_alive()}
            if not alive:
                break
            
            try:
                summaries = tp.play_round(alive)
            except MoveError:
                errors += 1
                break
            
            for s in summaries:
                if s['squad'] == agg_squad.name:
                    total_drain += s.get('food_drain_dealt', 0)
                elif s['squad'] == turtle_squad.name:
                    total_harvest += s.get('food_harvested', 0)
    finally:
        PURCHASE_OBJ.clear()
        PURCHASE_OBJ.update(old_obj)
    
    # --- Metrics ---
    agg_alive = agg_squad.king_alive()
    turtle_alive = turtle_squad.king_alive()
    agg_placed = len([p for p in agg_squad.pieces if p.place is not None])
    turtle_placed = len([p for p in turtle_squad.pieces if p.place is not None])
    agg_king = next((p for p in agg_squad.pieces 
                     if p.type == PieceType.KING and p.place is not None), None)
    trt_king = next((p for p in turtle_squad.pieces 
                     if p.type == PieceType.KING and p.place is not None), None)
    
    return {
        'agg_drain': total_drain,
        'turtle_harvest': total_harvest,
        'agg_alive': agg_alive,
        'turtle_alive': turtle_alive,
        'agg_placed': agg_placed,
        'turtle_placed': turtle_placed,
        'agg_pantry': agg_king.place.pantry if agg_king else 0,
        'turtle_pantry': trt_king.place.pantry if trt_king else 0,
        'turns': tp.turn,
        'game_over': tp.game_over,
        'purchases': purchases,
        'purchase_log': purchase_log,
        'errors': errors,
    }
```

Run with `verbose=True` first to see what's happening:

```python
result = run_trial(board, 
    agg_comp={PieceType.PAWN: 3, PieceType.KNIGHT: 2, PieceType.BISHOP: 1},
    turtle_comp={PieceType.PAWN: 4, PieceType.BISHOP: 3},
    agg_weights=(0.5, 3.0, 1.0, 1.0, 0.5),
    turtle_weights=(3.0, 0.3, 4.0, 1.0, 1.0),
    seed=42, verbose=True)

print(f"\nTurns: {result['turns']}  Game over: {result['game_over']}  Errors: {result['errors']}")
print(f"Purchases: {result['purchases']}")
print(f"AGG:    alive={result['agg_alive']}  placed={result['agg_placed']}  pantry={result['agg_pantry']}  drain={result['agg_drain']:.1f}")
print(f"TURTLE: alive={result['turtle_alive']}  placed={result['turtle_placed']}  pantry={result['turtle_pantry']}  harvest={result['turtle_harvest']:.1f}")
```

This will tell us exactly how many purchases are landing, which types, and whether errors are eating them silently.

def _score_knight(idx, parts, squad, current_visible, occupied, land_set):
    if idx not in land_set or idx in occupied: return -999
    hp = parts.i2hp(idx)
    loc = parts.location_by_index(idx)
    eff_sight = max(2, loc.level)
    dirs = HexPosition.directions()
    enemy_idxs = {parts.hp2i(p.place.location)
                  for sq in parts.squads if sq is not squad
                  for p in sq.pieces if p.place is not None}
    
    if not enemy_idxs:
        # No enemies visible yet — score on forward positioning
        king = next((p for p in squad.pieces 
                     if p.type == PieceType.KING and p.place is not None), None)
        if king:
            # Prefer hexes closer to map center / away from king (forward deploy)
            return loc.level * 1.5 - hp.distance(king.place.location) * 0.3
        return loc.level * 1.5
    
    # Normal drain scoring
    best = -999
    for d in dirs:
        cone = set(parts.hp2i(h) for h in hp.field_of_view(d, eff_sight))
        enemies_hit = len(cone & enemy_idxs)
        roi = enemies_hit * FOOD_TABLE[PieceType.KNIGHT].drain - FOOD_TABLE[PieceType.KNIGHT].cost
        if roi <= 0: continue
        best = max(best, roi + loc.level * 0.3)
    
    return best if best > -999 else loc.level * 0.5  # fallback if no ROI


In [ ]:
# --- Build board once ---
board = GameParts()
board.grid.adjustRadius(20)

study = optuna.create_study(
    directions=["maximize", "maximize"],
    study_name="agg_vs_turtle_v1",
)
study.optimize(objective, n_trials=5)

any thoughts before we go to 50 runs?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

A few things worth fixing first:

**1. AGG drain is tiny (97-268) vs Turtle harvest (4768-5904)**
The aggressive squad isn't getting near enough enemies to drain. The squads start `min_distance=10` apart — they may never meet on a 20-radius map. Try reducing to `min_distance=6` or even `5` to force contact earlier.

**2. AGG army is stuck at 2 pawns + 2 bishops**
The output shows `AGG army: ['KING', 'PAWN', 'PAWN', 'BISHOP', 'BISHOP']` almost every turn — no knights ever get placed. Either `next_purchase_type` is never returning KNIGHT, or `execute_queue` can't find a valid hex for them. Knights need enemies in sight to score positive ROI — if enemies are too far, `_score_knight` returns `loc.level * 0.5` which may lose out to bishop scores every time.

**3. Same army printed repeatedly**
The "AGG army / TRT army" lines repeating each turn means we're printing inside the trial loop — that's the oasis output fix still needed.

**Before 50 runs, I'd suggest:**

```python
# 1. Closer starts
starts = parts.find_starts(n=2, ring_radius=3, min_distance=6)

# 2. Give knights a fallback score boost so they still get placed
# In _score_knight, when no enemies:
return loc.level * 2.0 + 5.0  # enough to sometimes beat bishops

# 3. Silence the army print (remove whatever debug print is in run_trial)
```

Also — are `food_drain_dealt` and `food_harvested` actually keys in the summary dict? The drain total being suspiciously round numbers (97, 268) suggests it might be counting something else. Worth a quick `print(summary)` to verify.

In [ ]:
print(summary)

Does the summary say anything?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yes, very revealing! Several things stand out:

**1. `drain_edges` are all `amount=0`**
The drain edges exist (Paul and Michael are targeting Blazing Butterflies) but delivering zero drain. This means enemies are technically in sight range but the drain calculation is returning 0. Worth checking if the drain formula requires the enemy to be *in the king component* or if there's a distance/connectivity condition blocking it.

**2. `food_drain_dealt` = 0.0 in summary**
This confirms drain is never actually happening — so our Optuna metric for aggressive is essentially measuring nothing. The 97/268 values we saw are probably from `play_round` summaries, not purchases.

**3. `king_pantry_before` = `king_pantry_after` = 152**
The purchase cost (`purchase_cost: 9`) isn't being deducted from the king pantry — or it's being deducted and re-added. Something in the flow resolution is off.

**4. Individual `pantries` are all 0 except king**
All non-king pieces have pantry=0, meaning food isn't flowing out from them to king — it's all pooling at the king already. So `resolve_flow` is working (centralizing treasury) but drain edges aren't firing.

The core bug is in drain — before running 50 trials, we need to know why `DrainEdge.amount=0` when enemies are in sight. Can you show `!cat ../../HexMagic/magic/turns.py` focusing on where drain edges are computed?

In [ ]:
!cat ../../HexMagic/magic/turns.py

Does the objective function or trial need to be rewriten. There aren't drain the opponent strategies as much as they are about building up economy and getting space

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yes — looking at `_apply_drain`, the drain is `int(p.food.drain)`. Bishop drain is `0.75` → `int(0.75) = 0`. **Bishops never drain anything.** Knights drain 3 but they're not being placed. So the aggressive metric was measuring nothing real.

More fundamentally you're right — these strategies are all expressions of **economy and space**, just different flavors:

| Strategy | What it's really doing | Better metric |
|---|---|---|
| **Aggressive** | Build fast, pressure enemy economy | Enemy king pantry reduction (delta) |
| **Turtle** | Maximize accumulated wealth | Own king pantry at game end |
| **Balanced** | Sustain a large connected army | `placed_count × net_food_per_turn` |
| **Expansive** | Claim territory | Peak unique land hexes in sight |

All measurable directly from the summary dict without drain ever firing:

```python
# Aggressive: how much did we hurt enemy pantry?
enemy_pantry_start - enemy_pantry_end

# Turtle: how rich are we at the end?
result['turtle_pantry']

# Balanced: army size × sustainability  
result['bal_placed'] * (result['bal_harvest'] - result['bal_diet'])

# Expansive: peak territory
result['exp_max_sight']
```

This also means the Aggressive vs Turtle matchup makes more sense as: **does aggressive successfully drain enemy pantry while turtle accumulates wealth?** The tension is real — aggressive spending on knights costs pantry, turtle hoarding pantry avoids that pressure.

Want to rewrite the objective around these metrics?

I think we have ok parameters for aggressive and turtle, but I want something that works well for expansive

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

For expansive, bishops are the key piece (`bonus_sight=3`) so the composition search is simpler. The metric is clear: **average unique land hexes in sight over the game**.

A focused single-strategy study — run expansive vs a passive balanced opponent, only tune expansive params:

```python
def run_expansive_trial(parts: GameParts,
                        exp_comp: dict[PieceType, int],
                        exp_weights: tuple[float, ...],
                        starting_pantry: int = 100,
                        seed: int = 42) -> dict:
    """Run expansive vs passive balanced. Measure territory coverage."""
    
    if not hasattr(parts, '_oasis_idxs'):
        parts._oasis_idxs = parts.fy.find_oasis(3, min_distance=6)
    
    parts.squads.clear()
    parts.loadSquads(year=1950, seed=seed, levels=2)
    for squad in parts.squads:
        for p in squad.pieces:
            p.place = None
    
    starts = parts.find_starts(n=2, ring_radius=3, min_distance=8)
    for squad, start_idx in zip(parts.squads, starts):
        king = next(p for p in squad.pieces if p.type == PieceType.KING)
        loc = parts.location_by_index(start_idx)
        Move.from_squad(squad, placement=Placement(king, loc)).apply(parts)
        king.place.pantry = starting_pantry
    
    exp_squad, bal_squad = parts.squads[0], parts.squads[1]
    
    exp_squad.strategy = PlacementStrategy(
        "expansive", defend_threat=3, attack_pantry=80, attack_placed=6,
        min_reserve_per_piece=4, min_budget_ratio=0.8, buy_cooldown=0,
        target_composition=exp_comp, graduation="aggressive")
    
    # Balanced opponent uses fixed known-good params — not tuned this run
    bal_squad.strategy = PlacementStrategy(
        "balanced", defend_threat=2, attack_pantry=60, attack_placed=4,
        min_reserve_per_piece=5, min_budget_ratio=1.0, buy_cooldown=0,
        target_composition={PieceType.PAWN: 4, PieceType.BISHOP: 2, 
                            PieceType.KNIGHT: 1, PieceType.QUEEN: 1},
        graduation="aggressive")
    
    trial_weights = {
        "expansive": {"expand": exp_weights, "defend": exp_weights, "attack": exp_weights},
        "balanced":  {"expand": (2.0,1.0,2.0,2.0,1.0),  # fixed
                      "defend": (2.0,1.0,3.0,1.0,3.0),
                      "attack": (1.0,3.0,1.5,1.0,1.0)},
    }
    
    tp = TurnProcessor(parts, parts.squads)
    for squad in parts.squads:
        squad.place_optimal_pawns(parts, tp, k=3)
    
    land_set = {i for i in range(len(parts.terr.elevations))
                if parts.terr.elevations[i] > 0}
    total_land = len(land_set)
    
    total_pieces = sum(exp_comp.values())
    max_rounds = total_pieces * 4 + 20
    
    sight_samples = []
    peak_sight = 0
    
    old_obj = dict(PURCHASE_OBJ)
    PURCHASE_OBJ.update(trial_weights)
    
    try:
        for turn in range(max_rounds):
            if tp.game_over: break
            
            for squad in parts.squads:
                if not squad.king_alive(): continue
                old_strat = squad.strategy
                squad.reconsider(parts)
                squad.strategy = old_strat
                move = squad.execute_queue(parts)
                if move is not None:
                    try: tp.resolve_move(squad, move)
                    except MoveError:
                        squad.queue.clear()
                        squad.plan_ctx = None
            
            # Measure expansive sight
            exp_placed = [p for p in exp_squad.pieces if p.place is not None]
            if exp_placed:
                visible = set()
                for p in exp_placed: visible |= p.sight().hexes
                coverage = len(visible & land_set)
                sight_samples.append(coverage)
                peak_sight = max(peak_sight, coverage)
            
            alive = {sq: Move() for sq in parts.squads if sq.king_alive()}
            if not alive: break
            try: tp.play_round(alive)
            except MoveError: break
    finally:
        PURCHASE_OBJ.clear()
        PURCHASE_OBJ.update(old_obj)
    
    exp_king = next((p for p in exp_squad.pieces 
                     if p.type == PieceType.KING and p.place is not None), None)
    
    return {
        'avg_sight': np.mean(sight_samples) if sight_samples else 0,
        'peak_sight': peak_sight,
        'pct_land_covered': peak_sight / max(1, total_land),
        'exp_alive': exp_squad.king_alive(),
        'exp_placed': len([p for p in exp_squad.pieces if p.place is not None]),
        'exp_pantry': exp_king.place.pantry if exp_king else 0,
        'turns': tp.turn,
    }

def objective_expansive(trial: optuna.Trial) -> tuple[float, float]:
    """Maximize avg sight coverage AND peak sight."""
    
    exp_pawns   = trial.suggest_int("exp_pawns", 3, 6)
    exp_bishops = trial.suggest_int("exp_bishops", 3, 7)  # bishops are the point
    exp_knights = trial.suggest_int("exp_knights", 0, 2)
    exp_queens  = trial.suggest_int("exp_queens", 0, 1)
    exp_comp = {PieceType.PAWN: exp_pawns, PieceType.BISHOP: exp_bishops,
                PieceType.KNIGHT: exp_knights, PieceType.QUEEN: exp_queens}
    
    # Weights: food, drain, connect, frontier, threat
    # For expansive, frontier (w3) should matter most
    exp_w = tuple(trial.suggest_float(f"exp_w{i}", 0.1, 5.0) for i in range(5))
    
    avg_sights, peak_sights = [], []
    for seed in range(5):
        result = run_expansive_trial(board, exp_comp, exp_w, seed=seed)
        avg_sights.append(result['avg_sight'])
        peak_sights.append(result['peak_sight'])
    
    return np.mean(avg_sights), np.mean(peak_sights)

study_exp = optuna.create_study(
    directions=["maximize", "maximize"],
    study_name="expansive_v1",
)
study_exp.optimize(objective_expansive, n_trials=50)
```

The key difference from the previous studies:
- **Single strategy tuned** — balanced opponent is fixed so noise is lower
- **Two sight metrics** — avg coverage (sustained territory) and peak (maximum reach ever achieved)  
- **Bishop range 3-7** — expansive lives or dies by bishop count
- **Frontier weight** (`exp_w3`) should naturally float high if the scorer is working

In [ ]:
def run_expansive_trial(parts: GameParts,
                        exp_comp: dict[PieceType, int],
                        exp_weights: tuple[float, ...],
                        starting_pantry: int = 100,
                        seed: int = 42) -> dict:
    """Run expansive vs passive balanced. Measure territory coverage."""
    
    if not hasattr(parts, '_oasis_idxs'):
        parts._oasis_idxs = parts.fy.find_oasis(3, min_distance=6)
    
    parts.squads.clear()
    parts.loadSquads(year=1950, seed=seed, levels=2)
    for squad in parts.squads:
        for p in squad.pieces:
            p.place = None
    
    starts = parts.find_starts(n=2, ring_radius=3, min_distance=8)
    for squad, start_idx in zip(parts.squads, starts):
        king = next(p for p in squad.pieces if p.type == PieceType.KING)
        loc = parts.location_by_index(start_idx)
        Move.from_squad(squad, placement=Placement(king, loc)).apply(parts)
        king.place.pantry = starting_pantry
    
    exp_squad, bal_squad = parts.squads[0], parts.squads[1]
    
    exp_squad.strategy = PlacementStrategy(
        "expansive", defend_threat=3, attack_pantry=80, attack_placed=6,
        min_reserve_per_piece=4, min_budget_ratio=0.8, buy_cooldown=0,
        target_composition=exp_comp, graduation="aggressive")
    
    # Balanced opponent uses fixed known-good params — not tuned this run
    bal_squad.strategy = PlacementStrategy(
        "balanced", defend_threat=2, attack_pantry=60, attack_placed=4,
        min_reserve_per_piece=5, min_budget_ratio=1.0, buy_cooldown=0,
        target_composition={PieceType.PAWN: 4, PieceType.BISHOP: 2, 
                            PieceType.KNIGHT: 1, PieceType.QUEEN: 1},
        graduation="aggressive")
    
    trial_weights = {
        "expansive": {"expand": exp_weights, "defend": exp_weights, "attack": exp_weights},
        "balanced":  {"expand": (2.0,1.0,2.0,2.0,1.0),  # fixed
                      "defend": (2.0,1.0,3.0,1.0,3.0),
                      "attack": (1.0,3.0,1.5,1.0,1.0)},
    }
    
    tp = TurnProcessor(parts, parts.squads)
    for squad in parts.squads:
        squad.place_optimal_pawns(parts, tp, k=3)
    
    land_set = {i for i in range(len(parts.terr.elevations))
                if parts.terr.elevations[i] > 0}
    total_land = len(land_set)
    
    total_pieces = sum(exp_comp.values())
    max_rounds = total_pieces * 4 + 20
    
    sight_samples = []
    peak_sight = 0
    
    old_obj = dict(PURCHASE_OBJ)
    PURCHASE_OBJ.update(trial_weights)
    
    try:
        for turn in range(max_rounds):
            if tp.game_over: break
            
            for squad in parts.squads:
                if not squad.king_alive(): continue
                old_strat = squad.strategy
                squad.reconsider(parts)
                squad.strategy = old_strat
                move = squad.execute_queue(parts)
                if move is not None:
                    try: tp.resolve_move(squad, move)
                    except MoveError:
                        squad.queue.clear()
                        squad.plan_ctx = None
            
            # Measure expansive sight
            exp_placed = [p for p in exp_squad.pieces if p.place is not None]
            if exp_placed:
                visible = set()
                for p in exp_placed: visible |= p.sight().hexes
                coverage = len(visible & land_set)
                sight_samples.append(coverage)
                peak_sight = max(peak_sight, coverage)
            
            alive = {sq: Move() for sq in parts.squads if sq.king_alive()}
            if not alive: break
            try: tp.play_round(alive)
            except MoveError: break
    finally:
        PURCHASE_OBJ.clear()
        PURCHASE_OBJ.update(old_obj)
    
    exp_king = next((p for p in exp_squad.pieces 
                     if p.type == PieceType.KING and p.place is not None), None)
    
    return {
        'avg_sight': np.mean(sight_samples) if sight_samples else 0,
        'peak_sight': peak_sight,
        'pct_land_covered': peak_sight / max(1, total_land),
        'exp_alive': exp_squad.king_alive(),
        'exp_placed': len([p for p in exp_squad.pieces if p.place is not None]),
        'exp_pantry': exp_king.place.pantry if exp_king else 0,
        'turns': tp.turn,
    }

def objective_expansive(trial: optuna.Trial) -> tuple[float, float]:
    """Maximize avg sight coverage AND peak sight."""
    
    exp_pawns   = trial.suggest_int("exp_pawns", 3, 6)
    exp_bishops = trial.suggest_int("exp_bishops", 3, 7)  # bishops are the point
    exp_knights = trial.suggest_int("exp_knights", 0, 2)
    exp_queens  = trial.suggest_int("exp_queens", 0, 1)
    exp_comp = {PieceType.PAWN: exp_pawns, PieceType.BISHOP: exp_bishops,
                PieceType.KNIGHT: exp_knights, PieceType.QUEEN: exp_queens}
    
    # Weights: food, drain, connect, frontier, threat
    # For expansive, frontier (w3) should matter most
    exp_w = tuple(trial.suggest_float(f"exp_w{i}", 0.1, 5.0) for i in range(5))
    
    avg_sights, peak_sights = [], []
    for seed in range(5):
        result = run_expansive_trial(board, exp_comp, exp_w, seed=seed)
        avg_sights.append(result['avg_sight'])
        peak_sights.append(result['peak_sight'])
    
    return np.mean(avg_sights), np.mean(peak_sights)

study_exp = optuna.create_study(
    directions=["maximize", "maximize"],
    study_name="expansive_v1",
)
study_exp.optimize(objective_expansive, n_trials=5)


So we aren't getting any variation with expansive which might make sense since there is only one way to expand by selection - keep doing bishops. So the question comes down to what can we optimize for.
1. how few turns it takes to reach oasis of different distances from the king. in theory there is a targeting distance vs reward trade off that is worth tinkering for. once you reach a place then you should switch strategies to either turtle or attack.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

That's a much better framing — expansive isn't about *how many* hexes you see, it's about **how quickly you reach a valuable target**. The optimization becomes:

**Objective**: minimize turns to get a piece in sight of an oasis, weighted by oasis richness/distance ratio.

The tunable parameters become:
1. **Pawn-to-bishop ratio** — how much food engine before pushing outward
2. **Facing strategy** — do you fan out wide or beeline toward highest-value oasis?
3. **When to switch** — once you reach the oasis, graduate to turtle (sit on it) or aggressive (deny it to others)?

```python
def run_oasis_race(parts: GameParts,
                   exp_comp: dict[PieceType, int],
                   graduation: str,  # "turtle" or "aggressive"
                   starting_pantry: int = 100,
                   seed: int = 42) -> dict:
    """Measure how quickly expansive reaches each oasis."""
    
    if not hasattr(parts, '_oasis_idxs'):
        parts._oasis_idxs = parts.fy.find_oasis(3, min_distance=6)
    
    parts.squads.clear()
    parts.loadSquads(year=1950, seed=seed, levels=1)  # solo — pure expansion test
    squad = parts.squads[0]
    for p in squad.pieces:
        p.place = None
    
    starts = parts.find_starts(n=1, ring_radius=3, min_distance=6)
    king = next(p for p in squad.pieces if p.type == PieceType.KING)
    loc = parts.location_by_index(starts[0])
    Move.from_squad(squad, placement=Placement(king, loc)).apply(parts)
    king.place.pantry = starting_pantry
    king_idx = starts[0]
    
    squad.strategy = PlacementStrategy(
        "expansive", defend_threat=3, attack_pantry=80, attack_placed=6,
        min_reserve_per_piece=4, min_budget_ratio=0.8, buy_cooldown=0,
        target_composition=exp_comp, graduation=graduation)
    
    tp = TurnProcessor(parts, parts.squads)
    squad.place_optimal_pawns(parts, tp, k=3)
    
    # Track when each oasis first enters sight
    oasis_discovery = {}  # oasis_idx -> turn discovered
    oasis_info = {}       # oasis_idx -> {dist, richness}
    king_hp = parts.i2hp(king_idx)
    
    for oidx in parts._oasis_idxs:
        ohp = parts.i2hp(oidx)
        oasis_info[oidx] = {
            'dist': king_hp.distance(ohp),
            'richness': float(parts.fy.tiers[oidx]),
        }
    
    total_pieces = sum(exp_comp.values())
    max_rounds = total_pieces * 4 + 10
    
    for turn in range(max_rounds):
        if tp.game_over: break
        
        old_strat = squad.strategy
        squad.reconsider(parts)
        squad.strategy = old_strat
        
        move = squad.execute_queue(parts)
        if move is not None:
            try: tp.resolve_move(squad, move)
            except MoveError:
                squad.queue.clear()
                squad.plan_ctx = None
        
        # Check sight coverage
        placed = [p for p in squad.pieces if p.place is not None]
        visible = set()
        for p in placed: visible |= p.sight().hexes
        
        for oidx in parts._oasis_idxs:
            if oidx not in oasis_discovery and oidx in visible:
                oasis_discovery[oidx] = turn
        
        alive = {squad: Move()}
        try: tp.play_round(alive)
        except MoveError: break
    
    # Score: sum of (richness / turns_to_discover) for each found oasis
    # Unfound oases get 0
    reach_score = 0.0
    for oidx, turn_found in oasis_discovery.items():
        info = oasis_info[oidx]
        reach_score += info['richness'] / max(1, turn_found + 1)
    
    exp_king = next((p for p in squad.pieces 
                     if p.type == PieceType.KING and p.place is not None), None)
    
    return {
        'reach_score': reach_score,
        'oases_found': len(oasis_discovery),
        'oases_total': len(parts._oasis_idxs),
        'discovery_turns': oasis_discovery,  # {idx: turn}
        'oasis_info': oasis_info,
        'exp_placed': len([p for p in squad.pieces if p.place is not None]),
        'exp_pantry': exp_king.place.pantry if exp_king else 0,
        'turns': tp.turn,
        'graduation': graduation,
    }
```

```python
def objective_oasis_race(trial: optuna.Trial) -> float:
    """Single objective: maximize reach_score (richness / speed)."""
    
    exp_pawns   = trial.suggest_int("exp_pawns", 2, 5)
    exp_bishops = trial.suggest_int("exp_bishops", 2, 6)
    exp_queens  = trial.suggest_int("exp_queens", 0, 1)
    exp_comp = {PieceType.PAWN: exp_pawns, PieceType.BISHOP: exp_bishops,
                PieceType.QUEEN: exp_queens}
    
    graduation = trial.suggest_categorical("graduation", ["turtle", "aggressive"])
    
    scores = []
    for seed in range(5):
        result = run_oasis_race(board, exp_comp, graduation, seed=seed)
        scores.append(result['reach_score'])
    
    return np.mean(scores)

study_oasis = optuna.create_study(
    direction="maximize",
    study_name="oasis_race_v1",
)
study_oasis.optimize(objective_oasis_race, n_trials=50)
```

Key changes:
- **Solo player** — no opponent noise, pure expansion measurement
- **`reach_score = Σ(richness / turns_to_discover)`** — rewards finding rich oases fast, penalizes slow expansion
- **Graduation is a tunable** — does turtle or aggressive work better after finding the oasis?
- **No knights** — expansive doesn't need them, removed from search space
- **Single objective** — simpler Pareto, just maximize reach efficiency

I think we should have 5 random seeds so we don't over optimize (if not more that we pick a sample of for each trial)

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Good call — and even better, sample from a larger pool so each trial sees different seeds:

```python
SEED_POOL = list(range(50))  # 50 possible seeds

def objective_oasis_race(trial: optuna.Trial) -> float:
    """Single objective: maximize reach_score (richness / speed)."""
    
    exp_pawns   = trial.suggest_int("exp_pawns", 2, 5)
    exp_bishops = trial.suggest_int("exp_bishops", 2, 6)
    exp_queens  = trial.suggest_int("exp_queens", 0, 1)
    exp_comp = {PieceType.PAWN: exp_pawns, PieceType.BISHOP: exp_bishops,
                PieceType.QUEEN: exp_queens}
    
    graduation = trial.suggest_categorical("graduation", ["turtle", "aggressive"])
    
    # Sample 5 random seeds from a pool of 50
    rng = random.Random(trial.number)  # deterministic per trial for reproducibility
    seeds = rng.sample(SEED_POOL, 5)
    
    scores = []
    for seed in seeds:
        result = run_oasis_race(board, exp_comp, graduation, seed=seed)
        scores.append(result['reach_score'])
    
    return np.mean(scores)
```

Using `Random(trial.number)` means:
- Each trial gets a **different** set of 5 seeds
- But the **same trial** always gets the same seeds if re-run (reproducible)
- Over 50 trials you'll touch most of the 50-seed pool, giving broad coverage
- No single lucky map position can dominate the results

In [ ]:
def run_oasis_race(parts: GameParts,
                   exp_comp: dict[PieceType, int],
                   graduation: str,  # "turtle" or "aggressive"
                   starting_pantry: int = 100,
                   seed: int = 42) -> dict:
    """Measure how quickly expansive reaches each oasis."""
    
    if not hasattr(parts, '_oasis_idxs'):
        parts._oasis_idxs = parts.fy.find_oasis(3, min_distance=6)
    
    parts.squads.clear()
    parts.loadSquads(year=1950, seed=seed, levels=1)  # solo — pure expansion test
    squad = parts.squads[0]
    for p in squad.pieces:
        p.place = None
    
    starts = parts.find_starts(n=1, ring_radius=3, min_distance=6)
    king = next(p for p in squad.pieces if p.type == PieceType.KING)
    loc = parts.location_by_index(starts[0])
    Move.from_squad(squad, placement=Placement(king, loc)).apply(parts)
    king.place.pantry = starting_pantry
    king_idx = starts[0]
    
    squad.strategy = PlacementStrategy(
        "expansive", defend_threat=3, attack_pantry=80, attack_placed=6,
        min_reserve_per_piece=4, min_budget_ratio=0.8, buy_cooldown=0,
        target_composition=exp_comp, graduation=graduation)
    
    tp = TurnProcessor(parts, parts.squads)
    tp._check_game_over = lambda: None  # disable for solo mode
    squad.place_optimal_pawns(parts, tp, k=3)
    
    # Track when each oasis first enters sight
    oasis_discovery = {}  # oasis_idx -> turn discovered
    oasis_info = {}       # oasis_idx -> {dist, richness}
    king_hp = parts.i2hp(king_idx)
    
    # Replace oasis_info setup with:
    # Replace the oasis_info block with:
    for oidx in parts._oasis_idxs:
        ohp = parts.i2hp(oidx)
        loc = parts.location_by_index(oidx)
        oasis_info[oidx] = {
            'dist': king_hp.distance(ohp),
            'richness': max(1.0, float(loc.level) + float(parts.fy.tiers[oidx])),
        }

    
    total_pieces = sum(exp_comp.values())
    max_rounds = total_pieces * 4 + 10
    
    for turn in range(max_rounds):
        if tp.game_over: break
        
        old_strat = squad.strategy
        squad.reconsider(parts)
        squad.strategy = old_strat
        
        move = squad.execute_queue(parts)
        if move is not None:
            try: tp.resolve_move(squad, move)
            except MoveError:
                squad.queue.clear()
                squad.plan_ctx = None
        
        # Check sight coverage
        placed = [p for p in squad.pieces if p.place is not None]
        visible = set()
        for p in placed: visible |= p.sight().hexes
        
        for oidx in parts._oasis_idxs:
            if oidx not in oasis_discovery and oidx in visible:
                oasis_discovery[oidx] = turn
        
        alive = {squad: Move()}
        try: tp.play_round(alive)
        except MoveError: break
    
    # Score: sum of (richness / turns_to_discover) for each found oasis
    # Unfound oases get 0
    reach_score = 0.0
    for oidx, turn_found in oasis_discovery.items():
        info = oasis_info[oidx]
        #reach_score += info['richness'] / max(1, turn_found + 1)
        # Reward finding far oases fast, near oases fast is less impressive
        reach_score += (info['dist'] * info['richness']) / max(1, turn_found + 1)
    
    exp_king = next((p for p in squad.pieces 
                     if p.type == PieceType.KING and p.place is not None), None)
    
    return {
        'reach_score': reach_score,
        'oases_found': len(oasis_discovery),
        'oases_total': len(parts._oasis_idxs),
        'discovery_turns': oasis_discovery,  # {idx: turn}
        'oasis_info': oasis_info,
        'exp_placed': len([p for p in squad.pieces if p.place is not None]),
        'exp_pantry': exp_king.place.pantry if exp_king else 0,
        'turns': tp.turn,
        'graduation': graduation,
    }


In [ ]:
SEED_POOL = list(range(50))  # 50 possible seeds

def objective_oasis_race(trial: optuna.Trial) -> float:
    """Single objective: maximize reach_score (richness / speed)."""
    
    exp_pawns   = trial.suggest_int("exp_pawns", 2, 5)
    exp_bishops = trial.suggest_int("exp_bishops", 2, 6)
    exp_queens  = trial.suggest_int("exp_queens", 0, 1)
    exp_comp = {PieceType.PAWN: exp_pawns, PieceType.BISHOP: exp_bishops,
                PieceType.QUEEN: exp_queens}
    
    graduation = trial.suggest_categorical("graduation", ["turtle", "aggressive"])
    
    # Sample 5 random seeds from a pool of 50
    rng = random.Random(trial.number)  # deterministic per trial for reproducibility
    seeds = rng.sample(SEED_POOL, 5)
    
    scores = []
    for seed in seeds:
        result = run_oasis_race(board, exp_comp, graduation, seed=seed)
        scores.append(result['reach_score'])
    
    return np.mean(scores)


In [ ]:
def objective_oasis_race(trial: optuna.Trial) -> float:
    """Single objective: maximize reach_score (richness / speed)."""
    
    exp_pawns   = trial.suggest_int("exp_pawns", 2, 5)
    exp_bishops = trial.suggest_int("exp_bishops", 2, 6)
    exp_queens  = trial.suggest_int("exp_queens", 0, 1)
    exp_comp = {PieceType.PAWN: exp_pawns, PieceType.BISHOP: exp_bishops,
                PieceType.QUEEN: exp_queens}
    
    graduation = trial.suggest_categorical("graduation", ["turtle", "aggressive"])
    
    scores = []
    for seed in range(5):
        result = run_oasis_race(board, exp_comp, graduation, seed=seed)
        scores.append(result['reach_score'])
    
    return np.mean(scores)

study_oasis = optuna.create_study(
    direction="maximize",
    study_name="oasis_race_v1",
)
study_oasis.optimize(objective_oasis_race, n_trials=2)


should it find n oasis? or one that is farther so it takes more steps?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Two problems visible in the output:

1. **Only 1 oasis found per seed** (`spread=0.00`) — `levels=1` probably constrains `find_oasis` to return fewer results
2. **Score always 1.0** — the oasis is being found on turn 0 (already in sight after pawn bootstrap), so `richness / (0+1) = richness` but normalized to 1.0

The score needs **distance variation** to be meaningful. A few options:

**Option A: Multiple oases at varying distances**
```python
# Use levels=1 but force find_oasis to return more, at greater spread
parts._oasis_idxs = parts.fy.find_oasis(5, min_distance=4)
```

**Option B: Single far oasis — measure turns to reach it**
```python
# Pick the *farthest* oasis from king start
farthest = max(parts._oasis_idxs, 
               key=lambda i: king_hp.distance(parts.i2hp(i)))
# Score = 1 / turns_to_discover (faster = better)
```

**Option C: Distance-weighted scoring**
```python
# Reward finding far oases fast, near oases fast is less impressive
reach_score += (info['dist'] * info['richness']) / max(1, turn_found + 1)
```

Option C feels most natural — a squad that scouts far rich territory quickly is genuinely better at expanding. Near oases that fall into sight immediately barely contribute.

Which direction feels right?

I don't think there is a purchase strategy that is going to make a difference. can you write the update to the strategies weights and compositions. we are going to pivot into choosing the best strategy and we will look these in for now.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

```python
# ── Locked compositions from Optuna studies ──────────────────────────────

PlacementStrategy.AGGRESSIVE = PlacementStrategy(
    "aggressive",
    defend_threat=1, attack_pantry=40, attack_placed=3,
    knight_roi_mult=0.7, min_reserve_per_piece=3,
    min_budget_ratio=0.6, buy_cooldown=0,
    target_composition={
        PieceType.PAWN:   5,
        PieceType.KNIGHT: 3,
        PieceType.BISHOP: 3,
    },
    graduation="random")

PlacementStrategy.TURTLE = PlacementStrategy(
    "turtle",
    defend_threat=1, attack_pantry=999, attack_placed=99,
    min_reserve_per_piece=4, min_budget_ratio=1.0, buy_cooldown=0,
    target_composition={
        PieceType.PAWN:   6,
        PieceType.BISHOP: 5,
        PieceType.QUEEN:  2,
    },
    graduation="balanced")

PlacementStrategy.EXPANSIVE = PlacementStrategy(
    "expansive",
    defend_threat=3, attack_pantry=80, attack_placed=6,
    min_reserve_per_piece=4, min_budget_ratio=0.8, buy_cooldown=0,
    target_composition={
        PieceType.PAWN:   4,
        PieceType.BISHOP: 5,  # bishops = sight range = territory
    },
    graduation="turtle")   # once you reach oasis, sit on it

PlacementStrategy.BALANCED = PlacementStrategy(
    "balanced",
    defend_threat=2, attack_pantry=60, attack_placed=4,
    min_reserve_per_piece=5, min_budget_ratio=1.0, buy_cooldown=0,
    target_composition={
        PieceType.PAWN:   4,
        PieceType.BISHOP: 2,
        PieceType.KNIGHT: 1,
        PieceType.QUEEN:  1,
    },
    graduation="aggressive")

# ── Locked PURCHASE_OBJ weights ───────────────────────────────────────────
#                        food  drain  connect  frontier  threat
PURCHASE_OBJ = {
    "aggressive": {
        "expand":        (0.7,  2.7,   3.7,     4.9,     4.6),
        "defend":        (0.7,  2.7,   3.7,     4.9,     4.6),
        "attack":        (0.7,  2.7,   3.7,     4.9,     4.6),
    },
    "turtle": {
        "expand":        (1.6,  4.3,   2.6,     2.4,     3.9),
        "defend":        (1.6,  4.3,   2.6,     2.4,     3.9),
        "attack":        (1.6,  4.3,   2.6,     2.4,     3.9),
    },
    "expansive": {
        "expand":        (1.1,  0.7,   1.5,     4.4,     1.0),
        "defend":        (1.1,  0.7,   1.5,     4.4,     1.0),
        "attack":        (1.1,  0.7,   1.5,     4.4,     1.0),
    },
    "balanced": {
        "expand":        (2.0,  1.0,   2.0,     2.0,     1.0),
        "defend":        (2.0,  1.0,   3.0,     1.0,     3.0),
        "attack":        (1.0,  3.0,   1.5,     1.0,     1.0),
    },
}

_STRATEGY_ROTATION = [
    PlacementStrategy.EXPANSIVE,
    PlacementStrategy.AGGRESSIVE,
    PlacementStrategy.BALANCED,
    PlacementStrategy.TURTLE,
]
```

Notes on the weights:
- **Aggressive**: frontier high (4.9) so it pushes toward enemies, threat high (4.6) for reactive defense
- **Turtle**: drain high (4.3) discourages enemies, threat high (3.9) for king protection  
- **Expansive**: frontier dominant (4.4), everything else low — pure territory
- **Balanced**: kept as the original hand-tuned values since it's the fallback graduation target

The weights won't move the needle much per the Optuna findings, but they at least encode the strategic intent correctly.

what I am wondering about is a table or chart that would help visualize what and why different players are doing on a turn basis and how much their strategy is working.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

A few context messages were trimmed from this dialog, but I have plenty to work with.

The `GameRecord` already snapshots per-turn data, so the question is how to surface **strategy effectiveness** alongside the raw numbers. Here's a dashboard DataFrame approach:

```python
def strategy_dashboard(tp: TurnProcessor) -> pd.DataFrame:
    """Per-turn, per-squad view of strategy effectiveness."""
    df = tp.record.df.copy()
    if df.empty:
        return df
    
    # Strategy-specific effectiveness score
    def effectiveness(row):
        stance = row.get('stance', 'expand')
        strategy = row.get('target_type', 'none')  # approximate
        
        # Each strategy measures success differently
        harvest = row['total_harvest']
        diet = row['total_diet']
        net = row['net_food']
        drain = row['drain_dealt']
        sight = row['sight_coverage']
        placed = row['pieces_placed']
        pantry = row['king_pantry']
        
        # Normalize to 0-100 scale roughly
        return {
            'food_efficiency': round(net / max(1, diet) * 100, 1),       # harvest ROI
            'army_strength': placed * max(0, net),                        # sustainable army
            'territory': sight,                                           # raw coverage
            'economy': pantry + net * 5,                                  # pantry + projected
            'pressure': drain,                                            # drain dealt
        }
    
    # Expand effectiveness into columns
    eff = df.apply(effectiveness, axis=1, result_type='expand')
    df = pd.concat([df, eff], axis=1)
    
    # Add turn-over-turn deltas
    for col in ['king_pantry', 'sight_coverage', 'pieces_placed']:
        df[f'{col}_delta'] = df.groupby('squad')[col].diff().fillna(0)
    
    return df[['turn', 'squad', 'alive', 'king_pantry', 'king_pantry_delta',
               'pieces_placed', 'pieces_placed_delta', 'total_harvest', 'total_diet',
               'net_food', 'drain_dealt', 'drain_taken', 'sight_coverage',
               'sight_coverage_delta', 'bottlenecks', 'isolated',
               'food_efficiency', 'army_strength', 'territory', 'economy', 'pressure',
               'stance']]
```

And a per-strategy sparkline chart:

```python
def strategy_chart(tp: TurnProcessor):
    """Matplotlib chart: each strategy's key metric over time."""
    df = strategy_dashboard(tp)
    if df.empty:
        return
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
    fig.suptitle("Strategy Effectiveness by Turn", fontsize=14, fontweight='bold')
    
    squads = df['squad'].unique()
    colors = plt.cm.Set2(range(len(squads)))
    
    # Panel 1: Economy (king pantry)
    ax = axes[0, 0]
    ax.set_title("💰 Economy (King Pantry)")
    for i, sq in enumerate(squads):
        sd = df[df.squad == sq]
        ax.plot(sd['turn'], sd['king_pantry'], label=sq, color=colors[i], linewidth=2)
    ax.legend(fontsize=8)
    ax.set_ylabel("Pantry")
    
    # Panel 2: Territory (sight coverage)
    ax = axes[0, 1]
    ax.set_title("🔭 Territory (Sight Coverage)")
    for i, sq in enumerate(squads):
        sd = df[df.squad == sq]
        ax.plot(sd['turn'], sd['sight_coverage'], color=colors[i], linewidth=2)
    ax.set_ylabel("Hexes Visible")
    
    # Panel 3: Army (pieces placed)
    ax = axes[1, 0]
    ax.set_title("⚔️ Army Size (Pieces Placed)")
    for i, sq in enumerate(squads):
        sd = df[df.squad == sq]
        ax.plot(sd['turn'], sd['pieces_placed'], color=colors[i], linewidth=2)
    ax.set_ylabel("Placed")
    ax.set_xlabel("Turn")
    
    # Panel 4: Food efficiency (net / diet)
    ax = axes[1, 1]
    ax.set_title("🌾 Food Efficiency (Net / Diet %)")
    for i, sq in enumerate(squads):
        sd = df[df.squad == sq]
        ax.plot(sd['turn'], sd['food_efficiency'], color=colors[i], linewidth=2)
    ax.axhline(0, color='red', linestyle='--', alpha=0.3)
    ax.set_ylabel("Efficiency %")
    ax.set_xlabel("Turn")
    
    plt.tight_layout()
    plt.show()
```

The key insight is that **each panel answers a different strategy's question**:
- 💰 Economy → Is turtle accumulating wealth?
- 🔭 Territory → Is expansive claiming land?
- ⚔️ Army → Is balanced sustaining a large force?
- 🌾 Efficiency → Is anyone starving?

Run it after a game:

```python
strategy_chart(tp)
strategy_dashboard(tp).tail(20)
```

In [ ]:
#| export
@patch
def strategy_dashboard(tp: TurnProcessor) -> pd.DataFrame:
    """Per-turn, per-squad view of strategy effectiveness."""
    df = tp.record.df.copy()
    if df.empty:
        return df
    
    # Strategy-specific effectiveness score
    def effectiveness(row):
        stance = row.get('stance', 'expand')
        strategy = row.get('target_type', 'none')  # approximate
        
        # Each strategy measures success differently
        harvest = row['total_harvest']
        diet = row['total_diet']
        net = row['net_food']
        drain = row['drain_dealt']
        sight = row['sight_coverage']
        placed = row['pieces_placed']
        pantry = row['king_pantry']
        
        # Normalize to 0-100 scale roughly
        return {
            'food_efficiency': round(net / max(1, diet) * 100, 1),       # harvest ROI
            'army_strength': placed * max(0, net),                        # sustainable army
            'territory': sight,                                           # raw coverage
            'economy': pantry + net * 5,                                  # pantry + projected
            'pressure': drain,                                            # drain dealt
        }
    
    # Expand effectiveness into columns
    eff = df.apply(effectiveness, axis=1, result_type='expand')
    df = pd.concat([df, eff], axis=1)
    
    # Add turn-over-turn deltas
    for col in ['king_pantry', 'sight_coverage', 'pieces_placed']:
        df[f'{col}_delta'] = df.groupby('squad')[col].diff().fillna(0)
    
    return df[['turn', 'squad', 'alive', 'king_pantry', 'king_pantry_delta',
               'pieces_placed', 'pieces_placed_delta', 'total_harvest', 'total_diet',
               'net_food', 'drain_dealt', 'drain_taken', 'sight_coverage',
               'sight_coverage_delta', 'bottlenecks', 'isolated',
               'food_efficiency', 'army_strength', 'territory', 'economy', 'pressure',
               'stance']]


In [ ]:
#| export
@patch
def strategy_chart(tp: TurnProcessor):
    """Matplotlib chart: each strategy's key metric over time."""
    df = tp.strategy_dashboard()
    if df.empty:
        return
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
    fig.suptitle("Strategy Effectiveness by Turn", fontsize=14, fontweight='bold')
    
    squads = df['squad'].unique()
    colors = plt.cm.Set2(range(len(squads)))
    
    # Panel 1: Economy (king pantry)
    ax = axes[0, 0]
    ax.set_title("💰 Economy (King Pantry)")
    for i, sq in enumerate(squads):
        sd = df[df.squad == sq]
        ax.plot(sd['turn'], sd['king_pantry'], label=sq, color=colors[i], linewidth=2)
    ax.legend(fontsize=8)
    ax.set_ylabel("Pantry")
    
    # Panel 2: Territory (sight coverage)
    ax = axes[0, 1]
    ax.set_title("🔭 Territory (Sight Coverage)")
    for i, sq in enumerate(squads):
        sd = df[df.squad == sq]
        ax.plot(sd['turn'], sd['sight_coverage'], color=colors[i], linewidth=2)
    ax.set_ylabel("Hexes Visible")
    
    # Panel 3: Army (pieces placed)
    ax = axes[1, 0]
    ax.set_title("⚔️ Army Size (Pieces Placed)")
    for i, sq in enumerate(squads):
        sd = df[df.squad == sq]
        ax.plot(sd['turn'], sd['pieces_placed'], color=colors[i], linewidth=2)
    ax.set_ylabel("Placed")
    ax.set_xlabel("Turn")
    
    # Panel 4: Food efficiency (net / diet)
    ax = axes[1, 1]
    ax.set_title("🌾 Food Efficiency (Net / Diet %)")
    for i, sq in enumerate(squads):
        sd = df[df.squad == sq]
        ax.plot(sd['turn'], sd['food_efficiency'], color=colors[i], linewidth=2)
    ax.axhline(0, color='red', linestyle='--', alpha=0.3)
    ax.set_ylabel("Efficiency %")
    ax.set_xlabel("Turn")
    
    plt.tight_layout()
    plt.show()
